# 11 — Arbitrage-Aware SVI/SSVI Surface Repair and Handoff

This notebook converts the raw SPY implied-volatility surface diagnostics from Notebook 10 into an arbitrage-aware repaired surface suitable for downstream stochastic-volatility calibration.

Notebook 10 established that the empirical option surface is usable but not fully clean. The filtered surface passed pointwise price bounds, monotonicity checks, and calendar checks, but retained material butterfly/convexity warnings. Therefore, this notebook does **not** proceed directly to Heston/Bates calibration. Its role is to construct, diagnose, repair, and audit a smooth calibration target first.

The central object in this notebook is the Black-Scholes implied **total variance** surface:

$$
w(k,T) = \sigma_{\mathrm{BS}}^2(k,T)T
$$

where

$$
k = \log\left(\frac{K}{F_T}\right)
$$

is log-forward moneyness, \(K\) is strike, \(F_T\) is the maturity-matched forward, \(T\) is time to maturity, and \(\sigma_{\mathrm{BS}}(k,T)\) is the Black-Scholes implied volatility.

The raw SVI slice parameterization for each maturity is

$$
w(k;\chi)
=
a
+
b
\left[
\rho(k-m)
+
\sqrt{(k-m)^2+\sigma^2}
\right],
$$

with parameter vector

$$
\chi = (a,b,\rho,m,\sigma),
$$

subject to economically meaningful restrictions such as

$$
b \ge 0,
\qquad
|\rho| < 1,
\qquad
\sigma > 0,
\qquad
w(k;\chi) \ge 0.
$$

The surface-level no-static-arbitrage requirement is decomposed into two checks:

$$
\text{No static arbitrage}
\iff
\text{No calendar-spread arbitrage}
+
\text{No butterfly arbitrage}.
$$

Calendar-spread consistency requires total variance to be nondecreasing in maturity at fixed log-forward moneyness:

$$
\frac{\partial w(k,T)}{\partial T} \ge 0
\qquad
\forall k,T.
$$

Butterfly-arbitrage consistency requires each maturity slice to imply a nonnegative risk-neutral density. Operationally, this notebook evaluates convexity and density-proxy diagnostics on fitted and repaired slices before allowing any surface to be handed off for calibration.

## Notebook purpose

The purpose of this notebook is to answer the following research question:

> Can the Notebook 10 SPY implied-volatility surface be transformed into a smooth, arbitrage-aware calibration target without materially rewriting the market information contained in the original option quotes?

The notebook will:

1. load the broad, strict, anchor, repair-queue, excluded, and full diagnostic handoffs from Notebook 10;
2. reconstruct the contract-level total-variance surface using log-forward moneyness;
3. fit SVI slices under multiple candidate weighting schemes;
4. diagnose butterfly and calendar violations before and after fitting;
5. apply repair/downweighting logic where convexity defects remain;
6. optionally evaluate an SSVI-style global regularization layer if per-expiry SVI is unstable;
7. quantify repair bias by expiry, moneyness, liquidity bucket, and source partition;
8. select a final repaired surface candidate;
9. export a contract-level repaired calibration target for Notebook 12.

## Input philosophy

Notebook 10 produced several partitions with different reliability levels. This notebook treats them asymmetrically:

| Partition | Interpretation | Intended role |
|---|---:|---|
| `anchor` | highest-confidence core quotes | high-weight fit anchors |
| `strict` | clean static-arbitrage subset | conservative fitting backbone |
| `broad` | usable filtered market surface | main information set |
| `repair_queue` | informative but convexity-stressed rows | low-weight or repair-aware input |
| `excluded` | rows failing quality or consistency requirements | audit only, zero fit weight |

The fitting logic is therefore not simply “use everything” or “delete everything suspicious.” The intended discipline is:

$$
\text{anchor + strict}
\rightarrow
\text{safe skeleton},
$$

$$
\text{broad}
\rightarrow
\text{market shape information},
$$

$$
\text{repair queue}
\rightarrow
\text{low-weight diagnostic and repair input},
$$

$$
\text{excluded}
\rightarrow
\text{audit only}.
$$

## Candidate fitting modes

This notebook compares several candidate surfaces rather than trusting a single fit:

| Candidate | Description |
|---|---|
| `strict_svi` | SVI fitted only to strict rows |
| `weighted_broad_svi` | SVI fitted to anchor, strict, and broad rows with quality weights |
| `repair_aware_svi` | SVI fitted with repair-queue information and convexity penalties |
| `calendar_repaired_svi` | SVI surface after maturity monotonicity repair |
| `ssvi_regularized` | optional global regularization layer if local SVI slices are unstable |

The selected surface is not chosen by RMSE alone. A low-error fit that preserves or introduces arbitrage defects is not acceptable. The ranking criterion is:

$$
\text{surface quality}
=
\text{fit accuracy}
+
\text{static-arbitrage consistency}
+
\text{repair-bias control}
+
\text{parameter stability}.
$$

## Final handoff requirement

The output of this notebook must be a contract-level repaired calibration target, not only a smooth plotted grid. For each retained contract, the final handoff should include:

$$
K,\quad
T,\quad
F_T,\quad
k,\quad
r_T,\quad
D_T,\quad
\sigma_{\mathrm{raw}},\quad
\sigma_{\mathrm{repaired}},\quad
w_{\mathrm{raw}},\quad
w_{\mathrm{repaired}},
$$

plus price targets, source labels, repair amounts, fitting weights, and downstream eligibility flags.

The repaired implied volatility is converted back into a Black-Scholes price target using

$$
C_{\mathrm{repaired}}
=
BS\left(
F_T,
K,
T,
D_T,
\sigma_{\mathrm{repaired}}
\right).
$$

The resulting target is the appropriate input for the next notebook:

**`12_heston_bates_pricing_engine_and_calibration_to_repaired_surface.ipynb`**

## Allowed claims

This notebook may claim that:

- the raw Notebook 10 surface is usable but convexity-stressed;
- SVI/SSVI-style smoothing can produce a cleaner calibration target;
- repair bias can be measured and localized;
- a repaired surface is more defensible for downstream calibration than raw quote-level IVs.

This notebook may **not** claim that:

- the market surface is truly arbitrage-free;
- SVI or SSVI is the true market model;
- repaired prices are more correct than observed market prices;
- Heston/Bates calibration is validated;
- out-of-sample pricing performance has been established.

The notebook should end with one of the following statuses:

| Status | Meaning |
|---|---|
| `READY_FOR_12` | repaired surface passes all core checks |
| `READY_FOR_12_WITH_WARNINGS` | usable surface, but repair or coverage caveats remain |
| `REPAIR_ONLY_NOT_CALIBRATION_READY` | diagnostics useful, but handoff should not be calibrated |
| `FAILED_SURFACE_CONSTRUCTION` | no defensible repaired surface was produced |

In [1]:
# ------------------------------------------------------------
# Notebook 11 setup: environment, paths, run config, thresholds
# ------------------------------------------------------------

from __future__ import annotations

import json
import math
import os
import platform
import sys
import time
import warnings
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from IPython.display import display
from scipy.interpolate import PchipInterpolator
from scipy.optimize import least_squares, minimize
from scipy.stats import norm

# ------------------------------------------------------------
# Global notebook identity
# ------------------------------------------------------------

NOTEBOOK_ID = 11
NOTEBOOK_NAME = "11_arbitrage_aware_svi_ssvi_surface_repair_and_handoff"
RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_TAG = datetime.now(timezone.utc).strftime("n11_%Y%m%dT%H%M%SZ")

RANDOM_SEED = 20260705
RNG = np.random.default_rng(RANDOM_SEED)

# ------------------------------------------------------------
# Display / numerical behavior
# ------------------------------------------------------------

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.10f}")

np.set_printoptions(precision=10, suppress=False, linewidth=180)
np.seterr(all="warn")

warnings.filterwarnings("default")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

# ------------------------------------------------------------
# Project root resolution
# ------------------------------------------------------------
# Override this only if the notebook is launched outside the project folder.
# Use a raw string if assigning a Windows path:
#
# PROJECT_ROOT_OVERRIDE = Path(r"D:\Derivative Pricing Project v1.0+\V1.1")

PROJECT_ROOT_OVERRIDE: Path | None = None
DEFAULT_PROJECT_ROOT = Path(r"D:\Derivative Pricing Project v1.0+\V1.1")


def find_project_root() -> Path:
    """
    Resolve the project root without relying on unsafe backslash string literals.

    Priority:
    1. explicit PROJECT_ROOT_OVERRIDE
    2. DERIVATIVE_PRICING_PROJECT_ROOT environment variable
    3. current working directory and its parents
    4. default known project root
    """
    candidates: list[Path] = []

    if PROJECT_ROOT_OVERRIDE is not None:
        candidates.append(Path(PROJECT_ROOT_OVERRIDE).expanduser())

    env_root = os.getenv("DERIVATIVE_PRICING_PROJECT_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.append(DEFAULT_PROJECT_ROOT)

    seen: set[str] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        key = str(candidate).lower()
        if key in seen:
            continue
        seen.add(key)

        if (
            (candidate / "outputs").exists()
            or (candidate / "data").exists()
            or (candidate / "paths.txt").exists()
        ):
            return candidate.resolve()

    raise FileNotFoundError(
        "Could not resolve project root. Set PROJECT_ROOT_OVERRIDE to the V1.1 project directory."
    )


PROJECT_ROOT = find_project_root()

# ------------------------------------------------------------
# Notebook 10 inputs
# ------------------------------------------------------------

N10_NAME = "10_raw_static_arbitrage_diagnostics_for_spy_iv_surface"
N10_OUTPUT_DIR = PROJECT_ROOT / "outputs" / N10_NAME
N10_HANDOFF_DIR = N10_OUTPUT_DIR / "handoff"
N10_TABLE_DIR = N10_OUTPUT_DIR / "tables"
N10_MANIFEST_DIR = N10_OUTPUT_DIR / "manifest"
N10_PLOT_DIR = N10_OUTPUT_DIR / "plots"

N10_INPUTS = {
    "full_static_diagnostic_handoff": N10_HANDOFF_DIR / "n10_full_static_diagnostic_handoff_latest.parquet",
    "broad_handoff": N10_HANDOFF_DIR / "n10_to_n11_broad_handoff_latest.parquet",
    "strict_handoff": N10_HANDOFF_DIR / "n10_to_n11_strict_handoff_latest.parquet",
    "anchor_handoff": N10_HANDOFF_DIR / "n10_to_n11_anchor_handoff_latest.parquet",
    "repair_queue": N10_HANDOFF_DIR / "n10_to_n11_repair_queue_latest.parquet",
    "excluded_handoff": N10_HANDOFF_DIR / "n10_to_n11_excluded_handoff_latest.parquet",
    "handoff_inventory": N10_HANDOFF_DIR / "handoff_inventory_latest.csv",
    "handoff_by_expiry": N10_HANDOFF_DIR / "handoff_by_expiry_latest.csv",
    "handoff_by_bucket": N10_HANDOFF_DIR / "handoff_by_bucket_latest.csv",
    "final_status_table": N10_TABLE_DIR / "final_status_table_latest.csv",
    "final_validation_ledger": N10_TABLE_DIR / "final_validation_ledger_latest.csv",
}

# ------------------------------------------------------------
# Notebook 11 output layout
# ------------------------------------------------------------

N11_OUTPUT_DIR = PROJECT_ROOT / "outputs" / NOTEBOOK_NAME
N11_TABLE_DIR = N11_OUTPUT_DIR / "tables"
N11_HANDOFF_DIR = N11_OUTPUT_DIR / "handoff"
N11_MANIFEST_DIR = N11_OUTPUT_DIR / "manifest"
N11_PLOT_DIR = N11_OUTPUT_DIR / "plots"
N11_GRID_DIR = N11_OUTPUT_DIR / "surface_grids"
N11_PARAM_DIR = N11_OUTPUT_DIR / "svi_parameters"
N11_DIAGNOSTIC_DIR = N11_OUTPUT_DIR / "diagnostics"

N11_DIRS = {
    "output": N11_OUTPUT_DIR,
    "tables": N11_TABLE_DIR,
    "handoff": N11_HANDOFF_DIR,
    "manifest": N11_MANIFEST_DIR,
    "plots": N11_PLOT_DIR,
    "surface_grids": N11_GRID_DIR,
    "svi_parameters": N11_PARAM_DIR,
    "diagnostics": N11_DIAGNOSTIC_DIR,
}

for directory in N11_DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Surface repair configuration
# ------------------------------------------------------------

@dataclass(frozen=True)
class SurfaceRepairConfig:
    notebook_id: int = NOTEBOOK_ID
    notebook_name: str = NOTEBOOK_NAME
    run_tag: str = RUN_TAG
    random_seed: int = RANDOM_SEED

    # Minimum support requirements
    min_primary_expiries: int = 4
    min_rows_per_svi_slice: int = 8
    min_unique_strikes_per_slice: int = 8
    min_log_moneyness_width: float = 0.0800
    min_positive_weight_fraction: float = 0.8000

    # Repair queue controls
    repair_queue_base_weight: float = 0.2500
    broad_base_weight: float = 1.0000
    strict_base_weight: float = 2.0000
    anchor_base_weight: float = 4.0000
    excluded_base_weight: float = 0.0000
    max_repair_queue_weight_share: float = 0.5000

    # SVI parameter bounds
    svi_a_lower: float = -1.0000
    svi_a_upper: float = 5.0000
    svi_b_lower: float = 1.0e-8
    svi_b_upper: float = 5.0000
    svi_rho_lower: float = -0.9990
    svi_rho_upper: float = 0.9990
    svi_m_lower: float = -3.0000
    svi_m_upper: float = 3.0000
    svi_sigma_lower: float = 1.0e-6
    svi_sigma_upper: float = 5.0000

    # Fitting controls
    n_multistart: int = 64
    max_nfev: int = 20_000
    loss_function: str = "soft_l1"
    soft_l1_f_scale: float = 0.0100

    # Diagnostic grids
    k_grid_size: int = 181
    k_grid_pad: float = 0.0250

    # Static-arbitrage tolerances
    positive_total_variance_floor: float = 1.0e-10
    density_tolerance: float = -1.0e-7
    convexity_tolerance: float = -1.0e-7
    calendar_tolerance: float = -1.0e-8

    # Repair-bias warning bands in absolute IV points
    atm_abs_iv_repair_warn: float = 0.0150
    atm_abs_iv_repair_fail: float = 0.0500
    global_abs_iv_repair_warn: float = 0.0250
    global_abs_iv_repair_fail: float = 0.0750

    # Final handoff controls
    selected_surface_default: str = "weighted_broad_svi"
    final_status_default: str = "UNSET"


CONFIG = SurfaceRepairConfig()

SVI_BOUNDS_LOWER = np.array(
    [
        CONFIG.svi_a_lower,
        CONFIG.svi_b_lower,
        CONFIG.svi_rho_lower,
        CONFIG.svi_m_lower,
        CONFIG.svi_sigma_lower,
    ],
    dtype=float,
)

SVI_BOUNDS_UPPER = np.array(
    [
        CONFIG.svi_a_upper,
        CONFIG.svi_b_upper,
        CONFIG.svi_rho_upper,
        CONFIG.svi_m_upper,
        CONFIG.svi_sigma_upper,
    ],
    dtype=float,
)

SVI_PARAM_NAMES = ["a", "b", "rho", "m", "sigma"]

# ------------------------------------------------------------
# Lightweight artifact helpers
# ------------------------------------------------------------

def write_json(obj: dict[str, Any], path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, sort_keys=True, default=str), encoding="utf-8")
    return path


def write_csv(df: pd.DataFrame, path: Path, index: bool = False) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)
    return path


def write_parquet(df: pd.DataFrame, path: Path, index: bool = False) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=index)
    return path


def latest_path(path: Path) -> Path:
    """
    Convert a timestamped n11 artifact path into its latest equivalent.
    Example:
    n11_table_n11_YYYYMMDDTHHMMSSZ.csv -> n11_table_latest.csv
    """
    return path.with_name(path.name.replace(RUN_TAG, "latest"))


def write_table_pair(df: pd.DataFrame, timestamped_stem: str, directory: Path) -> dict[str, Path]:
    parquet_path = directory / f"{timestamped_stem}_{RUN_TAG}.parquet"
    csv_path = directory / f"{timestamped_stem}_{RUN_TAG}.csv"

    write_parquet(df, parquet_path, index=False)
    write_csv(df, csv_path, index=False)

    latest_parquet = latest_path(parquet_path)
    latest_csv = latest_path(csv_path)

    write_parquet(df, latest_parquet, index=False)
    write_csv(df, latest_csv, index=False)

    return {
        "parquet": parquet_path,
        "csv": csv_path,
        "latest_parquet": latest_parquet,
        "latest_csv": latest_csv,
    }


# ------------------------------------------------------------
# Setup summaries
# ------------------------------------------------------------

environment_summary = pd.DataFrame(
    [
        ("notebook_id", NOTEBOOK_ID),
        ("notebook_name", NOTEBOOK_NAME),
        ("run_timestamp_utc", RUN_TIMESTAMP_UTC),
        ("run_tag", RUN_TAG),
        ("project_root", str(PROJECT_ROOT)),
        ("python_version", sys.version.split()[0]),
        ("platform", platform.platform()),
        ("numpy_version", np.__version__),
        ("pandas_version", pd.__version__),
        ("scipy_version", scipy.__version__),
        ("random_seed", RANDOM_SEED),
    ],
    columns=["item", "value"],
)

directory_summary = pd.DataFrame(
    [(name, str(path), path.exists()) for name, path in N11_DIRS.items()],
    columns=["directory_key", "path", "exists"],
)

input_path_summary = pd.DataFrame(
    [(name, str(path), path.exists()) for name, path in N10_INPUTS.items()],
    columns=["input_key", "path", "exists"],
)

config_summary = pd.DataFrame(
    list(asdict(CONFIG).items()),
    columns=["parameter", "value"],
)

write_json(
    {
        "environment": dict(environment_summary.values),
        "config": asdict(CONFIG),
        "n10_inputs": {key: str(path) for key, path in N10_INPUTS.items()},
        "n11_dirs": {key: str(path) for key, path in N11_DIRS.items()},
    },
    N11_MANIFEST_DIR / f"n11_setup_config_{RUN_TAG}.json",
)

write_json(
    {
        "environment": dict(environment_summary.values),
        "config": asdict(CONFIG),
        "n10_inputs": {key: str(path) for key, path in N10_INPUTS.items()},
        "n11_dirs": {key: str(path) for key, path in N11_DIRS.items()},
    },
    N11_MANIFEST_DIR / "n11_setup_config_latest.json",
)

print("Notebook 11 setup complete.")
display(environment_summary)
display(directory_summary)
display(input_path_summary)
display(config_summary)

Notebook 11 setup complete.


,item,value
0,notebook_id,11
1,notebook_name,11_arbitrage_aware_svi_ssvi_surface_repair_and...
2,run_timestamp_utc,2026-07-05T23:56:29Z
3,run_tag,n11_20260705T235629Z
4,project_root,D:\Derivative Pricing Project v1.0+\V1.1
5,python_version,3.11.9
6,platform,Windows-10-10.0.26200-SP0
7,numpy_version,2.2.0
8,pandas_version,2.3.2
9,scipy_version,1.16.1


,directory_key,path,exists
0,output,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True
1,tables,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True
2,handoff,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True
3,manifest,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True
4,plots,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True
5,surface_grids,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True
6,svi_parameters,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True
7,diagnostics,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True


,input_key,path,exists
0,full_static_diagnostic_handoff,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,False
1,broad_handoff,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,False
2,strict_handoff,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,False
3,anchor_handoff,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,False
4,repair_queue,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,False
5,excluded_handoff,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,False
6,handoff_inventory,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,False
7,handoff_by_expiry,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,False
8,handoff_by_bucket,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,False
9,final_status_table,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,False


,parameter,value
0,notebook_id,11
1,notebook_name,11_arbitrage_aware_svi_ssvi_surface_repair_and...
2,run_tag,n11_20260705T235629Z
3,random_seed,20260705
4,min_primary_expiries,4
5,min_rows_per_svi_slice,8
6,min_unique_strikes_per_slice,8
7,min_log_moneyness_width,0.0800000000
8,min_positive_weight_fraction,0.8000000000
9,repair_queue_base_weight,0.2500000000


In [2]:
# ------------------------------------------------------------
# Notebook 11 cell 02: Notebook 10 artifact discovery and reconciliation
# ------------------------------------------------------------
# This cell repairs the brittle path assumptions from setup.
# It discovers Notebook 10 artifacts from:
#   1. paths.txt registry, if available
#   2. expected latest paths
#   3. timestamped glob fallback
#
# Do not proceed to SVI fitting until this cell passes.

import re
from collections import defaultdict

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def normalize_windows_path_string(value: str) -> Path:
    """
    Convert a path string from paths.txt into a Path safely.

    paths.txt often stores Windows paths as plain text. We do not evaluate
    those strings as Python literals because backslash sequences such as
    \\t, \\n, and \\f can corrupt paths.
    """
    cleaned = str(value).strip().strip('"').strip("'")
    return Path(cleaned)


def parse_paths_txt(paths_file: Path) -> dict[str, Path]:
    """
    Parse KEY = "path" assignments from paths.txt without executing it.
    """
    registry: dict[str, Path] = {}

    if not paths_file.exists():
        return registry

    assignment_pattern = re.compile(r"^\s*([A-Za-z_][A-Za-z0-9_]*)\s*=\s*(.+?)\s*$")

    for raw_line in paths_file.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw_line.strip()

        if not line or line.startswith("#"):
            continue

        match = assignment_pattern.match(line)
        if not match:
            continue

        key, value = match.groups()

        # Remove inline comments only if they occur outside the quoted body in normal usage.
        value = value.strip()
        if value.startswith(("'", '"')) and value.endswith(("'", '"')):
            registry[key] = normalize_windows_path_string(value)

    return registry


def newest_existing_match(patterns: list[str], search_roots: list[Path]) -> Path | None:
    """
    Find the newest existing file matching any pattern under the provided roots.
    """
    matches: list[Path] = []

    for root in search_roots:
        if not root.exists():
            continue

        for pattern in patterns:
            matches.extend(root.rglob(pattern))

    matches = [p for p in matches if p.exists() and p.is_file()]

    if not matches:
        return None

    return max(matches, key=lambda p: p.stat().st_mtime)


def choose_artifact(
    logical_name: str,
    registry_keys: list[str],
    expected_path: Path | None,
    fallback_patterns: list[str],
    search_roots: list[Path],
    registry: dict[str, Path],
) -> dict[str, Any]:
    """
    Resolve one artifact using registry -> expected path -> glob fallback.
    """
    attempts: list[tuple[str, str, bool]] = []

    for key in registry_keys:
        candidate = registry.get(key)
        if candidate is not None:
            exists = candidate.exists()
            attempts.append((f"paths.txt:{key}", str(candidate), exists))
            if exists:
                return {
                    "logical_name": logical_name,
                    "resolved_path": candidate,
                    "exists": True,
                    "source": f"paths.txt:{key}",
                    "attempts": attempts,
                }

    if expected_path is not None:
        exists = expected_path.exists()
        attempts.append(("expected_latest", str(expected_path), exists))
        if exists:
            return {
                "logical_name": logical_name,
                "resolved_path": expected_path,
                "exists": True,
                "source": "expected_latest",
                "attempts": attempts,
            }

    fallback = newest_existing_match(fallback_patterns, search_roots)
    if fallback is not None:
        attempts.append(("glob_fallback", str(fallback), True))
        return {
            "logical_name": logical_name,
            "resolved_path": fallback,
            "exists": True,
            "source": "glob_fallback",
            "attempts": attempts,
        }

    attempts.append(("glob_fallback", " | ".join(fallback_patterns), False))
    return {
        "logical_name": logical_name,
        "resolved_path": None,
        "exists": False,
        "source": "unresolved",
        "attempts": attempts,
    }


def flatten_attempts(resolution_records: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []

    for record in resolution_records:
        for source, path, exists in record["attempts"]:
            rows.append(
                {
                    "logical_name": record["logical_name"],
                    "attempt_source": source,
                    "attempt_path": path,
                    "exists": bool(exists),
                    "selected": (
                        record["resolved_path"] is not None
                        and str(record["resolved_path"]) == path
                        and record["source"] == source
                    ),
                }
            )

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Registry discovery
# ------------------------------------------------------------

PATHS_TXT_CANDIDATES = [
    PROJECT_ROOT / "paths.txt",
    Path.cwd() / "paths.txt",
    *[parent / "paths.txt" for parent in Path.cwd().resolve().parents],
]

PATHS_TXT = next((p for p in PATHS_TXT_CANDIDATES if p.exists()), None)
PATH_REGISTRY = parse_paths_txt(PATHS_TXT) if PATHS_TXT is not None else {}

print(f"paths.txt found: {PATHS_TXT if PATHS_TXT is not None else 'NO'}")
print(f"paths.txt keys parsed: {len(PATH_REGISTRY)}")

# ------------------------------------------------------------
# Search roots
# ------------------------------------------------------------

N10_SEARCH_ROOTS = [
    PROJECT_ROOT / "outputs",
    PROJECT_ROOT / "data",
    PROJECT_ROOT,
]

N10_SEARCH_ROOTS = [p for p in N10_SEARCH_ROOTS if p.exists()]

# ------------------------------------------------------------
# Expected path fallback from original setup
# ------------------------------------------------------------

EXPECTED_N10_LATEST = {
    "full_static_diagnostic_handoff": N10_HANDOFF_DIR / "n10_full_static_diagnostic_handoff_latest.parquet",
    "broad_handoff": N10_HANDOFF_DIR / "n10_to_n11_broad_handoff_latest.parquet",
    "strict_handoff": N10_HANDOFF_DIR / "n10_to_n11_strict_handoff_latest.parquet",
    "anchor_handoff": N10_HANDOFF_DIR / "n10_to_n11_anchor_handoff_latest.parquet",
    "repair_queue": N10_HANDOFF_DIR / "n10_to_n11_repair_queue_latest.parquet",
    "excluded_handoff": N10_HANDOFF_DIR / "n10_to_n11_excluded_handoff_latest.parquet",
    "handoff_inventory": N10_HANDOFF_DIR / "handoff_inventory_latest.csv",
    "handoff_by_expiry": N10_HANDOFF_DIR / "handoff_by_expiry_latest.csv",
    "handoff_by_bucket": N10_HANDOFF_DIR / "handoff_by_bucket_latest.csv",
    "final_status_table": N10_TABLE_DIR / "final_status_table_latest.csv",
    "final_validation_ledger": N10_TABLE_DIR / "final_validation_ledger_latest.csv",
}

# ------------------------------------------------------------
# Resolution specification
# ------------------------------------------------------------

RESOLUTION_SPEC = {
    "full_static_diagnostic_handoff": {
        "registry_keys": [
            "N10_FULL_STATIC_DIAGNOSTIC_HANDOFF_LATEST",
            "N10_FULL_STATIC_DIAGNOSTIC_HANDOFF",
        ],
        "patterns": [
            "n10_full_static_diagnostic_handoff_latest.parquet",
            "n10_full_static_diagnostic_handoff_n10_*.parquet",
            "*full_static*diagnostic*handoff*.parquet",
        ],
    },
    "broad_handoff": {
        "registry_keys": [
            "NOTEBOOK_11_PRIMARY_INPUT_BROAD",
            "NOTEBOOK_11_DEFAULT_INPUT",
            "N10_N11_BROAD_HANDOFF_LATEST",
            "N10_N11_BROAD_HANDOFF",
        ],
        "patterns": [
            "n10_to_n11_broad_handoff_latest.parquet",
            "n10_to_n11_broad_handoff_n10_*.parquet",
            "*broad*handoff*.parquet",
        ],
    },
    "strict_handoff": {
        "registry_keys": [
            "NOTEBOOK_11_PRIMARY_INPUT_STRICT",
            "N10_N11_STRICT_HANDOFF_LATEST",
            "N10_N11_STRICT_HANDOFF",
        ],
        "patterns": [
            "n10_to_n11_strict_handoff_latest.parquet",
            "n10_to_n11_strict_handoff_n10_*.parquet",
            "*strict*handoff*.parquet",
        ],
    },
    "anchor_handoff": {
        "registry_keys": [
            "NOTEBOOK_11_PRIMARY_INPUT_ANCHOR",
            "N10_N11_ANCHOR_HANDOFF_LATEST",
            "N10_N11_ANCHOR_HANDOFF",
        ],
        "patterns": [
            "n10_to_n11_anchor_handoff_latest.parquet",
            "n10_to_n11_anchor_handoff_n10_*.parquet",
            "*anchor*handoff*.parquet",
        ],
    },
    "repair_queue": {
        "registry_keys": [
            "NOTEBOOK_11_REPAIR_QUEUE_INPUT",
            "N10_N11_REPAIR_QUEUE_LATEST",
            "N10_N11_REPAIR_QUEUE",
        ],
        "patterns": [
            "n10_to_n11_repair_queue_latest.parquet",
            "n10_to_n11_repair_queue_n10_*.parquet",
            "*repair*queue*.parquet",
        ],
    },
    "excluded_handoff": {
        "registry_keys": [
            "NOTEBOOK_11_EXCLUDED_INPUT",
            "N10_N11_EXCLUDED_HANDOFF_LATEST",
            "N10_N11_EXCLUDED_HANDOFF",
        ],
        "patterns": [
            "n10_to_n11_excluded_handoff_latest.parquet",
            "n10_to_n11_excluded_handoff_n10_*.parquet",
            "*excluded*handoff*.parquet",
        ],
    },
    "handoff_inventory": {
        "registry_keys": [
            "N10_HANDOFF_INVENTORY_LATEST_CSV",
            "N10_HANDOFF_INVENTORY_CSV",
        ],
        "patterns": [
            "handoff_inventory_latest.csv",
            "handoff_inventory_n10_*.csv",
            "*handoff*inventory*.csv",
        ],
    },
    "handoff_by_expiry": {
        "registry_keys": [
            "N10_HANDOFF_BY_EXPIRY_LATEST_CSV",
            "N10_HANDOFF_BY_EXPIRY_CSV",
        ],
        "patterns": [
            "handoff_by_expiry_latest.csv",
            "handoff_by_expiry_n10_*.csv",
            "*handoff*expiry*.csv",
        ],
    },
    "handoff_by_bucket": {
        "registry_keys": [
            "N10_HANDOFF_BY_BUCKET_LATEST_CSV",
            "N10_HANDOFF_BY_BUCKET_CSV",
        ],
        "patterns": [
            "handoff_by_bucket_latest.csv",
            "handoff_by_bucket_n10_*.csv",
            "*handoff*bucket*.csv",
        ],
    },
    "final_status_table": {
        "registry_keys": [
            "N10_FINAL_STATUS_TABLE_LATEST",
            "N10_FINAL_STATUS_TABLE",
        ],
        "patterns": [
            "final_status_table_latest.csv",
            "final_status_table_n10_*.csv",
            "*final*status*table*.csv",
        ],
    },
    "final_validation_ledger": {
        "registry_keys": [
            "N10_FINAL_VALIDATION_LEDGER_LATEST",
            "N10_FINAL_VALIDATION_LEDGER",
        ],
        "patterns": [
            "final_validation_ledger_latest.csv",
            "final_validation_ledger_n10_*.csv",
            "*final*validation*ledger*.csv",
        ],
    },
}

# ------------------------------------------------------------
# Resolve actual Notebook 10 inputs
# ------------------------------------------------------------

resolution_records: list[dict[str, Any]] = []

for logical_name, spec in RESOLUTION_SPEC.items():
    record = choose_artifact(
        logical_name=logical_name,
        registry_keys=spec["registry_keys"],
        expected_path=EXPECTED_N10_LATEST.get(logical_name),
        fallback_patterns=spec["patterns"],
        search_roots=N10_SEARCH_ROOTS,
        registry=PATH_REGISTRY,
    )
    resolution_records.append(record)

n10_resolution_summary = pd.DataFrame(
    [
        {
            "logical_name": r["logical_name"],
            "resolved_path": str(r["resolved_path"]) if r["resolved_path"] is not None else None,
            "exists": r["exists"],
            "source": r["source"],
        }
        for r in resolution_records
    ]
)

n10_resolution_attempts = flatten_attempts(resolution_records)

# Replace N10_INPUTS with resolved paths only after discovery.
N10_INPUTS_RESOLVED = {
    r["logical_name"]: r["resolved_path"]
    for r in resolution_records
    if r["resolved_path"] is not None and r["exists"]
}

required_n10_core_inputs = [
    "broad_handoff",
    "strict_handoff",
    "anchor_handoff",
    "repair_queue",
    "excluded_handoff",
    "final_status_table",
]

missing_required_n10_core_inputs = [
    key for key in required_n10_core_inputs if key not in N10_INPUTS_RESOLVED
]

N10_ARTIFACT_DISCOVERY_PASS = len(missing_required_n10_core_inputs) == 0

# Persist discovery audit.
write_table_pair(
    n10_resolution_summary,
    "n11_n10_artifact_resolution_summary",
    N11_TABLE_DIR,
)

write_table_pair(
    n10_resolution_attempts,
    "n11_n10_artifact_resolution_attempts",
    N11_TABLE_DIR,
)

write_json(
    {
        "run_tag": RUN_TAG,
        "project_root": str(PROJECT_ROOT),
        "paths_txt": str(PATHS_TXT) if PATHS_TXT is not None else None,
        "paths_txt_keys_parsed": len(PATH_REGISTRY),
        "artifact_discovery_pass": N10_ARTIFACT_DISCOVERY_PASS,
        "missing_required_core_inputs": missing_required_n10_core_inputs,
        "resolved_inputs": {k: str(v) for k, v in N10_INPUTS_RESOLVED.items()},
    },
    N11_MANIFEST_DIR / f"n11_n10_artifact_discovery_{RUN_TAG}.json",
)

write_json(
    {
        "run_tag": RUN_TAG,
        "project_root": str(PROJECT_ROOT),
        "paths_txt": str(PATHS_TXT) if PATHS_TXT is not None else None,
        "paths_txt_keys_parsed": len(PATH_REGISTRY),
        "artifact_discovery_pass": N10_ARTIFACT_DISCOVERY_PASS,
        "missing_required_core_inputs": missing_required_n10_core_inputs,
        "resolved_inputs": {k: str(v) for k, v in N10_INPUTS_RESOLVED.items()},
    },
    N11_MANIFEST_DIR / "n11_n10_artifact_discovery_latest.json",
)

print("Notebook 10 artifact discovery complete.")
print(f"Artifact discovery pass: {N10_ARTIFACT_DISCOVERY_PASS}")

if missing_required_n10_core_inputs:
    print("Missing required core inputs:")
    for item in missing_required_n10_core_inputs:
        print(f"  - {item}")

display(n10_resolution_summary)
display(n10_resolution_attempts)

if not N10_ARTIFACT_DISCOVERY_PASS:
    raise FileNotFoundError(
        "Notebook 10 core handoff artifacts were not resolved. "
        "Do not continue to SVI fitting until Notebook 10 outputs are available or paths.txt points to valid files."
    )

paths.txt found: D:\Derivative Pricing Project v1.0+\V1.1\paths.txt
paths.txt keys parsed: 57
Notebook 10 artifact discovery complete.
Artifact discovery pass: True


,logical_name,resolved_path,exists,source
0,full_static_diagnostic_handoff,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True,glob_fallback
1,broad_handoff,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True,glob_fallback
2,strict_handoff,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True,glob_fallback
3,anchor_handoff,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True,glob_fallback
4,repair_queue,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True,glob_fallback
5,excluded_handoff,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True,glob_fallback
6,handoff_inventory,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True,glob_fallback
7,handoff_by_expiry,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True,glob_fallback
8,handoff_by_bucket,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True,glob_fallback
9,final_status_table,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True,glob_fallback


,logical_name,attempt_source,attempt_path,exists,selected
0,full_static_diagnostic_handoff,paths.txt:N10_FULL_STATIC_DIAGNOSTIC_HANDOFF_L...,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,False,False
1,full_static_diagnostic_handoff,paths.txt:N10_FULL_STATIC_DIAGNOSTIC_HANDOFF,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,False,False
2,full_static_diagnostic_handoff,expected_latest,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,False,False
3,full_static_diagnostic_handoff,glob_fallback,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True,True
4,broad_handoff,paths.txt:NOTEBOOK_11_PRIMARY_INPUT_BROAD,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,False,False
5,broad_handoff,paths.txt:NOTEBOOK_11_DEFAULT_INPUT,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,False,False
6,broad_handoff,paths.txt:N10_N11_BROAD_HANDOFF_LATEST,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,False,False
7,broad_handoff,paths.txt:N10_N11_BROAD_HANDOFF,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,False,False
8,broad_handoff,expected_latest,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,False,False
9,broad_handoff,glob_fallback,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True,True


In [3]:
# ------------------------------------------------------------
# Notebook 11 cell 03: Native Notebook 10 schema load and canonicalization
# ------------------------------------------------------------
# This cell uses the ACTUAL Notebook 10 handoff schema:
#
#   tau_years                 -> time to maturity
#   selected_iv               -> implied volatility target
#   total_variance            -> total variance target
#   forward                   -> maturity-matched forward
#   discount_factor           -> discount factor
#   bid_price / ask_price     -> quote bounds
#   mid_price                 -> quote midpoint
#   diagnostic_weight         -> upstream surface-quality weight
#   has_convexity_violation   -> butterfly / convexity flag
#
# Do not alias this schema as if it came from a generic option-chain vendor.
# Notebook 10 already canonicalized the market data. Notebook 11 should consume
# the Notebook 10 handoff contract directly.

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Fallbacks in case this cell is run after kernel restart
# ------------------------------------------------------------

if "N10_INPUTS_RESOLVED" not in globals():
    raise NameError(
        "N10_INPUTS_RESOLVED is missing. Run the Notebook 10 artifact discovery cell first."
    )

if "RUN_TAG" not in globals():
    RUN_TAG = pd.Timestamp.utcnow().strftime("n11_%Y%m%dT%H%M%SZ")

if "N11_TABLE_DIR" not in globals():
    N11_TABLE_DIR = Path.cwd() / "outputs" / "11_arbitrage_aware_svi_ssvi_surface_repair_and_handoff" / "tables"
    N11_TABLE_DIR.mkdir(parents=True, exist_ok=True)

if "N11_HANDOFF_DIR" not in globals():
    N11_HANDOFF_DIR = Path.cwd() / "outputs" / "11_arbitrage_aware_svi_ssvi_surface_repair_and_handoff" / "handoff"
    N11_HANDOFF_DIR.mkdir(parents=True, exist_ok=True)

if "N11_MANIFEST_DIR" not in globals():
    N11_MANIFEST_DIR = Path.cwd() / "outputs" / "11_arbitrage_aware_svi_ssvi_surface_repair_and_handoff" / "manifest"
    N11_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

if "CONFIG" not in globals():
    class _ConfigFallback:
        min_primary_expiries = 4
        min_rows_per_svi_slice = 8
        min_unique_strikes_per_slice = 8
        min_log_moneyness_width = 0.0800
        min_positive_weight_fraction = 0.8000
        anchor_base_weight = 4.0000
        strict_base_weight = 2.0000
        broad_base_weight = 1.0000
        repair_queue_base_weight = 0.2500
        excluded_base_weight = 0.0000
        positive_total_variance_floor = 1.0e-10

    CONFIG = _ConfigFallback()


def _write_csv(df: pd.DataFrame, path: Path, index: bool = False) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)
    return path


def _write_parquet(df: pd.DataFrame, path: Path, index: bool = False) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=index)
    return path


def _write_table_pair(df: pd.DataFrame, stem: str, directory: Path) -> dict[str, Path]:
    csv_path = directory / f"{stem}_{RUN_TAG}.csv"
    pq_path = directory / f"{stem}_{RUN_TAG}.parquet"
    latest_csv_path = directory / f"{stem}_latest.csv"
    latest_pq_path = directory / f"{stem}_latest.parquet"

    _write_csv(df, csv_path, index=False)
    _write_parquet(df, pq_path, index=False)
    _write_csv(df, latest_csv_path, index=False)
    _write_parquet(df, latest_pq_path, index=False)

    return {
        "csv": csv_path,
        "parquet": pq_path,
        "latest_csv": latest_csv_path,
        "latest_parquet": latest_pq_path,
    }


# ------------------------------------------------------------
# Load Notebook 10 artifacts
# ------------------------------------------------------------

def read_artifact(path: Path) -> pd.DataFrame:
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)

    if suffix == ".csv":
        return pd.read_csv(path)

    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)

    raise ValueError(f"Unsupported artifact type: {path}")


N10_ARTIFACTS = {}

for logical_name, path in N10_INPUTS_RESOLVED.items():
    N10_ARTIFACTS[logical_name] = read_artifact(path)

print("Notebook 10 artifacts loaded:")
for name, df in N10_ARTIFACTS.items():
    print(f"  {name:<35} rows={len(df):>8,} cols={df.shape[1]:>4}")


# ------------------------------------------------------------
# Actual Notebook 10 schema contract
# ------------------------------------------------------------

N10_NATIVE_COLUMN_CONTRACT = {
    "row_id": ["n10_row_id"],
    "expiry": ["expiry"],
    "expiry_rank": ["expiry_rank"],
    "dte_calendar": ["dte_calendar"],
    "time_to_maturity": ["tau_years"],
    "maturity_bucket": ["maturity_bucket"],
    "option_type": ["option_type"],
    "strike": ["strike"],
    "spot": ["spot"],
    "forward": ["forward"],
    "discount_factor": ["discount_factor"],
    "log_moneyness": ["log_moneyness"],
    "abs_log_moneyness": ["abs_log_moneyness"],
    "moneyness_bucket": ["moneyness_bucket_n10"],
    "implied_vol": ["selected_iv"],
    "total_variance": ["total_variance"],
    "market_price": ["market_selected_price"],
    "call_equiv_price": ["market_call_equiv_price"],
    "put_equiv_price": ["market_put_equiv_price"],
    "bsm_price": ["bsm_selected_price"],
    "bid": ["bid_price"],
    "ask": ["ask_price"],
    "mid": ["mid_price"],
    "spread_abs": ["price_spread"],
    "spread_pct_mid": ["relative_price_spread"],
    "iv_bid": ["iv_bid"],
    "iv_ask": ["iv_ask"],
    "iv_mid": ["iv_mid"],
    "iv_spread": ["iv_spread"],
    "iv_uncertainty_width": ["relative_iv_uncertainty_width"],
    "diagnostic_weight": ["diagnostic_weight"],
    "quality_score": ["quality_score"],
    "surface_row_source": ["surface_row_source"],
    "quote_date": ["quote_date"],
    "snapshot_date": ["snapshot_date"],
    "underlying": ["underlying_symbol"],
    "contract_id": ["contract_symbol"],
    "source_row_id": ["source_row_id"],
    "n09_row_id": ["n09_row_id"],
    "pointwise_violation_count": ["pointwise_violation_count"],
    "monotonicity_violation_count": ["monotonicity_violation_count"],
    "convexity_violation_count": ["convexity_violation_count"],
    "calendar_violation_count": ["calendar_violation_count"],
    "total_static_violation_count": ["total_static_violation_count"],
    "max_pointwise_violation_amount": ["max_pointwise_violation_amount"],
    "max_monotonicity_violation_amount": ["max_monotonicity_violation_amount"],
    "max_convexity_violation_amount": ["max_convexity_violation_amount"],
    "max_calendar_violation_amount": ["max_calendar_violation_amount"],
    "max_negative_density_proxy_amount": ["max_negative_density_proxy_amount"],
    "pointwise_flag": ["has_pointwise_violation"],
    "monotonicity_flag": ["has_monotonicity_violation"],
    "butterfly_flag": ["has_convexity_violation"],
    "calendar_flag": ["has_calendar_violation"],
    "static_arbitrage_flag": ["has_any_static_arbitrage_flag"],
    "max_static_severity": ["max_static_severity"],
    "max_static_severity_rank": ["max_static_severity_rank"],
    "iv_roughness_flag": ["touched_by_any_iv_roughness_flag"],
    "high_iv_slope_flag": ["touched_by_high_iv_slope"],
    "high_iv_jump_flag": ["touched_by_high_iv_jump_vs_uncertainty"],
    "high_iv_curvature_flag": ["touched_by_high_iv_curvature"],
    "high_local_iv_range_flag": ["touched_by_high_local_iv_range_vs_uncertainty"],
    "raw_static_arbitrage_score": ["raw_static_arbitrage_score"],
    "n10_broad_candidate": ["n11_broad_candidate"],
    "n10_strict_candidate": ["n11_strict_candidate"],
    "n10_anchor_candidate": ["n11_anchor_candidate"],
    "n10_exclusion_reason": ["n11_exclusion_reason"],
}


def resolve_native_column(df: pd.DataFrame, aliases: list[str]) -> str | None:
    columns = list(df.columns)
    normalized_to_original = {str(c).strip().lower(): c for c in columns}

    for alias in aliases:
        key = str(alias).strip().lower()
        if key in normalized_to_original:
            return normalized_to_original[key]

    return None


def get_col(df: pd.DataFrame, canonical_name: str, default=np.nan) -> pd.Series:
    aliases = N10_NATIVE_COLUMN_CONTRACT.get(canonical_name, [])
    resolved = resolve_native_column(df, aliases)

    if resolved is None:
        return pd.Series(default, index=df.index)

    return df[resolved]


def to_numeric_series(s: pd.Series, default=np.nan) -> pd.Series:
    out = pd.to_numeric(s, errors="coerce")
    if not (isinstance(default, float) and np.isnan(default)):
        out = out.fillna(default)
    return out


def to_bool_series(s: pd.Series, default=False) -> pd.Series:
    if s is None:
        return pd.Series(default)

    if s.dtype == bool:
        return s.fillna(default).astype(bool)

    text = s.astype(str).str.strip().str.lower()
    true_values = {"true", "1", "yes", "y", "t"}
    false_values = {"false", "0", "no", "n", "f", "", "nan", "none", "<na>"}

    out = pd.Series(default, index=s.index, dtype=bool)
    out.loc[text.isin(true_values)] = True
    out.loc[text.isin(false_values)] = False

    return out


# ------------------------------------------------------------
# Column-resolution audit
# ------------------------------------------------------------

schema_rows = []

for table_name, df in N10_ARTIFACTS.items():
    if table_name not in {
        "full_static_diagnostic_handoff",
        "broad_handoff",
        "strict_handoff",
        "anchor_handoff",
        "repair_queue",
        "excluded_handoff",
    }:
        continue

    for canonical_name, aliases in N10_NATIVE_COLUMN_CONTRACT.items():
        resolved = resolve_native_column(df, aliases)
        schema_rows.append(
            {
                "table_name": table_name,
                "canonical_name": canonical_name,
                "resolved_column": resolved,
                "resolved": resolved is not None,
                "aliases_tried": " | ".join(aliases),
            }
        )

n11_native_schema_resolution = pd.DataFrame(schema_rows)

_write_table_pair(
    n11_native_schema_resolution,
    "n11_native_schema_resolution",
    N11_TABLE_DIR,
)


# ------------------------------------------------------------
# Canonicalize each partition
# ------------------------------------------------------------

PARTITION_ORDER = [
    "full_static_diagnostic_handoff",
    "broad_handoff",
    "strict_handoff",
    "anchor_handoff",
    "repair_queue",
    "excluded_handoff",
]

PARTITION_LABELS = {
    "full_static_diagnostic_handoff": "full",
    "broad_handoff": "broad",
    "strict_handoff": "strict",
    "anchor_handoff": "anchor",
    "repair_queue": "repair",
    "excluded_handoff": "excluded",
}


def canonicalize_n10_partition(df: pd.DataFrame, partition_name: str) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    out["source_partition"] = partition_name
    out["partition_label"] = PARTITION_LABELS.get(partition_name, partition_name)

    out["n10_row_id"] = get_col(df, "row_id")
    out["expiry"] = pd.to_datetime(get_col(df, "expiry"), errors="coerce")
    out["expiry_rank"] = to_numeric_series(get_col(df, "expiry_rank"))
    out["dte_calendar"] = to_numeric_series(get_col(df, "dte_calendar"))
    out["tau_years"] = to_numeric_series(get_col(df, "time_to_maturity"))
    out["maturity_bucket"] = get_col(df, "maturity_bucket")
    out["option_type"] = get_col(df, "option_type").astype(str).str.lower().str.strip()

    out["strike"] = to_numeric_series(get_col(df, "strike"))
    out["spot"] = to_numeric_series(get_col(df, "spot"))
    out["forward"] = to_numeric_series(get_col(df, "forward"))
    out["discount_factor"] = to_numeric_series(get_col(df, "discount_factor"))

    out["log_moneyness"] = to_numeric_series(get_col(df, "log_moneyness"))
    missing_k = ~np.isfinite(out["log_moneyness"])
    can_derive_k = (
        missing_k
        & np.isfinite(out["strike"])
        & np.isfinite(out["forward"])
        & (out["strike"] > 0)
        & (out["forward"] > 0)
    )
    out.loc[can_derive_k, "log_moneyness"] = np.log(
        out.loc[can_derive_k, "strike"] / out.loc[can_derive_k, "forward"]
    )

    out["abs_log_moneyness"] = to_numeric_series(get_col(df, "abs_log_moneyness"))
    missing_abs_k = ~np.isfinite(out["abs_log_moneyness"])
    out.loc[missing_abs_k, "abs_log_moneyness"] = out.loc[missing_abs_k, "log_moneyness"].abs()

    out["moneyness_bucket"] = get_col(df, "moneyness_bucket")

    out["selected_iv"] = to_numeric_series(get_col(df, "implied_vol"))
    out["total_variance"] = to_numeric_series(get_col(df, "total_variance"))

    missing_iv = (
        ~np.isfinite(out["selected_iv"])
        & np.isfinite(out["total_variance"])
        & np.isfinite(out["tau_years"])
        & (out["total_variance"] > 0)
        & (out["tau_years"] > 0)
    )
    out.loc[missing_iv, "selected_iv"] = np.sqrt(
        out.loc[missing_iv, "total_variance"] / out.loc[missing_iv, "tau_years"]
    )

    missing_w = (
        ~np.isfinite(out["total_variance"])
        & np.isfinite(out["selected_iv"])
        & np.isfinite(out["tau_years"])
        & (out["selected_iv"] > 0)
        & (out["tau_years"] > 0)
    )
    out.loc[missing_w, "total_variance"] = (
        out.loc[missing_w, "selected_iv"] ** 2
        * out.loc[missing_w, "tau_years"]
    )

    out["market_selected_price"] = to_numeric_series(get_col(df, "market_price"))
    out["market_call_equiv_price"] = to_numeric_series(get_col(df, "call_equiv_price"))
    out["market_put_equiv_price"] = to_numeric_series(get_col(df, "put_equiv_price"))
    out["bsm_selected_price"] = to_numeric_series(get_col(df, "bsm_price"))

    out["bid_price"] = to_numeric_series(get_col(df, "bid"))
    out["ask_price"] = to_numeric_series(get_col(df, "ask"))
    out["mid_price"] = to_numeric_series(get_col(df, "mid"))
    out["price_spread"] = to_numeric_series(get_col(df, "spread_abs"))
    out["relative_price_spread"] = to_numeric_series(get_col(df, "spread_pct_mid"))

    out["iv_bid"] = to_numeric_series(get_col(df, "iv_bid"))
    out["iv_ask"] = to_numeric_series(get_col(df, "iv_ask"))
    out["iv_mid"] = to_numeric_series(get_col(df, "iv_mid"))
    out["iv_spread"] = to_numeric_series(get_col(df, "iv_spread"))
    out["relative_iv_uncertainty_width"] = to_numeric_series(get_col(df, "iv_uncertainty_width"))

    out["diagnostic_weight"] = to_numeric_series(get_col(df, "diagnostic_weight"), default=1.0)
    out["quality_score"] = to_numeric_series(get_col(df, "quality_score"))

    out["surface_row_source"] = get_col(df, "surface_row_source")
    out["quote_date"] = pd.to_datetime(get_col(df, "quote_date"), errors="coerce")
    out["snapshot_date"] = pd.to_datetime(get_col(df, "snapshot_date"), errors="coerce")
    out["underlying_symbol"] = get_col(df, "underlying")
    out["contract_symbol"] = get_col(df, "contract_id")
    out["source_row_id"] = get_col(df, "source_row_id")
    out["n09_row_id"] = get_col(df, "n09_row_id")

    out["pointwise_violation_count"] = to_numeric_series(get_col(df, "pointwise_violation_count"), default=0.0)
    out["monotonicity_violation_count"] = to_numeric_series(get_col(df, "monotonicity_violation_count"), default=0.0)
    out["convexity_violation_count"] = to_numeric_series(get_col(df, "convexity_violation_count"), default=0.0)
    out["calendar_violation_count"] = to_numeric_series(get_col(df, "calendar_violation_count"), default=0.0)
    out["total_static_violation_count"] = to_numeric_series(get_col(df, "total_static_violation_count"), default=0.0)

    out["max_pointwise_violation_amount"] = to_numeric_series(get_col(df, "max_pointwise_violation_amount"), default=0.0)
    out["max_monotonicity_violation_amount"] = to_numeric_series(get_col(df, "max_monotonicity_violation_amount"), default=0.0)
    out["max_convexity_violation_amount"] = to_numeric_series(get_col(df, "max_convexity_violation_amount"), default=0.0)
    out["max_calendar_violation_amount"] = to_numeric_series(get_col(df, "max_calendar_violation_amount"), default=0.0)
    out["max_negative_density_proxy_amount"] = to_numeric_series(get_col(df, "max_negative_density_proxy_amount"), default=0.0)

    out["has_pointwise_violation"] = to_bool_series(get_col(df, "pointwise_flag"), default=False)
    out["has_monotonicity_violation"] = to_bool_series(get_col(df, "monotonicity_flag"), default=False)
    out["has_convexity_violation"] = to_bool_series(get_col(df, "butterfly_flag"), default=False)
    out["has_calendar_violation"] = to_bool_series(get_col(df, "calendar_flag"), default=False)
    out["has_any_static_arbitrage_flag"] = to_bool_series(get_col(df, "static_arbitrage_flag"), default=False)

    out["max_static_severity"] = get_col(df, "max_static_severity")
    out["max_static_severity_rank"] = to_numeric_series(get_col(df, "max_static_severity_rank"))

    out["touched_by_any_iv_roughness_flag"] = to_bool_series(get_col(df, "iv_roughness_flag"), default=False)
    out["touched_by_high_iv_slope"] = to_bool_series(get_col(df, "high_iv_slope_flag"), default=False)
    out["touched_by_high_iv_jump_vs_uncertainty"] = to_bool_series(get_col(df, "high_iv_jump_flag"), default=False)
    out["touched_by_high_iv_curvature"] = to_bool_series(get_col(df, "high_iv_curvature_flag"), default=False)
    out["touched_by_high_local_iv_range_vs_uncertainty"] = to_bool_series(get_col(df, "high_local_iv_range_flag"), default=False)

    out["raw_static_arbitrage_score"] = to_numeric_series(get_col(df, "raw_static_arbitrage_score"))

    out["n10_broad_candidate"] = to_bool_series(get_col(df, "n10_broad_candidate"), default=False)
    out["n10_strict_candidate"] = to_bool_series(get_col(df, "n10_strict_candidate"), default=False)
    out["n10_anchor_candidate"] = to_bool_series(get_col(df, "n10_anchor_candidate"), default=False)
    out["n10_exclusion_reason"] = get_col(df, "n10_exclusion_reason").astype(str)

    # Derived risk-free rate from discount factor:
    #
    #     D(T) = exp(-rT)
    #     r    = -log(D(T)) / T
    #
    out["risk_free_rate"] = np.nan
    can_derive_r = (
        np.isfinite(out["discount_factor"])
        & np.isfinite(out["tau_years"])
        & (out["discount_factor"] > 0)
        & (out["tau_years"] > 0)
    )
    out.loc[can_derive_r, "risk_free_rate"] = -np.log(
        out.loc[can_derive_r, "discount_factor"]
    ) / out.loc[can_derive_r, "tau_years"]

    # Notebook 10 does not need to carry q if the forward and discount factor
    # are already present. Keep q as explicit zero placeholder for downstream
    # Black-Scholes reconstruction.
    out["dividend_yield"] = 0.0

    # Stable contract key.
    key_source = out["n10_row_id"].astype(str)
    fallback_key = (
        out["expiry"].astype(str)
        + "|"
        + out["option_type"].astype(str)
        + "|"
        + out["strike"].round(8).astype(str)
    )
    out["contract_key"] = np.where(
        key_source.notna() & (key_source != "nan") & (key_source != "<NA>"),
        key_source,
        fallback_key,
    )

    return out.reset_index(drop=True)


canonical_partitions = {}

for partition_name in PARTITION_ORDER:
    if partition_name in N10_ARTIFACTS:
        canonical_partitions[partition_name] = canonicalize_n10_partition(
            N10_ARTIFACTS[partition_name],
            partition_name,
        )


# ------------------------------------------------------------
# Build one row per contract with partition memberships
# ------------------------------------------------------------

membership_frames = []

for partition_name, df in canonical_partitions.items():
    if partition_name == "full_static_diagnostic_handoff":
        continue

    tmp = df[["contract_key"]].copy()
    tmp[f"in_{PARTITION_LABELS[partition_name]}"] = True
    membership_frames.append(tmp.drop_duplicates("contract_key"))

all_memberships = None

for frame in membership_frames:
    if all_memberships is None:
        all_memberships = frame.copy()
    else:
        all_memberships = all_memberships.merge(frame, on="contract_key", how="outer")

if all_memberships is None:
    all_memberships = pd.DataFrame(columns=["contract_key"])

for col in ["in_broad", "in_strict", "in_anchor", "in_repair", "in_excluded"]:
    if col not in all_memberships.columns:
        all_memberships[col] = False
    all_memberships[col] = all_memberships[col].fillna(False).astype(bool)


if "full_static_diagnostic_handoff" in canonical_partitions:
    base_surface = canonical_partitions["full_static_diagnostic_handoff"].copy()
else:
    base_surface = pd.concat(
        [
            df
            for name, df in canonical_partitions.items()
            if name != "excluded_handoff"
        ],
        ignore_index=True,
    ).drop_duplicates("contract_key")

n11_canonical_surface = (
    base_surface
    .drop_duplicates("contract_key")
    .merge(all_memberships, on="contract_key", how="left")
)

for col in ["in_broad", "in_strict", "in_anchor", "in_repair", "in_excluded"]:
    n11_canonical_surface[col] = n11_canonical_surface[col].fillna(False).astype(bool)


# ------------------------------------------------------------
# Fit weights
# ------------------------------------------------------------

def assign_base_weight(row: pd.Series) -> float:
    if bool(row.get("in_excluded", False)):
        return float(CONFIG.excluded_base_weight)

    if bool(row.get("in_anchor", False)):
        return float(CONFIG.anchor_base_weight)

    if bool(row.get("in_strict", False)):
        return float(CONFIG.strict_base_weight)

    if bool(row.get("in_repair", False)):
        return float(CONFIG.repair_queue_base_weight)

    if bool(row.get("in_broad", False)):
        return float(CONFIG.broad_base_weight)

    return 0.0


n11_canonical_surface["base_fit_weight"] = n11_canonical_surface.apply(assign_base_weight, axis=1)

upstream_weight = pd.to_numeric(n11_canonical_surface["diagnostic_weight"], errors="coerce")
upstream_weight = upstream_weight.where(np.isfinite(upstream_weight) & (upstream_weight > 0), 1.0)

quality_multiplier = pd.Series(1.0, index=n11_canonical_surface.index)

if "quality_score" in n11_canonical_surface.columns:
    quality_score = pd.to_numeric(n11_canonical_surface["quality_score"], errors="coerce")
    quality_multiplier = quality_score.clip(lower=0.25, upper=2.00)
    quality_multiplier = quality_multiplier.where(np.isfinite(quality_multiplier), 1.0)

convexity_penalty = np.where(n11_canonical_surface["has_convexity_violation"], 0.50, 1.00)
roughness_penalty = np.where(n11_canonical_surface["touched_by_any_iv_roughness_flag"], 0.75, 1.00)

n11_canonical_surface["fit_weight"] = (
    n11_canonical_surface["base_fit_weight"].astype(float)
    * upstream_weight.astype(float)
    * quality_multiplier.astype(float)
    * convexity_penalty.astype(float)
    * roughness_penalty.astype(float)
)

n11_canonical_surface.loc[n11_canonical_surface["in_excluded"], "fit_weight"] = 0.0

valid_fit_mask = (
    (n11_canonical_surface["fit_weight"] > 0)
    & np.isfinite(n11_canonical_surface["fit_weight"])
    & np.isfinite(n11_canonical_surface["strike"])
    & (n11_canonical_surface["strike"] > 0)
    & np.isfinite(n11_canonical_surface["forward"])
    & (n11_canonical_surface["forward"] > 0)
    & np.isfinite(n11_canonical_surface["tau_years"])
    & (n11_canonical_surface["tau_years"] > 0)
    & np.isfinite(n11_canonical_surface["selected_iv"])
    & (n11_canonical_surface["selected_iv"] > 0)
    & np.isfinite(n11_canonical_surface["total_variance"])
    & (n11_canonical_surface["total_variance"] > CONFIG.positive_total_variance_floor)
    & np.isfinite(n11_canonical_surface["log_moneyness"])
)

n11_fit_pool = n11_canonical_surface.loc[valid_fit_mask].copy()

# ------------------------------------------------------------
# Partition diagnostics
# ------------------------------------------------------------

partition_summary_rows = []

for partition_name, df in canonical_partitions.items():
    partition_summary_rows.append(
        {
            "partition": partition_name,
            "rows": len(df),
            "unique_contract_keys": df["contract_key"].nunique(),
            "quote_dates": int(df["quote_date"].nunique(dropna=True)) if "quote_date" in df.columns else 0,
            "snapshot_dates": int(df["snapshot_date"].nunique(dropna=True)) if "snapshot_date" in df.columns else 0,
            "expiries": int(df["expiry"].nunique(dropna=True)),
            "unique_strikes": int(df["strike"].nunique(dropna=True)),
            "finite_strike_rows": int(np.isfinite(df["strike"]).sum()),
            "positive_strike_rows": int((df["strike"] > 0).sum()),
            "finite_forward_rows": int(np.isfinite(df["forward"]).sum()),
            "positive_forward_rows": int((df["forward"] > 0).sum()),
            "finite_ttm_rows": int(np.isfinite(df["tau_years"]).sum()),
            "positive_ttm_rows": int((df["tau_years"] > 0).sum()),
            "finite_iv_rows": int(np.isfinite(df["selected_iv"]).sum()),
            "positive_iv_rows": int((df["selected_iv"] > 0).sum()),
            "finite_total_variance_rows": int(np.isfinite(df["total_variance"]).sum()),
            "positive_total_variance_rows": int((df["total_variance"] > 0).sum()),
            "finite_log_moneyness_rows": int(np.isfinite(df["log_moneyness"]).sum()),
        }
    )

n11_partition_summary = pd.DataFrame(partition_summary_rows)


# ------------------------------------------------------------
# Expiry-level SVI eligibility
# ------------------------------------------------------------

if len(n11_fit_pool) > 0:
    n11_expiry_fit_support = (
        n11_fit_pool
        .groupby("expiry", dropna=False)
        .agg(
            rows=("contract_key", "size"),
            unique_contracts=("contract_key", "nunique"),
            unique_strikes=("strike", "nunique"),
            min_ttm=("tau_years", "min"),
            max_ttm=("tau_years", "max"),
            min_k=("log_moneyness", "min"),
            max_k=("log_moneyness", "max"),
            sum_fit_weight=("fit_weight", "sum"),
            anchor_rows=("in_anchor", "sum"),
            strict_rows=("in_strict", "sum"),
            broad_rows=("in_broad", "sum"),
            repair_rows=("in_repair", "sum"),
            convexity_flag_rows=("has_convexity_violation", "sum"),
            roughness_flag_rows=("touched_by_any_iv_roughness_flag", "sum"),
            median_iv=("selected_iv", "median"),
            median_total_variance=("total_variance", "median"),
        )
        .reset_index()
    )

    n11_expiry_fit_support["k_width"] = (
        n11_expiry_fit_support["max_k"] - n11_expiry_fit_support["min_k"]
    )

    n11_expiry_fit_support["eligible_for_svi"] = (
        (n11_expiry_fit_support["rows"] >= CONFIG.min_rows_per_svi_slice)
        & (n11_expiry_fit_support["unique_strikes"] >= CONFIG.min_unique_strikes_per_slice)
        & (n11_expiry_fit_support["k_width"] >= CONFIG.min_log_moneyness_width)
        & (n11_expiry_fit_support["sum_fit_weight"] > 0)
    )
else:
    n11_expiry_fit_support = pd.DataFrame(
        columns=[
            "expiry",
            "rows",
            "unique_contracts",
            "unique_strikes",
            "min_ttm",
            "max_ttm",
            "min_k",
            "max_k",
            "sum_fit_weight",
            "anchor_rows",
            "strict_rows",
            "broad_rows",
            "repair_rows",
            "convexity_flag_rows",
            "roughness_flag_rows",
            "median_iv",
            "median_total_variance",
            "k_width",
            "eligible_for_svi",
        ]
    )


eligible_svi_expiry_count = (
    int(n11_expiry_fit_support["eligible_for_svi"].sum())
    if len(n11_expiry_fit_support)
    else 0
)

positive_weight_fit_rows = int((n11_fit_pool["fit_weight"] > 0).sum()) if len(n11_fit_pool) else 0
fit_pool_total_weight = float(n11_fit_pool["fit_weight"].sum()) if len(n11_fit_pool) else 0.0

n11_canonicalization_gate = pd.DataFrame(
    [
        {
            "check_name": "n10_artifacts_loaded",
            "check_type": "blocking",
            "passes": len(N10_ARTIFACTS) >= 6,
            "observed_value": float(len(N10_ARTIFACTS)),
            "threshold_or_requirement": ">= 6 core artifacts",
        },
        {
            "check_name": "broad_handoff_nonempty",
            "check_type": "blocking",
            "passes": len(N10_ARTIFACTS.get("broad_handoff", [])) > 0,
            "observed_value": float(len(N10_ARTIFACTS.get("broad_handoff", []))),
            "threshold_or_requirement": "> 0",
        },
        {
            "check_name": "strict_handoff_nonempty",
            "check_type": "blocking",
            "passes": len(N10_ARTIFACTS.get("strict_handoff", [])) > 0,
            "observed_value": float(len(N10_ARTIFACTS.get("strict_handoff", []))),
            "threshold_or_requirement": "> 0",
        },
        {
            "check_name": "anchor_handoff_nonempty",
            "check_type": "warning",
            "passes": len(N10_ARTIFACTS.get("anchor_handoff", [])) > 0,
            "observed_value": float(len(N10_ARTIFACTS.get("anchor_handoff", []))),
            "threshold_or_requirement": "> 0 preferred",
        },
        {
            "check_name": "repair_queue_available",
            "check_type": "warning",
            "passes": len(N10_ARTIFACTS.get("repair_queue", [])) > 0,
            "observed_value": float(len(N10_ARTIFACTS.get("repair_queue", []))),
            "threshold_or_requirement": "> 0 expected from Notebook 10 warnings",
        },
        {
            "check_name": "eligible_svi_expiry_count",
            "check_type": "blocking",
            "passes": eligible_svi_expiry_count >= CONFIG.min_primary_expiries,
            "observed_value": float(eligible_svi_expiry_count),
            "threshold_or_requirement": f">= {CONFIG.min_primary_expiries}",
        },
        {
            "check_name": "fit_pool_has_positive_weight",
            "check_type": "blocking",
            "passes": positive_weight_fit_rows > 0 and fit_pool_total_weight > 0,
            "observed_value": float(fit_pool_total_weight),
            "threshold_or_requirement": "> 0",
        },
    ]
)

N11_CANONICALIZATION_PASS = bool(
    n11_canonicalization_gate
    .query("check_type == 'blocking'")["passes"]
    .all()
)


# ------------------------------------------------------------
# Persist canonicalization artifacts
# ------------------------------------------------------------

_write_table_pair(
    n11_partition_summary,
    "n11_partition_summary",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_expiry_fit_support,
    "n11_expiry_fit_support",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_canonicalization_gate,
    "n11_canonicalization_gate",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_canonical_surface,
    "n11_canonical_surface",
    N11_HANDOFF_DIR,
)

_write_table_pair(
    n11_fit_pool,
    "n11_fit_pool",
    N11_HANDOFF_DIR,
)


# ------------------------------------------------------------
# Console summary
# ------------------------------------------------------------

print("Notebook 11 native canonicalization complete.")
print(f"Canonicalization pass: {N11_CANONICALIZATION_PASS}")
print(f"Canonical surface rows: {len(n11_canonical_surface):,}")
print(f"Fit pool rows: {len(n11_fit_pool):,}")
print(f"Positive-weight fit pool rows: {positive_weight_fit_rows:,}")
print(f"Fit pool total weight: {fit_pool_total_weight:,.10f}")
print(f"Eligible SVI expiries: {eligible_svi_expiry_count:,}")

display(n11_partition_summary)
display(n11_expiry_fit_support)
display(n11_canonicalization_gate)

unresolved_core = n11_native_schema_resolution.query(
    "resolved == False and canonical_name in ['time_to_maturity', 'implied_vol', 'total_variance', 'forward', 'strike', 'log_moneyness']"
)

if len(unresolved_core) > 0:
    print("Unresolved core schema fields:")
    display(unresolved_core)

if not N11_CANONICALIZATION_PASS:
    raise RuntimeError(
        "Notebook 11 native canonicalization gate failed. "
        "Check n11_partition_summary, n11_expiry_fit_support, and unresolved_core above."
    )

Notebook 10 artifacts loaded:
  full_static_diagnostic_handoff      rows=   1,333 cols=  71
  broad_handoff                       rows=   1,260 cols=  71
  strict_handoff                      rows=     224 cols=  71
  anchor_handoff                      rows=     196 cols=  71
  repair_queue                        rows=     518 cols=  35
  excluded_handoff                    rows=      73 cols=  71
  handoff_inventory                   rows=       6 cols=   9
  handoff_by_expiry                   rows=       8 cols=  17
  handoff_by_bucket                   rows=      22 cols=  15
  final_status_table                  rows=       1 cols=  18
  final_validation_ledger             rows=       6 cols=   3
Notebook 11 native canonicalization complete.
Canonicalization pass: True
Canonical surface rows: 1,333
Fit pool rows: 1,260
Positive-weight fit pool rows: 1,260
Fit pool total weight: 1,375.9427235130
Eligible SVI expiries: 8


,partition,rows,unique_contract_keys,quote_dates,snapshot_dates,expiries,unique_strikes,finite_strike_rows,positive_strike_rows,finite_forward_rows,positive_forward_rows,finite_ttm_rows,positive_ttm_rows,finite_iv_rows,positive_iv_rows,finite_total_variance_rows,positive_total_variance_rows,finite_log_moneyness_rows
0,full_static_diagnostic_handoff,1333,1333,0,0,8,265,1333,1333,1333,1333,1333,1333,1333,1333,1333,1333,1333
1,broad_handoff,1260,1260,0,0,8,260,1260,1260,1260,1260,1260,1260,1260,1260,1260,1260,1260
2,strict_handoff,224,224,0,0,8,84,224,224,224,224,224,224,224,224,224,224,224
3,anchor_handoff,196,196,0,0,8,58,196,196,196,196,196,196,196,196,196,196,196
4,repair_queue,518,518,0,0,8,209,518,518,0,0,0,0,0,0,518,518,518
5,excluded_handoff,73,73,0,0,6,65,73,73,73,73,73,73,73,73,73,73,73


,expiry,rows,unique_contracts,unique_strikes,min_ttm,max_ttm,min_k,max_k,sum_fit_weight,anchor_rows,strict_rows,broad_rows,repair_rows,convexity_flag_rows,roughness_flag_rows,median_iv,median_total_variance,k_width,eligible_for_svi
0,2026-07-10 00:00:00+00:00,97,97,97,0.0136986301,0.0136986301,-0.1242195547,0.0303652471,143.8694890514,28,28,97,0,10,16,0.1815591998,0.0004515581,0.1545848018,True
1,2026-07-17 00:00:00+00:00,130,130,130,0.0328767123,0.0328767123,-0.3318085078,0.0516084631,172.9101306437,31,31,130,0,35,19,0.1952066471,0.0012528117,0.3834169709,True
2,2026-07-24 00:00:00+00:00,152,152,152,0.0520547945,0.0520547945,-0.2958546667,0.0685381653,179.0850496218,29,29,152,0,41,16,0.1858582741,0.0017981710,0.3643928320,True
3,2026-07-31 00:00:00+00:00,175,175,175,0.0712328767,0.0712328767,-0.4009080739,0.0815180753,205.2834649752,27,45,175,0,55,39,0.2050262310,0.0029943278,0.4824261492,True
4,2026-08-07 00:00:00+00:00,131,131,131,0.0904109589,0.0904109589,-0.3342401078,0.0988765317,105.0814992812,9,9,131,0,18,34,0.1699667678,0.0026118553,0.4331166394,True
5,2026-08-21 00:00:00+00:00,173,173,173,0.1287671233,0.1287671233,-0.5084862228,0.1391985836,195.1488148586,29,33,173,0,42,33,0.1943689717,0.0048647314,0.6476848065,True
6,2026-08-31 00:00:00+00:00,187,187,187,0.1561643836,0.1561643836,-0.3842214430,0.1207044586,202.7470896275,27,33,187,0,49,44,0.1957450772,0.0059836156,0.5049259016,True
7,2026-09-18 00:00:00+00:00,215,215,215,0.2054794521,0.2054794521,-0.4252077096,0.2047605693,171.8171854536,16,16,215,0,69,57,0.2073878979,0.0088376178,0.6299682789,True


,check_name,check_type,passes,observed_value,threshold_or_requirement
0,n10_artifacts_loaded,blocking,True,11.0000000000,>= 6 core artifacts
1,broad_handoff_nonempty,blocking,True,"1,260.0000000000",> 0
2,strict_handoff_nonempty,blocking,True,224.0000000000,> 0
3,anchor_handoff_nonempty,warning,True,196.0000000000,> 0 preferred
4,repair_queue_available,warning,True,518.0000000000,> 0 expected from Notebook 10 warnings
5,eligible_svi_expiry_count,blocking,True,8.0000000000,>= 4
6,fit_pool_has_positive_weight,blocking,True,"1,375.9427235130",> 0


Unresolved core schema fields:


,table_name,canonical_name,resolved_column,resolved,aliases_tried
268,repair_queue,time_to_maturity,None,False,tau_years
273,repair_queue,forward,None,False,forward
278,repair_queue,implied_vol,None,False,selected_iv


In [4]:
# ------------------------------------------------------------
# Notebook 11 cell 04: SVI, Black-Scholes, and no-arbitrage primitives
# ------------------------------------------------------------
# This cell defines the mathematical primitives needed before any SVI fitting:
#
#   1. raw SVI total-variance slice
#   2. analytic first and second SVI derivatives
#   3. Gatheral-Jacquier butterfly density diagnostic g(k)
#   4. forward-measure Black-Scholes reconstruction
#   5. diagnostic k-grids for per-expiry and cross-expiry checks
#
# It does not fit anything yet.

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Iterable

import numpy as np
import pandas as pd
from scipy.stats import norm

# ------------------------------------------------------------
# Numerical constants
# ------------------------------------------------------------

N11_EPS = 1.0e-12
N11_TINY = 1.0e-14


# ------------------------------------------------------------
# Raw SVI parameter container
# ------------------------------------------------------------

@dataclass(frozen=True)
class RawSVIParams:
    a: float
    b: float
    rho: float
    m: float
    sigma: float

    def as_array(self) -> np.ndarray:
        return np.array([self.a, self.b, self.rho, self.m, self.sigma], dtype=float)

    @classmethod
    def from_array(cls, theta: Iterable[float]) -> "RawSVIParams":
        a, b, rho, m, sigma = np.asarray(theta, dtype=float)
        return cls(float(a), float(b), float(rho), float(m), float(sigma))


def as_svi_array(theta: RawSVIParams | Iterable[float]) -> np.ndarray:
    if isinstance(theta, RawSVIParams):
        return theta.as_array()
    return np.asarray(theta, dtype=float)


# ------------------------------------------------------------
# Raw SVI total variance and derivatives
# ------------------------------------------------------------

def raw_svi_total_variance(k: np.ndarray | float, theta: RawSVIParams | Iterable[float]) -> np.ndarray:
    """
    Raw SVI total variance:

        w(k) = a + b * [rho * (k - m) + sqrt((k - m)^2 + sigma^2)]

    Parameter order:

        theta = [a, b, rho, m, sigma]
    """
    a, b, rho, m, sigma = as_svi_array(theta)
    k_arr = np.asarray(k, dtype=float)

    x = k_arr - m
    root = np.sqrt(x * x + sigma * sigma)

    return a + b * (rho * x + root)


def raw_svi_derivatives(
    k: np.ndarray | float,
    theta: RawSVIParams | Iterable[float],
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Return raw SVI total variance and its first two k-derivatives.

        w(k)   = a + b * [rho * x + sqrt(x^2 + sigma^2)]
        w'(k)  = b * [rho + x / sqrt(x^2 + sigma^2)]
        w''(k) = b * sigma^2 / (x^2 + sigma^2)^(3/2)

    where x = k - m.
    """
    a, b, rho, m, sigma = as_svi_array(theta)
    k_arr = np.asarray(k, dtype=float)

    x = k_arr - m
    root = np.sqrt(x * x + sigma * sigma)
    root = np.maximum(root, N11_TINY)

    w = a + b * (rho * x + root)
    w_first = b * (rho + x / root)
    w_second = b * sigma * sigma / np.maximum(root**3, N11_TINY)

    return w, w_first, w_second


def raw_svi_minimum_total_variance(theta: RawSVIParams | Iterable[float]) -> dict[str, float]:
    """
    Analytic minimum of the raw SVI slice when |rho| < 1 and sigma > 0.

        k_min = m - rho * sigma / sqrt(1 - rho^2)

        w_min = a + b * sigma * sqrt(1 - rho^2)
    """
    a, b, rho, m, sigma = as_svi_array(theta)

    one_minus_rho2 = max(1.0 - rho * rho, N11_TINY)
    sqrt_term = math.sqrt(one_minus_rho2)

    k_min = m - rho * sigma / sqrt_term
    w_min = a + b * sigma * sqrt_term

    return {
        "k_min": float(k_min),
        "w_min": float(w_min),
    }


def raw_svi_parameter_constraints(theta: RawSVIParams | Iterable[float]) -> dict[str, float | bool]:
    """
    Basic raw-SVI parameter feasibility checks.
    """
    a, b, rho, m, sigma = as_svi_array(theta)
    min_info = raw_svi_minimum_total_variance(theta)

    finite_params = bool(np.isfinite([a, b, rho, m, sigma]).all())
    positive_b = bool(b >= 0.0)
    rho_inside_unit_interval = bool(abs(rho) < 1.0)
    positive_sigma = bool(sigma > 0.0)
    nonnegative_min_variance = bool(min_info["w_min"] >= 0.0)

    return {
        "finite_params": finite_params,
        "positive_b": positive_b,
        "rho_inside_unit_interval": rho_inside_unit_interval,
        "positive_sigma": positive_sigma,
        "k_min": float(min_info["k_min"]),
        "w_min": float(min_info["w_min"]),
        "nonnegative_min_variance": nonnegative_min_variance,
        "basic_feasible": bool(
            finite_params
            and positive_b
            and rho_inside_unit_interval
            and positive_sigma
            and nonnegative_min_variance
        ),
    }


# ------------------------------------------------------------
# Butterfly-arbitrage diagnostic for an SVI slice
# ------------------------------------------------------------

def svi_d_minus(k: np.ndarray | float, w: np.ndarray | float) -> np.ndarray:
    """
    Gatheral-Jacquier normalized Black-Scholes d_minus:

        d_-(k) = -k / sqrt(w) - sqrt(w) / 2
    """
    k_arr = np.asarray(k, dtype=float)
    w_arr = np.maximum(np.asarray(w, dtype=float), N11_TINY)
    sqrt_w = np.sqrt(w_arr)
    return -k_arr / sqrt_w - 0.5 * sqrt_w


def svi_d_plus(k: np.ndarray | float, w: np.ndarray | float) -> np.ndarray:
    """
    Gatheral-Jacquier normalized Black-Scholes d_plus:

        d_+(k) = -k / sqrt(w) + sqrt(w) / 2
    """
    k_arr = np.asarray(k, dtype=float)
    w_arr = np.maximum(np.asarray(w, dtype=float), N11_TINY)
    sqrt_w = np.sqrt(w_arr)
    return -k_arr / sqrt_w + 0.5 * sqrt_w


def svi_butterfly_g(
    k: np.ndarray | float,
    theta: RawSVIParams | Iterable[float],
) -> np.ndarray:
    """
    Gatheral-Jacquier butterfly diagnostic:

        g(k)
        =
        (1 - k w'(k) / (2w(k)))^2
        - (w'(k)^2 / 4) * (1 / w(k) + 1 / 4)
        + w''(k) / 2

    A smooth slice is butterfly-arbitrage-free on a checked grid when g(k) >= 0
    within tolerance and total variance is positive.
    """
    k_arr = np.asarray(k, dtype=float)
    w, w_first, w_second = raw_svi_derivatives(k_arr, theta)

    w_safe = np.maximum(w, N11_TINY)

    term_1 = (1.0 - (k_arr * w_first) / (2.0 * w_safe)) ** 2
    term_2 = (w_first * w_first / 4.0) * (1.0 / w_safe + 0.25)
    term_3 = 0.5 * w_second

    return term_1 - term_2 + term_3


def svi_slice_grid_diagnostics(
    k_grid: np.ndarray,
    theta: RawSVIParams | Iterable[float],
    *,
    density_tolerance: float | None = None,
) -> pd.DataFrame:
    """
    Evaluate total variance, derivatives, and butterfly diagnostic on a k-grid.
    """
    if density_tolerance is None:
        density_tolerance = getattr(CONFIG, "density_tolerance", -1.0e-7)

    k_grid = np.asarray(k_grid, dtype=float)

    w, w_first, w_second = raw_svi_derivatives(k_grid, theta)
    g = svi_butterfly_g(k_grid, theta)

    out = pd.DataFrame(
        {
            "k": k_grid,
            "svi_total_variance": w,
            "svi_w_first": w_first,
            "svi_w_second": w_second,
            "svi_g": g,
            "svi_d_plus": svi_d_plus(k_grid, w),
            "svi_d_minus": svi_d_minus(k_grid, w),
        }
    )

    out["positive_total_variance"] = out["svi_total_variance"] > 0.0
    out["passes_butterfly_grid"] = out["svi_g"] >= density_tolerance

    return out


def summarize_svi_grid_diagnostics(
    grid_diagnostics: pd.DataFrame,
    *,
    density_tolerance: float | None = None,
) -> dict[str, float | int | bool]:
    """
    Summarize one SVI grid diagnostic table.
    """
    if density_tolerance is None:
        density_tolerance = getattr(CONFIG, "density_tolerance", -1.0e-7)

    g = pd.to_numeric(grid_diagnostics["svi_g"], errors="coerce")
    w = pd.to_numeric(grid_diagnostics["svi_total_variance"], errors="coerce")

    positive_w = np.isfinite(w) & (w > 0.0)
    finite_g = np.isfinite(g)
    g_violation = finite_g & (g < density_tolerance)

    return {
        "grid_points": int(len(grid_diagnostics)),
        "finite_w_points": int(np.isfinite(w).sum()),
        "positive_w_points": int(positive_w.sum()),
        "finite_g_points": int(finite_g.sum()),
        "g_violation_points": int(g_violation.sum()),
        "min_total_variance": float(np.nanmin(w)) if len(w) else np.nan,
        "min_g": float(np.nanmin(g)) if len(g) else np.nan,
        "max_g_violation_amount": float(max(0.0, density_tolerance - np.nanmin(g))) if len(g) else np.nan,
        "positive_total_variance_pass": bool(positive_w.all()) if len(w) else False,
        "butterfly_grid_pass": bool((g >= density_tolerance).all()) if len(g) else False,
    }


# ------------------------------------------------------------
# Forward-measure Black-Scholes reconstruction
# ------------------------------------------------------------

def bsm_forward_price(
    forward: np.ndarray | float,
    strike: np.ndarray | float,
    tau: np.ndarray | float,
    discount_factor: np.ndarray | float,
    volatility: np.ndarray | float,
    option_type: np.ndarray | str = "call",
) -> np.ndarray:
    """
    Black-Scholes price using forward and discount factor.

    Call:

        C = D * [F N(d1) - K N(d2)]

    Put:

        P = D * [K N(-d2) - F N(-d1)]

    where

        d1 = [log(F/K) + 0.5 sigma^2 T] / [sigma sqrt(T)]
        d2 = d1 - sigma sqrt(T)
    """
    F = np.asarray(forward, dtype=float)
    K = np.asarray(strike, dtype=float)
    T = np.asarray(tau, dtype=float)
    D = np.asarray(discount_factor, dtype=float)
    vol = np.asarray(volatility, dtype=float)

    F, K, T, D, vol = np.broadcast_arrays(F, K, T, D, vol)

    opt = np.asarray(option_type)
    if opt.shape == ():
        opt = np.full(F.shape, str(option_type).lower())
    else:
        opt = np.char.lower(opt.astype(str))
        opt = np.broadcast_to(opt, F.shape)

    is_call = np.isin(opt, ["call", "c"])
    is_put = np.isin(opt, ["put", "p"])

    sqrt_T = np.sqrt(np.maximum(T, 0.0))
    vol_sqrt_T = vol * sqrt_T

    intrinsic_call = D * np.maximum(F - K, 0.0)
    intrinsic_put = D * np.maximum(K - F, 0.0)

    regular = (
        np.isfinite(F)
        & np.isfinite(K)
        & np.isfinite(T)
        & np.isfinite(D)
        & np.isfinite(vol)
        & (F > 0.0)
        & (K > 0.0)
        & (T > 0.0)
        & (D > 0.0)
        & (vol > 0.0)
        & (vol_sqrt_T > 0.0)
    )

    price = np.full(F.shape, np.nan, dtype=float)

    price[is_call] = intrinsic_call[is_call]
    price[is_put] = intrinsic_put[is_put]

    if regular.any():
        d1 = np.full(F.shape, np.nan, dtype=float)
        d2 = np.full(F.shape, np.nan, dtype=float)

        d1[regular] = (
            np.log(F[regular] / K[regular])
            + 0.5 * vol[regular] * vol[regular] * T[regular]
        ) / vol_sqrt_T[regular]
        d2[regular] = d1[regular] - vol_sqrt_T[regular]

        call_regular = regular & is_call
        put_regular = regular & is_put

        price[call_regular] = D[call_regular] * (
            F[call_regular] * norm.cdf(d1[call_regular])
            - K[call_regular] * norm.cdf(d2[call_regular])
        )

        price[put_regular] = D[put_regular] * (
            K[put_regular] * norm.cdf(-d2[put_regular])
            - F[put_regular] * norm.cdf(-d1[put_regular])
        )

    return price


def bsm_forward_price_from_total_variance(
    forward: np.ndarray | float,
    strike: np.ndarray | float,
    tau: np.ndarray | float,
    discount_factor: np.ndarray | float,
    total_variance: np.ndarray | float,
    option_type: np.ndarray | str = "call",
) -> np.ndarray:
    """
    Convert total variance into implied volatility, then reconstruct BSM price.

        sigma = sqrt(w / T)
    """
    T = np.asarray(tau, dtype=float)
    w = np.asarray(total_variance, dtype=float)

    T_b, w_b = np.broadcast_arrays(T, w)
    vol = np.full(T_b.shape, np.nan, dtype=float)

    ok = np.isfinite(T_b) & np.isfinite(w_b) & (T_b > 0.0) & (w_b > 0.0)
    vol[ok] = np.sqrt(w_b[ok] / T_b[ok])

    return bsm_forward_price(
        forward=forward,
        strike=strike,
        tau=tau,
        discount_factor=discount_factor,
        volatility=vol,
        option_type=option_type,
    )


def bsm_forward_delta(
    forward: np.ndarray | float,
    strike: np.ndarray | float,
    tau: np.ndarray | float,
    discount_factor: np.ndarray | float,
    volatility: np.ndarray | float,
    option_type: np.ndarray | str = "call",
) -> np.ndarray:
    """
    Forward-measure Black-Scholes delta with respect to spot only if forward scales
    one-for-one with spot. Used here mainly as a consistency diagnostic.
    """
    F = np.asarray(forward, dtype=float)
    K = np.asarray(strike, dtype=float)
    T = np.asarray(tau, dtype=float)
    D = np.asarray(discount_factor, dtype=float)
    vol = np.asarray(volatility, dtype=float)

    F, K, T, D, vol = np.broadcast_arrays(F, K, T, D, vol)

    opt = np.asarray(option_type)
    if opt.shape == ():
        opt = np.full(F.shape, str(option_type).lower())
    else:
        opt = np.char.lower(opt.astype(str))
        opt = np.broadcast_to(opt, F.shape)

    is_call = np.isin(opt, ["call", "c"])
    is_put = np.isin(opt, ["put", "p"])

    out = np.full(F.shape, np.nan, dtype=float)

    regular = (
        np.isfinite(F)
        & np.isfinite(K)
        & np.isfinite(T)
        & np.isfinite(D)
        & np.isfinite(vol)
        & (F > 0.0)
        & (K > 0.0)
        & (T > 0.0)
        & (D > 0.0)
        & (vol > 0.0)
    )

    if regular.any():
        vol_sqrt_T = vol[regular] * np.sqrt(T[regular])
        d1 = (
            np.log(F[regular] / K[regular])
            + 0.5 * vol[regular] * vol[regular] * T[regular]
        ) / vol_sqrt_T

        idx_regular = np.flatnonzero(regular)

        call_idx = idx_regular[is_call[regular]]
        put_idx = idx_regular[is_put[regular]]

        out[call_idx] = D[call_idx] * norm.cdf(d1[is_call[regular]])
        out[put_idx] = -D[put_idx] * norm.cdf(-d1[is_put[regular]])

    return out


# ------------------------------------------------------------
# Diagnostic k-grid construction
# ------------------------------------------------------------

def make_k_grid_from_bounds(
    min_k: float,
    max_k: float,
    *,
    grid_size: int | None = None,
    pad: float | None = None,
) -> np.ndarray:
    if grid_size is None:
        grid_size = getattr(CONFIG, "k_grid_size", 181)

    if pad is None:
        pad = getattr(CONFIG, "k_grid_pad", 0.0250)

    min_k = float(min_k)
    max_k = float(max_k)

    if not np.isfinite(min_k) or not np.isfinite(max_k):
        raise ValueError("Cannot construct k-grid from non-finite bounds.")

    if max_k <= min_k:
        center = 0.5 * (min_k + max_k)
        min_k = center - 0.10
        max_k = center + 0.10

    return np.linspace(min_k - pad, max_k + pad, int(grid_size))


def make_expiry_k_grids(
    fit_pool: pd.DataFrame,
    *,
    grid_size: int | None = None,
    pad: float | None = None,
) -> dict[pd.Timestamp, np.ndarray]:
    grids: dict[pd.Timestamp, np.ndarray] = {}

    if len(fit_pool) == 0:
        return grids

    for expiry, grp in fit_pool.groupby("expiry", dropna=True):
        k = pd.to_numeric(grp["log_moneyness"], errors="coerce")
        k = k[np.isfinite(k)]

        if len(k) == 0:
            continue

        grids[expiry] = make_k_grid_from_bounds(
            float(k.min()),
            float(k.max()),
            grid_size=grid_size,
            pad=pad,
        )

    return grids


def make_common_k_grid(
    fit_pool: pd.DataFrame,
    *,
    grid_size: int | None = None,
    pad: float | None = None,
    inner_quantile: float | None = None,
) -> np.ndarray:
    """
    Construct a common k-grid for calendar checks.

    If inner_quantile is supplied, use central quantile bounds to reduce the
    influence of extreme wings. Otherwise use full observed support.
    """
    if len(fit_pool) == 0:
        raise ValueError("Cannot construct common k-grid from an empty fit pool.")

    k = pd.to_numeric(fit_pool["log_moneyness"], errors="coerce")
    k = k[np.isfinite(k)]

    if len(k) == 0:
        raise ValueError("Cannot construct common k-grid without finite log-moneyness.")

    if inner_quantile is not None:
        q = float(inner_quantile)
        if not (0.0 < q < 0.5):
            raise ValueError("inner_quantile must be in (0, 0.5).")
        min_k = float(k.quantile(q))
        max_k = float(k.quantile(1.0 - q))
    else:
        min_k = float(k.min())
        max_k = float(k.max())

    return make_k_grid_from_bounds(
        min_k,
        max_k,
        grid_size=grid_size,
        pad=pad,
    )


N11_EXPIRY_K_GRIDS = make_expiry_k_grids(n11_fit_pool)
N11_COMMON_K_GRID = make_common_k_grid(n11_fit_pool)
N11_COMMON_K_GRID_INNER = make_common_k_grid(n11_fit_pool, inner_quantile=0.025)


n11_diagnostic_grid_summary = pd.DataFrame(
    [
        {
            "grid_name": "common_full_support",
            "grid_points": len(N11_COMMON_K_GRID),
            "min_k": float(np.min(N11_COMMON_K_GRID)),
            "max_k": float(np.max(N11_COMMON_K_GRID)),
            "width": float(np.max(N11_COMMON_K_GRID) - np.min(N11_COMMON_K_GRID)),
        },
        {
            "grid_name": "common_inner_95pct_support",
            "grid_points": len(N11_COMMON_K_GRID_INNER),
            "min_k": float(np.min(N11_COMMON_K_GRID_INNER)),
            "max_k": float(np.max(N11_COMMON_K_GRID_INNER)),
            "width": float(np.max(N11_COMMON_K_GRID_INNER) - np.min(N11_COMMON_K_GRID_INNER)),
        },
    ]
)

expiry_grid_rows = []

for expiry, grid in N11_EXPIRY_K_GRIDS.items():
    expiry_grid_rows.append(
        {
            "expiry": expiry,
            "grid_points": len(grid),
            "min_k": float(np.min(grid)),
            "max_k": float(np.max(grid)),
            "width": float(np.max(grid) - np.min(grid)),
        }
    )

n11_expiry_grid_summary = pd.DataFrame(expiry_grid_rows).sort_values("expiry").reset_index(drop=True)


# ------------------------------------------------------------
# Primitive self-tests
# ------------------------------------------------------------

def finite_difference_first_derivative(func, x: np.ndarray, theta, h: float = 1.0e-5) -> np.ndarray:
    return (func(x + h, theta) - func(x - h, theta)) / (2.0 * h)


def finite_difference_second_derivative(func, x: np.ndarray, theta, h: float = 1.0e-4) -> np.ndarray:
    return (func(x + h, theta) - 2.0 * func(x, theta) + func(x - h, theta)) / (h * h)


test_theta = RawSVIParams(
    a=0.0200,
    b=0.1200,
    rho=-0.3500,
    m=-0.0200,
    sigma=0.3500,
)

test_k = np.linspace(-0.50, 0.50, 101)
test_w, test_w1, test_w2 = raw_svi_derivatives(test_k, test_theta)

fd_w1 = finite_difference_first_derivative(raw_svi_total_variance, test_k, test_theta)
fd_w2 = finite_difference_second_derivative(raw_svi_total_variance, test_k, test_theta)

test_g = svi_butterfly_g(test_k, test_theta)
test_constraints = raw_svi_parameter_constraints(test_theta)

test_forward = 100.0
test_strike = 100.0
test_tau = 1.0
test_discount = 0.9700
test_vol = 0.2000

test_call = float(
    bsm_forward_price(
        test_forward,
        test_strike,
        test_tau,
        test_discount,
        test_vol,
        "call",
    )
)

test_put = float(
    bsm_forward_price(
        test_forward,
        test_strike,
        test_tau,
        test_discount,
        test_vol,
        "put",
    )
)

test_parity_error = test_call - test_put - test_discount * (test_forward - test_strike)

n11_svi_primitive_self_tests = pd.DataFrame(
    [
        {
            "check_name": "svi_total_variance_finite",
            "passes": bool(np.isfinite(test_w).all()),
            "observed_value": float(np.nanmin(test_w)),
            "threshold_or_requirement": "all finite",
        },
        {
            "check_name": "svi_total_variance_positive",
            "passes": bool((test_w > 0.0).all()),
            "observed_value": float(np.nanmin(test_w)),
            "threshold_or_requirement": "> 0 on test grid",
        },
        {
            "check_name": "svi_first_derivative_matches_finite_difference",
            "passes": bool(np.nanmax(np.abs(test_w1 - fd_w1)) < 1.0e-6),
            "observed_value": float(np.nanmax(np.abs(test_w1 - fd_w1))),
            "threshold_or_requirement": "< 1e-6",
        },
        {
            "check_name": "svi_second_derivative_matches_finite_difference",
            "passes": bool(np.nanmax(np.abs(test_w2 - fd_w2)) < 1.0e-5),
            "observed_value": float(np.nanmax(np.abs(test_w2 - fd_w2))),
            "threshold_or_requirement": "< 1e-5",
        },
        {
            "check_name": "svi_g_finite",
            "passes": bool(np.isfinite(test_g).all()),
            "observed_value": float(np.nanmin(test_g)),
            "threshold_or_requirement": "all finite",
        },
        {
            "check_name": "raw_svi_basic_constraints",
            "passes": bool(test_constraints["basic_feasible"]),
            "observed_value": float(test_constraints["w_min"]),
            "threshold_or_requirement": "b >= 0, |rho| < 1, sigma > 0, min w >= 0",
        },
        {
            "check_name": "bsm_forward_put_call_parity",
            "passes": bool(abs(test_parity_error) < 1.0e-10),
            "observed_value": float(test_parity_error),
            "threshold_or_requirement": "abs(error) < 1e-10",
        },
        {
            "check_name": "expiry_k_grids_available",
            "passes": bool(len(N11_EXPIRY_K_GRIDS) >= getattr(CONFIG, "min_primary_expiries", 4)),
            "observed_value": float(len(N11_EXPIRY_K_GRIDS)),
            "threshold_or_requirement": f">= {getattr(CONFIG, 'min_primary_expiries', 4)}",
        },
        {
            "check_name": "common_k_grid_available",
            "passes": bool(len(N11_COMMON_K_GRID) > 0 and np.isfinite(N11_COMMON_K_GRID).all()),
            "observed_value": float(len(N11_COMMON_K_GRID)),
            "threshold_or_requirement": "> 0 and finite",
        },
    ]
)

N11_PRIMITIVE_SELF_TEST_PASS = bool(n11_svi_primitive_self_tests["passes"].all())


# ------------------------------------------------------------
# Persist primitive diagnostics
# ------------------------------------------------------------

_write_table_pair(
    n11_svi_primitive_self_tests,
    "n11_svi_primitive_self_tests",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_diagnostic_grid_summary,
    "n11_diagnostic_grid_summary",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_expiry_grid_summary,
    "n11_expiry_grid_summary",
    N11_TABLE_DIR,
)


# ------------------------------------------------------------
# Console summary
# ------------------------------------------------------------

print("Notebook 11 SVI/BSM primitive layer complete.")
print(f"Primitive self-test pass: {N11_PRIMITIVE_SELF_TEST_PASS}")
print(f"Expiry grids available: {len(N11_EXPIRY_K_GRIDS):,}")
print(f"Common grid points: {len(N11_COMMON_K_GRID):,}")

display(n11_svi_primitive_self_tests)
display(n11_diagnostic_grid_summary)
display(n11_expiry_grid_summary)

if not N11_PRIMITIVE_SELF_TEST_PASS:
    raise RuntimeError(
        "Notebook 11 SVI/BSM primitive self-tests failed. "
        "Do not proceed to SVI fitting until the primitive layer passes."
    )

Notebook 11 SVI/BSM primitive layer complete.
Primitive self-test pass: True
Expiry grids available: 8
Common grid points: 181


,check_name,passes,observed_value,threshold_or_requirement
0,svi_total_variance_finite,True,0.0593435713,all finite
1,svi_total_variance_positive,True,0.0593435713,> 0 on test grid
2,svi_first_derivative_matches_finite_difference,True,0.0000000000,< 1e-6
3,svi_second_derivative_matches_finite_difference,True,0.0000000084,< 1e-5
4,svi_g_finite,True,0.4642635932,all finite
5,raw_svi_basic_constraints,True,0.0593434874,"b >= 0, |rho| < 1, sigma > 0, min w >= 0"
6,bsm_forward_put_call_parity,True,0.0000000000,abs(error) < 1e-10
7,expiry_k_grids_available,True,8.0000000000,>= 4
8,common_k_grid_available,True,181.0000000000,> 0 and finite


,grid_name,grid_points,min_k,max_k,width
0,common_full_support,181,-0.5334862228,0.2297605693,0.7632467922
1,common_inner_95pct_support,181,-0.3618804111,0.1052233201,0.4671037312


,expiry,grid_points,min_k,max_k,width
0,2026-07-10 00:00:00+00:00,181,-0.1492195547,0.0553652471,0.2045848018
1,2026-07-17 00:00:00+00:00,181,-0.3568085078,0.0766084631,0.4334169709
2,2026-07-24 00:00:00+00:00,181,-0.3208546667,0.0935381653,0.4143928320
3,2026-07-31 00:00:00+00:00,181,-0.4259080739,0.1065180753,0.5324261492
4,2026-08-07 00:00:00+00:00,181,-0.3592401078,0.1238765317,0.4831166394
5,2026-08-21 00:00:00+00:00,181,-0.5334862228,0.1641985836,0.6976848065
6,2026-08-31 00:00:00+00:00,181,-0.4092214430,0.1457044586,0.5549259016
7,2026-09-18 00:00:00+00:00,181,-0.4502077096,0.2297605693,0.6799682789


In [5]:
# ------------------------------------------------------------
# Notebook 11 cell 05: Candidate fit design and SVI initialization
# ------------------------------------------------------------
# This cell prepares the per-expiry fitting datasets and initial parameter seeds.
#
# It does not run the nonlinear optimizer yet. It creates:
#
#   1. candidate-mode definitions
#   2. mode-specific fit pools
#   3. per-expiry support diagnostics by candidate mode
#   4. deterministic SVI initial guesses
#   5. multistart SVI seed tables
#
# The next cell can consume `N11_SVI_SEED_TABLE` and run least-squares fitting.

from __future__ import annotations

import itertools
from dataclasses import dataclass
from typing import Callable

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Candidate-mode configuration
# ------------------------------------------------------------

@dataclass(frozen=True)
class SVIFitMode:
    mode_name: str
    description: str
    use_anchor: bool
    use_strict: bool
    use_broad: bool
    use_repair: bool
    use_excluded: bool
    convexity_penalty_strength: float
    roughness_penalty_strength: float
    wing_downweight_strength: float
    atm_emphasis_strength: float
    minimum_rows: int
    minimum_unique_strikes: int
    minimum_k_width: float


N11_SVI_FIT_MODES: dict[str, SVIFitMode] = {
    "strict_svi": SVIFitMode(
        mode_name="strict_svi",
        description="SVI fitted only to clean strict/anchor rows; conservative baseline.",
        use_anchor=True,
        use_strict=True,
        use_broad=False,
        use_repair=False,
        use_excluded=False,
        convexity_penalty_strength=1.00,
        roughness_penalty_strength=1.00,
        wing_downweight_strength=0.00,
        atm_emphasis_strength=0.25,
        minimum_rows=max(CONFIG.min_rows_per_svi_slice, 8),
        minimum_unique_strikes=max(CONFIG.min_unique_strikes_per_slice, 8),
        minimum_k_width=CONFIG.min_log_moneyness_width,
    ),
    "weighted_broad_svi": SVIFitMode(
        mode_name="weighted_broad_svi",
        description="SVI fitted to broad rows, with anchor/strict rows naturally receiving larger weights.",
        use_anchor=True,
        use_strict=True,
        use_broad=True,
        use_repair=False,
        use_excluded=False,
        convexity_penalty_strength=0.50,
        roughness_penalty_strength=0.50,
        wing_downweight_strength=0.10,
        atm_emphasis_strength=0.15,
        minimum_rows=max(CONFIG.min_rows_per_svi_slice, 8),
        minimum_unique_strikes=max(CONFIG.min_unique_strikes_per_slice, 8),
        minimum_k_width=CONFIG.min_log_moneyness_width,
    ),
    "repair_aware_svi": SVIFitMode(
        mode_name="repair_aware_svi",
        description="SVI fitted to broad rows while retaining convexity/roughness penalties for repair-aware smoothing.",
        use_anchor=True,
        use_strict=True,
        use_broad=True,
        use_repair=True,
        use_excluded=False,
        convexity_penalty_strength=0.25,
        roughness_penalty_strength=0.35,
        wing_downweight_strength=0.20,
        atm_emphasis_strength=0.10,
        minimum_rows=max(CONFIG.min_rows_per_svi_slice, 8),
        minimum_unique_strikes=max(CONFIG.min_unique_strikes_per_slice, 8),
        minimum_k_width=CONFIG.min_log_moneyness_width,
    ),
}


n11_svi_fit_mode_summary = pd.DataFrame(
    [
        {
            "mode_name": mode.mode_name,
            "description": mode.description,
            "use_anchor": mode.use_anchor,
            "use_strict": mode.use_strict,
            "use_broad": mode.use_broad,
            "use_repair": mode.use_repair,
            "use_excluded": mode.use_excluded,
            "convexity_penalty_strength": mode.convexity_penalty_strength,
            "roughness_penalty_strength": mode.roughness_penalty_strength,
            "wing_downweight_strength": mode.wing_downweight_strength,
            "atm_emphasis_strength": mode.atm_emphasis_strength,
            "minimum_rows": mode.minimum_rows,
            "minimum_unique_strikes": mode.minimum_unique_strikes,
            "minimum_k_width": mode.minimum_k_width,
        }
        for mode in N11_SVI_FIT_MODES.values()
    ]
)


# ------------------------------------------------------------
# Mode-specific row selection and weights
# ------------------------------------------------------------

def make_mode_membership_mask(df: pd.DataFrame, mode: SVIFitMode) -> pd.Series:
    mask = pd.Series(False, index=df.index)

    if mode.use_anchor and "in_anchor" in df.columns:
        mask = mask | df["in_anchor"].astype(bool)

    if mode.use_strict and "in_strict" in df.columns:
        mask = mask | df["in_strict"].astype(bool)

    if mode.use_broad and "in_broad" in df.columns:
        mask = mask | df["in_broad"].astype(bool)

    if mode.use_repair and "in_repair" in df.columns:
        mask = mask | df["in_repair"].astype(bool)

    if mode.use_excluded and "in_excluded" in df.columns:
        mask = mask | df["in_excluded"].astype(bool)

    if not mode.use_excluded and "in_excluded" in df.columns:
        mask = mask & ~df["in_excluded"].astype(bool)

    return mask


def make_mode_weight(df: pd.DataFrame, mode: SVIFitMode) -> pd.Series:
    """
    Construct fitting weights for one SVI candidate mode.

    The base is the canonical fit weight. Then:
      - convexity-warning rows are downweighted
      - roughness-warning rows are downweighted
      - far-wing rows can be mildly downweighted
      - near-ATM rows can be mildly emphasized
    """
    weight = pd.to_numeric(df["fit_weight"], errors="coerce").fillna(0.0).clip(lower=0.0)

    if "has_convexity_violation" in df.columns:
        convexity_flag = df["has_convexity_violation"].fillna(False).astype(bool)
        weight = weight * np.where(convexity_flag, mode.convexity_penalty_strength, 1.0)

    if "touched_by_any_iv_roughness_flag" in df.columns:
        rough_flag = df["touched_by_any_iv_roughness_flag"].fillna(False).astype(bool)
        weight = weight * np.where(rough_flag, mode.roughness_penalty_strength, 1.0)

    abs_k = pd.to_numeric(df["log_moneyness"], errors="coerce").abs()
    abs_k = abs_k.where(np.isfinite(abs_k), np.nan)

    if mode.wing_downweight_strength > 0:
        # Smooth wing penalty bounded away from zero.
        wing_multiplier = 1.0 / (1.0 + mode.wing_downweight_strength * (abs_k / 0.10).fillna(0.0) ** 2)
        wing_multiplier = wing_multiplier.clip(lower=0.25, upper=1.00)
        weight = weight * wing_multiplier

    if mode.atm_emphasis_strength > 0:
        # Gentle ATM emphasis; not a hard ATM-only fit.
        atm_multiplier = 1.0 + mode.atm_emphasis_strength * np.exp(-0.5 * (abs_k.fillna(999.0) / 0.075) ** 2)
        weight = weight * atm_multiplier

    return pd.Series(weight, index=df.index).astype(float)


def make_candidate_fit_pool(df: pd.DataFrame, mode: SVIFitMode) -> pd.DataFrame:
    membership_mask = make_mode_membership_mask(df, mode)

    numeric_valid_mask = (
        np.isfinite(pd.to_numeric(df["strike"], errors="coerce"))
        & (pd.to_numeric(df["strike"], errors="coerce") > 0)
        & np.isfinite(pd.to_numeric(df["forward"], errors="coerce"))
        & (pd.to_numeric(df["forward"], errors="coerce") > 0)
        & np.isfinite(pd.to_numeric(df["tau_years"], errors="coerce"))
        & (pd.to_numeric(df["tau_years"], errors="coerce") > 0)
        & np.isfinite(pd.to_numeric(df["selected_iv"], errors="coerce"))
        & (pd.to_numeric(df["selected_iv"], errors="coerce") > 0)
        & np.isfinite(pd.to_numeric(df["total_variance"], errors="coerce"))
        & (pd.to_numeric(df["total_variance"], errors="coerce") > CONFIG.positive_total_variance_floor)
        & np.isfinite(pd.to_numeric(df["log_moneyness"], errors="coerce"))
    )

    out = df.loc[membership_mask & numeric_valid_mask].copy()
    out["svi_mode"] = mode.mode_name
    out["mode_fit_weight"] = make_mode_weight(out, mode)

    out = out.loc[np.isfinite(out["mode_fit_weight"]) & (out["mode_fit_weight"] > 0)].copy()

    return out.reset_index(drop=True)


N11_SVI_MODE_POOLS = {
    mode_name: make_candidate_fit_pool(n11_canonical_surface, mode)
    for mode_name, mode in N11_SVI_FIT_MODES.items()
}


# ------------------------------------------------------------
# Per-mode / per-expiry support diagnostics
# ------------------------------------------------------------

support_rows = []

for mode_name, pool in N11_SVI_MODE_POOLS.items():
    mode = N11_SVI_FIT_MODES[mode_name]

    if len(pool) == 0:
        support_rows.append(
            {
                "mode_name": mode_name,
                "expiry": pd.NaT,
                "rows": 0,
                "unique_contracts": 0,
                "unique_strikes": 0,
                "min_ttm": np.nan,
                "max_ttm": np.nan,
                "min_k": np.nan,
                "max_k": np.nan,
                "k_width": np.nan,
                "sum_mode_fit_weight": 0.0,
                "anchor_rows": 0,
                "strict_rows": 0,
                "broad_rows": 0,
                "repair_rows": 0,
                "convexity_flag_rows": 0,
                "roughness_flag_rows": 0,
                "median_iv": np.nan,
                "median_total_variance": np.nan,
                "eligible_for_mode_fit": False,
            }
        )
        continue

    grouped = pool.groupby("expiry", dropna=False)

    for expiry, grp in grouped:
        min_k = float(grp["log_moneyness"].min())
        max_k = float(grp["log_moneyness"].max())
        k_width = max_k - min_k

        rows = int(len(grp))
        unique_strikes = int(grp["strike"].nunique())
        sum_weight = float(grp["mode_fit_weight"].sum())

        eligible = (
            rows >= mode.minimum_rows
            and unique_strikes >= mode.minimum_unique_strikes
            and k_width >= mode.minimum_k_width
            and sum_weight > 0
        )

        support_rows.append(
            {
                "mode_name": mode_name,
                "expiry": expiry,
                "rows": rows,
                "unique_contracts": int(grp["contract_key"].nunique()),
                "unique_strikes": unique_strikes,
                "min_ttm": float(grp["tau_years"].min()),
                "max_ttm": float(grp["tau_years"].max()),
                "min_k": min_k,
                "max_k": max_k,
                "k_width": k_width,
                "sum_mode_fit_weight": sum_weight,
                "anchor_rows": int(grp["in_anchor"].sum()),
                "strict_rows": int(grp["in_strict"].sum()),
                "broad_rows": int(grp["in_broad"].sum()),
                "repair_rows": int(grp["in_repair"].sum()),
                "convexity_flag_rows": int(grp["has_convexity_violation"].sum()),
                "roughness_flag_rows": int(grp["touched_by_any_iv_roughness_flag"].sum()),
                "median_iv": float(grp["selected_iv"].median()),
                "median_total_variance": float(grp["total_variance"].median()),
                "eligible_for_mode_fit": bool(eligible),
            }
        )

n11_svi_mode_expiry_support = (
    pd.DataFrame(support_rows)
    .sort_values(["mode_name", "expiry"], na_position="last")
    .reset_index(drop=True)
)

n11_svi_mode_summary = (
    n11_svi_mode_expiry_support
    .groupby("mode_name", dropna=False)
    .agg(
        expiries=("expiry", "nunique"),
        eligible_expiries=("eligible_for_mode_fit", "sum"),
        total_rows=("rows", "sum"),
        total_unique_contracts=("unique_contracts", "sum"),
        total_weight=("sum_mode_fit_weight", "sum"),
        median_rows_per_expiry=("rows", "median"),
        median_unique_strikes_per_expiry=("unique_strikes", "median"),
        median_k_width=("k_width", "median"),
        total_convexity_flag_rows=("convexity_flag_rows", "sum"),
        total_roughness_flag_rows=("roughness_flag_rows", "sum"),
    )
    .reset_index()
)

n11_svi_mode_summary["passes_min_expiry_gate"] = (
    n11_svi_mode_summary["eligible_expiries"] >= CONFIG.min_primary_expiries
)


# ------------------------------------------------------------
# Initial SVI guess construction
# ------------------------------------------------------------

def weighted_quantile(values: np.ndarray, weights: np.ndarray, quantile: float) -> float:
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)

    values = values[valid]
    weights = weights[valid]

    if len(values) == 0:
        return np.nan

    sorter = np.argsort(values)
    values = values[sorter]
    weights = weights[sorter]

    cumulative = np.cumsum(weights)
    cutoff = quantile * cumulative[-1]

    return float(values[np.searchsorted(cumulative, cutoff, side="left")])


def robust_weighted_mean(values: np.ndarray, weights: np.ndarray, default: float = np.nan) -> float:
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)

    if not valid.any():
        return float(default)

    return float(np.average(values[valid], weights=weights[valid]))


def estimate_raw_svi_initial_guess(grp: pd.DataFrame) -> RawSVIParams:
    """
    Build a stable raw-SVI initial guess from the observed total-variance smile.

    This is intentionally conservative. The nonlinear optimizer can move away
    from this point, but the seed should:
      - keep total variance positive
      - start near observed ATM variance
      - use observed smile slope/skew only mildly
      - avoid extreme rho and sigma values
    """
    k = pd.to_numeric(grp["log_moneyness"], errors="coerce").to_numpy(dtype=float)
    w = pd.to_numeric(grp["total_variance"], errors="coerce").to_numpy(dtype=float)
    weight = pd.to_numeric(grp["mode_fit_weight"], errors="coerce").to_numpy(dtype=float)

    valid = np.isfinite(k) & np.isfinite(w) & np.isfinite(weight) & (w > 0) & (weight > 0)
    k = k[valid]
    w = w[valid]
    weight = weight[valid]

    if len(k) < 3:
        return RawSVIParams(a=0.0010, b=0.0500, rho=-0.3000, m=0.0000, sigma=0.1000)

    k_min = float(np.min(k))
    k_max = float(np.max(k))
    k_width = max(k_max - k_min, 0.05)

    atm_idx = int(np.argmin(np.abs(k)))
    atm_w = float(w[atm_idx])

    weighted_median_k = weighted_quantile(k, weight, 0.50)
    weighted_median_w = weighted_quantile(w, weight, 0.50)

    left_mask = k <= weighted_quantile(k, weight, 0.25)
    right_mask = k >= weighted_quantile(k, weight, 0.75)

    left_w = robust_weighted_mean(w[left_mask], weight[left_mask], default=weighted_median_w)
    right_w = robust_weighted_mean(w[right_mask], weight[right_mask], default=weighted_median_w)

    skew_signal = right_w - left_w

    # Index smiles typically have negative skew: left wing higher than right wing.
    rho0 = float(np.clip(3.0 * skew_signal / max(abs(left_w) + abs(right_w), N11_EPS), -0.75, 0.75))

    # Start with a reasonably smooth smile width.
    sigma0 = float(np.clip(0.35 * k_width, 0.025, 0.50))

    # Use wing spread to initialize b, bounded conservatively.
    wing_w_spread = float(np.nanpercentile(w, 90) - np.nanpercentile(w, 10))
    b0 = float(np.clip(wing_w_spread / max(k_width, 0.05), 0.005, 1.00))

    m0 = float(np.clip(weighted_median_k, k_min, k_max))

    # Pick a so that the ATM level is roughly matched.
    x_atm = 0.0 - m0
    root_atm = math.sqrt(x_atm * x_atm + sigma0 * sigma0)
    a0 = atm_w - b0 * (rho0 * x_atm + root_atm)

    # Keep the analytic minimum positive.
    min_variance_buffer = max(1.0e-6, 0.05 * atm_w)
    w_min0 = a0 + b0 * sigma0 * math.sqrt(max(1.0 - rho0 * rho0, N11_TINY))
    if w_min0 < min_variance_buffer:
        a0 = a0 + (min_variance_buffer - w_min0)

    a0 = float(np.clip(a0, CONFIG.svi_a_lower, CONFIG.svi_a_upper))
    b0 = float(np.clip(b0, CONFIG.svi_b_lower * 10.0, CONFIG.svi_b_upper))
    rho0 = float(np.clip(rho0, CONFIG.svi_rho_lower + 1.0e-4, CONFIG.svi_rho_upper - 1.0e-4))
    m0 = float(np.clip(m0, CONFIG.svi_m_lower, CONFIG.svi_m_upper))
    sigma0 = float(np.clip(sigma0, CONFIG.svi_sigma_lower * 10.0, CONFIG.svi_sigma_upper))

    theta0 = RawSVIParams(a=a0, b=b0, rho=rho0, m=m0, sigma=sigma0)

    if not raw_svi_parameter_constraints(theta0)["basic_feasible"]:
        theta0 = RawSVIParams(
            a=max(min_variance_buffer, atm_w * 0.50),
            b=max(0.005, b0),
            rho=np.clip(rho0, -0.50, 0.50),
            m=m0,
            sigma=max(0.050, sigma0),
        )

    return theta0


def make_svi_multistart_seeds(
    base_theta: RawSVIParams,
    grp: pd.DataFrame,
    *,
    n_multistart: int,
    rng: np.random.Generator,
) -> pd.DataFrame:
    """
    Deterministic + randomized seed generation around a robust base point.

    Seeds are bounded and then shifted upward if the analytic SVI minimum is
    negative. This keeps the optimizer away from immediately invalid regions.
    """
    base = base_theta.as_array()

    k = pd.to_numeric(grp["log_moneyness"], errors="coerce")
    w = pd.to_numeric(grp["total_variance"], errors="coerce")

    finite_k = k[np.isfinite(k)]
    finite_w = w[np.isfinite(w) & (w > 0)]

    k_min = float(finite_k.min()) if len(finite_k) else -0.25
    k_max = float(finite_k.max()) if len(finite_k) else 0.25
    k_width = max(k_max - k_min, 0.05)
    median_w = float(finite_w.median()) if len(finite_w) else max(base[0], 0.001)

    deterministic = []

    a_scales = [0.50, 0.80, 1.00, 1.25, 1.75]
    b_scales = [0.50, 0.85, 1.00, 1.40, 2.00]
    rho_values = [
        base[2],
        -0.75,
        -0.50,
        -0.25,
        0.00,
        0.25,
    ]
    m_values = [
        base[3],
        0.0,
        float(np.clip(k_min + 0.25 * k_width, CONFIG.svi_m_lower, CONFIG.svi_m_upper)),
        float(np.clip(k_min + 0.50 * k_width, CONFIG.svi_m_lower, CONFIG.svi_m_upper)),
        float(np.clip(k_min + 0.75 * k_width, CONFIG.svi_m_lower, CONFIG.svi_m_upper)),
    ]
    sigma_values = [
        base[4],
        float(np.clip(0.15 * k_width, CONFIG.svi_sigma_lower * 10.0, CONFIG.svi_sigma_upper)),
        float(np.clip(0.30 * k_width, CONFIG.svi_sigma_lower * 10.0, CONFIG.svi_sigma_upper)),
        float(np.clip(0.60 * k_width, CONFIG.svi_sigma_lower * 10.0, CONFIG.svi_sigma_upper)),
    ]

    for a_scale, b_scale, rho, m, sigma in itertools.product(
        a_scales,
        b_scales,
        rho_values,
        m_values,
        sigma_values,
    ):
        deterministic.append(
            np.array(
                [
                    max(CONFIG.svi_a_lower, min(CONFIG.svi_a_upper, median_w * a_scale)),
                    max(CONFIG.svi_b_lower * 10.0, min(CONFIG.svi_b_upper, base[1] * b_scale)),
                    np.clip(rho, CONFIG.svi_rho_lower + 1.0e-4, CONFIG.svi_rho_upper - 1.0e-4),
                    np.clip(m, CONFIG.svi_m_lower, CONFIG.svi_m_upper),
                    np.clip(sigma, CONFIG.svi_sigma_lower * 10.0, CONFIG.svi_sigma_upper),
                ],
                dtype=float,
            )
        )

    # Put the base seed first.
    seeds = [base.copy()]
    seeds.extend(deterministic)

    # Random seeds around data scale.
    while len(seeds) < n_multistart:
        a_rand = rng.uniform(0.20, 1.80) * median_w
        b_rand = 10.0 ** rng.uniform(-2.5, 0.0)
        rho_rand = rng.uniform(-0.90, 0.40)
        m_rand = rng.uniform(k_min - 0.10 * k_width, k_max + 0.10 * k_width)
        sigma_rand = rng.uniform(0.05 * k_width, 0.90 * k_width)

        seeds.append(
            np.array(
                [
                    a_rand,
                    b_rand,
                    rho_rand,
                    m_rand,
                    sigma_rand,
                ],
                dtype=float,
            )
        )

    cleaned = []
    seen = set()

    for seed in seeds:
        seed = np.asarray(seed, dtype=float).copy()

        seed[0] = np.clip(seed[0], CONFIG.svi_a_lower, CONFIG.svi_a_upper)
        seed[1] = np.clip(seed[1], CONFIG.svi_b_lower * 10.0, CONFIG.svi_b_upper)
        seed[2] = np.clip(seed[2], CONFIG.svi_rho_lower + 1.0e-4, CONFIG.svi_rho_upper - 1.0e-4)
        seed[3] = np.clip(seed[3], CONFIG.svi_m_lower, CONFIG.svi_m_upper)
        seed[4] = np.clip(seed[4], CONFIG.svi_sigma_lower * 10.0, CONFIG.svi_sigma_upper)

        min_info = raw_svi_minimum_total_variance(seed)
        min_buffer = max(1.0e-8, 0.01 * median_w)
        if min_info["w_min"] < min_buffer:
            seed[0] = seed[0] + (min_buffer - min_info["w_min"])
            seed[0] = np.clip(seed[0], CONFIG.svi_a_lower, CONFIG.svi_a_upper)

        key = tuple(np.round(seed, 10))
        if key in seen:
            continue
        seen.add(key)

        feasibility = raw_svi_parameter_constraints(seed)
        if feasibility["basic_feasible"]:
            cleaned.append(seed)

        if len(cleaned) >= n_multistart:
            break

    # Guaranteed fallback if strict filtering removed too much.
    if len(cleaned) == 0:
        cleaned = [base.copy()]

    rows = []

    for seed_rank, seed in enumerate(cleaned):
        feasibility = raw_svi_parameter_constraints(seed)
        rows.append(
            {
                "seed_rank": seed_rank,
                "a": float(seed[0]),
                "b": float(seed[1]),
                "rho": float(seed[2]),
                "m": float(seed[3]),
                "sigma": float(seed[4]),
                "k_min_analytic": float(feasibility["k_min"]),
                "w_min_analytic": float(feasibility["w_min"]),
                "basic_feasible": bool(feasibility["basic_feasible"]),
                "is_base_seed": seed_rank == 0,
            }
        )

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Build initial guesses and multistart seed table
# ------------------------------------------------------------

base_guess_rows = []
seed_frames = []

for mode_name, pool in N11_SVI_MODE_POOLS.items():
    mode_support = n11_svi_mode_expiry_support.query(
        "mode_name == @mode_name and eligible_for_mode_fit == True"
    )

    for _, support_row in mode_support.iterrows():
        expiry = support_row["expiry"]
        grp = pool.loc[pool["expiry"] == expiry].copy()

        if len(grp) == 0:
            continue

        theta0 = estimate_raw_svi_initial_guess(grp)
        theta0_constraints = raw_svi_parameter_constraints(theta0)

        base_guess_rows.append(
            {
                "mode_name": mode_name,
                "expiry": expiry,
                "rows": len(grp),
                "unique_strikes": int(grp["strike"].nunique()),
                "k_min_observed": float(grp["log_moneyness"].min()),
                "k_max_observed": float(grp["log_moneyness"].max()),
                "k_width_observed": float(grp["log_moneyness"].max() - grp["log_moneyness"].min()),
                "median_total_variance": float(grp["total_variance"].median()),
                "median_iv": float(grp["selected_iv"].median()),
                "a0": theta0.a,
                "b0": theta0.b,
                "rho0": theta0.rho,
                "m0": theta0.m,
                "sigma0": theta0.sigma,
                "k_min_analytic": float(theta0_constraints["k_min"]),
                "w_min_analytic": float(theta0_constraints["w_min"]),
                "basic_feasible": bool(theta0_constraints["basic_feasible"]),
            }
        )

        seeds = make_svi_multistart_seeds(
            theta0,
            grp,
            n_multistart=CONFIG.n_multistart,
            rng=RNG,
        )
        seeds.insert(0, "mode_name", mode_name)
        seeds.insert(1, "expiry", expiry)
        seeds.insert(2, "rows", len(grp))
        seeds.insert(3, "unique_strikes", int(grp["strike"].nunique()))

        seed_frames.append(seeds)

n11_svi_initial_guess_summary = (
    pd.DataFrame(base_guess_rows)
    .sort_values(["mode_name", "expiry"])
    .reset_index(drop=True)
)

N11_SVI_SEED_TABLE = (
    pd.concat(seed_frames, ignore_index=True)
    if seed_frames
    else pd.DataFrame(
        columns=[
            "mode_name",
            "expiry",
            "rows",
            "unique_strikes",
            "seed_rank",
            "a",
            "b",
            "rho",
            "m",
            "sigma",
            "k_min_analytic",
            "w_min_analytic",
            "basic_feasible",
            "is_base_seed",
        ]
    )
)

n11_svi_seed_summary = (
    N11_SVI_SEED_TABLE
    .groupby(["mode_name", "expiry"], dropna=False)
    .agg(
        seeds=("seed_rank", "count"),
        feasible_seeds=("basic_feasible", "sum"),
        min_seed_w_min=("w_min_analytic", "min"),
        max_seed_w_min=("w_min_analytic", "max"),
        base_seed_count=("is_base_seed", "sum"),
    )
    .reset_index()
    if len(N11_SVI_SEED_TABLE)
    else pd.DataFrame(
        columns=[
            "mode_name",
            "expiry",
            "seeds",
            "feasible_seeds",
            "min_seed_w_min",
            "max_seed_w_min",
            "base_seed_count",
        ]
    )
)

n11_svi_candidate_design_gate = pd.DataFrame(
    [
        {
            "check_name": "candidate_modes_defined",
            "check_type": "blocking",
            "passes": len(N11_SVI_FIT_MODES) >= 2,
            "observed_value": float(len(N11_SVI_FIT_MODES)),
            "threshold_or_requirement": ">= 2",
        },
        {
            "check_name": "weighted_broad_mode_available",
            "check_type": "blocking",
            "passes": "weighted_broad_svi" in N11_SVI_FIT_MODES,
            "observed_value": float("weighted_broad_svi" in N11_SVI_FIT_MODES),
            "threshold_or_requirement": "True",
        },
        {
            "check_name": "at_least_one_mode_passes_expiry_gate",
            "check_type": "blocking",
            "passes": bool(n11_svi_mode_summary["passes_min_expiry_gate"].any()),
            "observed_value": float(n11_svi_mode_summary["passes_min_expiry_gate"].sum()),
            "threshold_or_requirement": ">= 1 mode",
        },
        {
            "check_name": "weighted_broad_passes_expiry_gate",
            "check_type": "blocking",
            "passes": bool(
                n11_svi_mode_summary
                .query("mode_name == 'weighted_broad_svi'")["passes_min_expiry_gate"]
                .iloc[0]
            ),
            "observed_value": float(
                n11_svi_mode_summary
                .query("mode_name == 'weighted_broad_svi'")["eligible_expiries"]
                .iloc[0]
            ),
            "threshold_or_requirement": f">= {CONFIG.min_primary_expiries}",
        },
        {
            "check_name": "initial_guess_table_nonempty",
            "check_type": "blocking",
            "passes": len(n11_svi_initial_guess_summary) > 0,
            "observed_value": float(len(n11_svi_initial_guess_summary)),
            "threshold_or_requirement": "> 0",
        },
        {
            "check_name": "all_base_guesses_feasible",
            "check_type": "blocking",
            "passes": bool(n11_svi_initial_guess_summary["basic_feasible"].all())
            if len(n11_svi_initial_guess_summary)
            else False,
            "observed_value": float(n11_svi_initial_guess_summary["basic_feasible"].mean())
            if len(n11_svi_initial_guess_summary)
            else 0.0,
            "threshold_or_requirement": "all True",
        },
        {
            "check_name": "seed_table_nonempty",
            "check_type": "blocking",
            "passes": len(N11_SVI_SEED_TABLE) > 0,
            "observed_value": float(len(N11_SVI_SEED_TABLE)),
            "threshold_or_requirement": "> 0",
        },
        {
            "check_name": "all_seeds_feasible",
            "check_type": "blocking",
            "passes": bool(N11_SVI_SEED_TABLE["basic_feasible"].all())
            if len(N11_SVI_SEED_TABLE)
            else False,
            "observed_value": float(N11_SVI_SEED_TABLE["basic_feasible"].mean())
            if len(N11_SVI_SEED_TABLE)
            else 0.0,
            "threshold_or_requirement": "all True",
        },
    ]
)

N11_SVI_CANDIDATE_DESIGN_PASS = bool(
    n11_svi_candidate_design_gate
    .query("check_type == 'blocking'")["passes"]
    .all()
)


# ------------------------------------------------------------
# Persist candidate-design artifacts
# ------------------------------------------------------------

_write_table_pair(
    n11_svi_fit_mode_summary,
    "n11_svi_fit_mode_summary",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_svi_mode_expiry_support,
    "n11_svi_mode_expiry_support",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_svi_mode_summary,
    "n11_svi_mode_summary",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_svi_initial_guess_summary,
    "n11_svi_initial_guess_summary",
    N11_TABLE_DIR,
)

_write_table_pair(
    N11_SVI_SEED_TABLE,
    "n11_svi_seed_table",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_svi_seed_summary,
    "n11_svi_seed_summary",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_svi_candidate_design_gate,
    "n11_svi_candidate_design_gate",
    N11_TABLE_DIR,
)


# ------------------------------------------------------------
# Console summary
# ------------------------------------------------------------

print("Notebook 11 SVI candidate design and initialization complete.")
print(f"Candidate design pass: {N11_SVI_CANDIDATE_DESIGN_PASS}")
print(f"Candidate modes: {len(N11_SVI_FIT_MODES):,}")
print(f"Initial guess rows: {len(n11_svi_initial_guess_summary):,}")
print(f"Multistart seed rows: {len(N11_SVI_SEED_TABLE):,}")

display(n11_svi_mode_summary)
display(n11_svi_initial_guess_summary)
display(n11_svi_seed_summary)
display(n11_svi_candidate_design_gate)

if not N11_SVI_CANDIDATE_DESIGN_PASS:
    raise RuntimeError(
        "Notebook 11 SVI candidate design gate failed. "
        "Do not proceed to nonlinear SVI fitting until this layer passes."
    )

Notebook 11 SVI candidate design and initialization complete.
Candidate design pass: True
Candidate modes: 3
Initial guess rows: 20
Multistart seed rows: 1,280


,mode_name,expiries,eligible_expiries,total_rows,total_unique_contracts,total_weight,median_rows_per_expiry,median_unique_strikes_per_expiry,median_k_width,total_convexity_flag_rows,total_roughness_flag_rows,passes_min_expiry_gate
0,repair_aware_svi,8,8,1260,1260,"1,304.5205585940",162.5000000000,162.5000000000,0.4577713943,319,258,True
1,strict_svi,8,4,224,224,"1,018.9060574314",30.0000000000,30.0000000000,0.0758496368,0,0,True
2,weighted_broad_svi,8,8,1260,1260,"1,411.9522124531",162.5000000000,162.5000000000,0.4577713943,319,258,True


,mode_name,expiry,rows,unique_strikes,k_min_observed,k_max_observed,k_width_observed,median_total_variance,median_iv,a0,b0,rho0,m0,sigma0,k_min_analytic,w_min_analytic,basic_feasible
0,repair_aware_svi,2026-07-10 00:00:00+00:00,97,97,-0.1242195547,0.0303652471,0.1545848018,0.0004515581,0.1815591998,-0.0002770124,0.0102708038,-0.7500000000,-0.0034016153,0.0541046806,0.0579473260,0.0000905477,True
1,repair_aware_svi,2026-07-17 00:00:00+00:00,130,130,-0.3318085078,0.0516084631,0.3834169709,0.0012528117,0.1952066471,-0.0010329963,0.0127296008,-0.7500000000,-0.0101314301,0.1341959398,0.1420324629,0.0000969120,True
2,repair_aware_svi,2026-07-24 00:00:00+00:00,152,152,-0.2958546667,0.0685381653,0.3643928320,0.0017981710,0.1858582741,-0.0003291639,0.0106662795,-0.7500000000,-0.0095248595,0.1275374912,0.1350890624,0.0005706234,True
3,repair_aware_svi,2026-07-31 00:00:00+00:00,175,175,-0.4009080739,0.0815180753,0.4824261492,0.0029943278,0.2050262310,-0.0008554449,0.0139944932,-0.7500000000,-0.0170071437,0.1688491522,0.1744497988,0.0007075052,True
4,repair_aware_svi,2026-08-07 00:00:00+00:00,131,131,-0.3342401078,0.0988765317,0.4331166394,0.0026118553,0.1699667678,-0.0007753134,0.0186196225,-0.7500000000,-0.0166363554,0.1515908238,0.1552514821,0.0010916372,True
5,repair_aware_svi,2026-08-21 00:00:00+00:00,173,173,-0.5084862228,0.1391985836,0.6476848065,0.0048647314,0.1943689717,-0.0034247193,0.0280520463,-0.7500000000,-0.0137899810,0.2266896823,0.2432519579,0.0007814363,True
6,repair_aware_svi,2026-08-31 00:00:00+00:00,187,187,-0.3842214430,0.1207044586,0.5049259016,0.0059836156,0.1957450772,-0.0003354978,0.0207982046,-0.7500000000,-0.0106315434,0.1767240656,0.1897547115,0.0020956456,True
7,repair_aware_svi,2026-09-18 00:00:00+00:00,215,215,-0.4252077096,0.2047605693,0.6299682789,0.0088376178,0.2073878979,-0.0002063745,0.0239692795,-0.7500000000,-0.0156692761,0.2204888976,0.2343416339,0.0032892980,True
8,strict_svi,2026-07-24 00:00:00+00:00,29,29,-0.0637424454,0.0198105774,0.0835530228,0.0010532855,0.1422468575,0.0007797177,0.0071877011,-0.7500000000,-0.0068221552,0.0292435580,0.0263369228,0.0009187480,True
9,strict_svi,2026-07-31 00:00:00+00:00,45,45,-0.1493714481,0.0243596615,0.1737311096,0.0020140017,0.1681472151,0.0004888754,0.0213889825,-0.7500000000,-0.0183704705,0.0608058884,0.0505769262,0.0013491257,True


,mode_name,expiry,seeds,feasible_seeds,min_seed_w_min,max_seed_w_min,base_seed_count
0,repair_aware_svi,2026-07-10 00:00:00+00:00,64,64,0.0000905477,0.0006869672,1
1,repair_aware_svi,2026-07-17 00:00:00+00:00,64,64,0.0000969120,0.0020441341,1
2,repair_aware_svi,2026-07-24 00:00:00+00:00,64,64,0.0005706234,0.0020280744,1
3,repair_aware_svi,2026-07-31 00:00:00+00:00,64,64,0.0007075052,0.0034582421,1
4,repair_aware_svi,2026-08-07 00:00:00+00:00,64,64,0.0010916372,0.0036484440,1
5,repair_aware_svi,2026-08-21 00:00:00+00:00,64,64,0.0007814363,0.0077099496,1
6,repair_aware_svi,2026-08-31 00:00:00+00:00,64,64,0.0020956456,0.0060422331,1
7,repair_aware_svi,2026-09-18 00:00:00+00:00,64,64,0.0032892980,0.0088049294,1
8,strict_svi,2026-07-24 00:00:00+00:00,64,64,0.0005564349,0.0009187480,1
9,strict_svi,2026-07-31 00:00:00+00:00,64,64,0.0011913402,0.0020863815,1


,check_name,check_type,passes,observed_value,threshold_or_requirement
0,candidate_modes_defined,blocking,True,3.0000000000,>= 2
1,weighted_broad_mode_available,blocking,True,1.0000000000,True
2,at_least_one_mode_passes_expiry_gate,blocking,True,3.0000000000,>= 1 mode
3,weighted_broad_passes_expiry_gate,blocking,True,8.0000000000,>= 4
4,initial_guess_table_nonempty,blocking,True,20.0000000000,> 0
5,all_base_guesses_feasible,blocking,True,1.0000000000,all True
6,seed_table_nonempty,blocking,True,"1,280.0000000000",> 0
7,all_seeds_feasible,blocking,True,1.0000000000,all True


In [6]:
# ------------------------------------------------------------
# Notebook 11 cell 06: Multistart SVI fitting by expiry and candidate mode
# ------------------------------------------------------------
# This cell runs nonlinear least-squares SVI fits for every eligible:
#
#   (candidate mode, expiry, seed)
#
# It produces:
#
#   1. full multistart fit result ledger
#   2. best-fit SVI parameters by mode and expiry
#   3. fitted contract-level points
#   4. fitted diagnostic grids
#   5. fit gate before repair/calendar checks
#
# This cell does NOT choose the final repaired surface yet.
# It fits candidate local SVI slices and records diagnostics.

from __future__ import annotations

import time
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.optimize import least_squares


# ------------------------------------------------------------
# Pre-flight checks
# ------------------------------------------------------------

required_globals_for_fit = [
    "N11_SVI_CANDIDATE_DESIGN_PASS",
    "N11_SVI_SEED_TABLE",
    "N11_SVI_MODE_POOLS",
    "N11_SVI_FIT_MODES",
    "raw_svi_total_variance",
    "raw_svi_parameter_constraints",
    "raw_svi_minimum_total_variance",
    "svi_butterfly_g",
    "svi_slice_grid_diagnostics",
    "summarize_svi_grid_diagnostics",
    "bsm_forward_price_from_total_variance",
    "SVI_BOUNDS_LOWER",
    "SVI_BOUNDS_UPPER",
    "CONFIG",
]

missing_fit_globals = [name for name in required_globals_for_fit if name not in globals()]
if missing_fit_globals:
    raise NameError(f"Missing required objects before SVI fitting: {missing_fit_globals}")

if not bool(N11_SVI_CANDIDATE_DESIGN_PASS):
    raise RuntimeError("SVI candidate design did not pass. Run/fix the previous cell first.")

if len(N11_SVI_SEED_TABLE) == 0:
    raise RuntimeError("N11_SVI_SEED_TABLE is empty. Cannot fit SVI slices.")


# ------------------------------------------------------------
# Objective configuration
# ------------------------------------------------------------

@dataclass(frozen=True)
class SVIObjectiveConfig:
    mode_name: str
    market_residual_weight: float
    positive_variance_penalty: float
    analytic_min_variance_penalty: float
    butterfly_penalty: float
    parameter_soft_penalty: float
    butterfly_scale: float
    objective_label: str


N11_SVI_OBJECTIVE_CONFIGS: dict[str, SVIObjectiveConfig] = {
    "strict_svi": SVIObjectiveConfig(
        mode_name="strict_svi",
        market_residual_weight=1.00,
        positive_variance_penalty=25.00,
        analytic_min_variance_penalty=50.00,
        butterfly_penalty=0.02,
        parameter_soft_penalty=0.01,
        butterfly_scale=0.0100,
        objective_label="strict_market_fit_light_butterfly_penalty",
    ),
    "weighted_broad_svi": SVIObjectiveConfig(
        mode_name="weighted_broad_svi",
        market_residual_weight=1.00,
        positive_variance_penalty=35.00,
        analytic_min_variance_penalty=75.00,
        butterfly_penalty=0.08,
        parameter_soft_penalty=0.01,
        butterfly_scale=0.0100,
        objective_label="weighted_market_fit_moderate_butterfly_penalty",
    ),
    "repair_aware_svi": SVIObjectiveConfig(
        mode_name="repair_aware_svi",
        market_residual_weight=1.00,
        positive_variance_penalty=50.00,
        analytic_min_variance_penalty=100.00,
        butterfly_penalty=0.25,
        parameter_soft_penalty=0.02,
        butterfly_scale=0.0100,
        objective_label="repair_aware_fit_heavier_butterfly_penalty",
    ),
}


def get_objective_config(mode_name: str) -> SVIObjectiveConfig:
    if mode_name in N11_SVI_OBJECTIVE_CONFIGS:
        return N11_SVI_OBJECTIVE_CONFIGS[mode_name]

    return SVIObjectiveConfig(
        mode_name=mode_name,
        market_residual_weight=1.00,
        positive_variance_penalty=35.00,
        analytic_min_variance_penalty=75.00,
        butterfly_penalty=0.05,
        parameter_soft_penalty=0.01,
        butterfly_scale=0.0100,
        objective_label="default_svi_objective",
    )


n11_svi_objective_config_summary = pd.DataFrame(
    [
        {
            "mode_name": cfg.mode_name,
            "market_residual_weight": cfg.market_residual_weight,
            "positive_variance_penalty": cfg.positive_variance_penalty,
            "analytic_min_variance_penalty": cfg.analytic_min_variance_penalty,
            "butterfly_penalty": cfg.butterfly_penalty,
            "parameter_soft_penalty": cfg.parameter_soft_penalty,
            "butterfly_scale": cfg.butterfly_scale,
            "objective_label": cfg.objective_label,
        }
        for cfg in N11_SVI_OBJECTIVE_CONFIGS.values()
    ]
)


# ------------------------------------------------------------
# Fit-array preparation
# ------------------------------------------------------------

def _finite_positive_weighted_mask(
    k: np.ndarray,
    w: np.ndarray,
    tau: np.ndarray,
    weight: np.ndarray,
) -> np.ndarray:
    return (
        np.isfinite(k)
        & np.isfinite(w)
        & np.isfinite(tau)
        & np.isfinite(weight)
        & (w > CONFIG.positive_total_variance_floor)
        & (tau > 0.0)
        & (weight > 0.0)
    )


@dataclass(frozen=True)
class SVIFitArrays:
    mode_name: str
    expiry: pd.Timestamp
    contract_key: np.ndarray
    k: np.ndarray
    tau: np.ndarray
    observed_w: np.ndarray
    observed_iv: np.ndarray
    weight: np.ndarray
    normalized_weight: np.ndarray
    forward: np.ndarray
    strike: np.ndarray
    discount_factor: np.ndarray
    option_type: np.ndarray
    market_price: np.ndarray
    scale_w: float
    k_grid: np.ndarray


def prepare_svi_fit_arrays(mode_name: str, expiry: pd.Timestamp) -> SVIFitArrays:
    pool = N11_SVI_MODE_POOLS[mode_name].copy()
    grp = pool.loc[pool["expiry"] == expiry].copy()

    if len(grp) == 0:
        raise ValueError(f"No rows found for mode={mode_name}, expiry={expiry}")

    k = pd.to_numeric(grp["log_moneyness"], errors="coerce").to_numpy(dtype=float)
    w = pd.to_numeric(grp["total_variance"], errors="coerce").to_numpy(dtype=float)
    tau = pd.to_numeric(grp["tau_years"], errors="coerce").to_numpy(dtype=float)
    weight = pd.to_numeric(grp["mode_fit_weight"], errors="coerce").to_numpy(dtype=float)

    valid = _finite_positive_weighted_mask(k, w, tau, weight)
    grp = grp.loc[valid].copy()

    if len(grp) == 0:
        raise ValueError(f"No valid weighted rows for mode={mode_name}, expiry={expiry}")

    k = pd.to_numeric(grp["log_moneyness"], errors="coerce").to_numpy(dtype=float)
    w = pd.to_numeric(grp["total_variance"], errors="coerce").to_numpy(dtype=float)
    tau = pd.to_numeric(grp["tau_years"], errors="coerce").to_numpy(dtype=float)
    observed_iv = pd.to_numeric(grp["selected_iv"], errors="coerce").to_numpy(dtype=float)
    weight = pd.to_numeric(grp["mode_fit_weight"], errors="coerce").to_numpy(dtype=float)

    normalized_weight = weight / max(float(np.nanmean(weight)), N11_TINY)
    normalized_weight = np.clip(normalized_weight, 1.0e-8, 1.0e4)

    scale_w = float(np.nanmedian(w))
    scale_w = max(scale_w, 1.0e-5)

    if expiry in N11_EXPIRY_K_GRIDS:
        k_grid = np.asarray(N11_EXPIRY_K_GRIDS[expiry], dtype=float)
    else:
        k_grid = make_k_grid_from_bounds(float(np.min(k)), float(np.max(k)))

    market_price = pd.to_numeric(grp["market_selected_price"], errors="coerce")
    fallback_mid = pd.to_numeric(grp["mid_price"], errors="coerce")
    market_price = market_price.where(np.isfinite(market_price), fallback_mid).to_numpy(dtype=float)

    return SVIFitArrays(
        mode_name=mode_name,
        expiry=expiry,
        contract_key=grp["contract_key"].astype(str).to_numpy(),
        k=k,
        tau=tau,
        observed_w=w,
        observed_iv=observed_iv,
        weight=weight,
        normalized_weight=normalized_weight,
        forward=pd.to_numeric(grp["forward"], errors="coerce").to_numpy(dtype=float),
        strike=pd.to_numeric(grp["strike"], errors="coerce").to_numpy(dtype=float),
        discount_factor=pd.to_numeric(grp["discount_factor"], errors="coerce").to_numpy(dtype=float),
        option_type=grp["option_type"].astype(str).to_numpy(),
        market_price=market_price,
        scale_w=scale_w,
        k_grid=k_grid,
    )


# ------------------------------------------------------------
# SVI residual function
# ------------------------------------------------------------

def svi_objective_residuals(theta: np.ndarray, arrays: SVIFitArrays) -> np.ndarray:
    cfg = get_objective_config(arrays.mode_name)
    theta = np.asarray(theta, dtype=float)

    if not np.isfinite(theta).all():
        return np.full(len(arrays.k) + 2 * len(arrays.k_grid) + 8, 1.0e6, dtype=float)

    a, b, rho, m, sigma = theta

    # Defensive handling. Bounds should already enforce these, but residuals
    # need to stay finite during optimizer exploration.
    if b <= 0.0 or sigma <= 0.0 or abs(rho) >= 1.0:
        return np.full(len(arrays.k) + 2 * len(arrays.k_grid) + 8, 1.0e6, dtype=float)

    fitted_w = raw_svi_total_variance(arrays.k, theta)
    fitted_grid_w = raw_svi_total_variance(arrays.k_grid, theta)

    if (not np.isfinite(fitted_w).all()) or (not np.isfinite(fitted_grid_w).all()):
        return np.full(len(arrays.k) + 2 * len(arrays.k_grid) + 8, 1.0e6, dtype=float)

    sqrt_weight = np.sqrt(arrays.normalized_weight)

    market_residual = (
        np.sqrt(cfg.market_residual_weight)
        * sqrt_weight
        * (fitted_w - arrays.observed_w)
        / arrays.scale_w
    )

    # Positive-variance penalty on the diagnostic grid.
    variance_floor = CONFIG.positive_total_variance_floor
    grid_negative_variance_residual = (
        np.sqrt(cfg.positive_variance_penalty)
        * np.minimum(fitted_grid_w - variance_floor, 0.0)
        / arrays.scale_w
    )

    # Analytic minimum penalty.
    min_info = raw_svi_minimum_total_variance(theta)
    min_w_gap = min_info["w_min"] - variance_floor
    analytic_min_residual = np.array(
        [
            np.sqrt(cfg.analytic_min_variance_penalty)
            * min(min_w_gap, 0.0)
            / arrays.scale_w
        ],
        dtype=float,
    )

    # Butterfly penalty on diagnostic grid.
    g = svi_butterfly_g(arrays.k_grid, theta)
    if np.isfinite(g).all():
        butterfly_residual = (
            np.sqrt(cfg.butterfly_penalty)
            * np.minimum(g - CONFIG.density_tolerance, 0.0)
            / max(cfg.butterfly_scale, N11_TINY)
        )
    else:
        butterfly_residual = np.full(len(arrays.k_grid), 1.0e6, dtype=float)

    # Very soft shape regularization to discourage boundary abuse.
    # These are not hard constraints; hard parameter bounds are already used.
    param_soft_residual = np.array(
        [
            np.sqrt(cfg.parameter_soft_penalty) * max(abs(rho) - 0.95, 0.0) / 0.05,
            np.sqrt(cfg.parameter_soft_penalty) * max(sigma - 2.0, 0.0) / 2.0,
            np.sqrt(cfg.parameter_soft_penalty) * max(b - 2.0, 0.0) / 2.0,
        ],
        dtype=float,
    )

    residual = np.concatenate(
        [
            market_residual,
            grid_negative_variance_residual,
            analytic_min_residual,
            butterfly_residual,
            param_soft_residual,
        ]
    )

    if not np.isfinite(residual).all():
        residual = np.nan_to_num(residual, nan=1.0e6, posinf=1.0e6, neginf=-1.0e6)

    return residual


# ------------------------------------------------------------
# Fit diagnostics
# ------------------------------------------------------------

def weighted_rmse(residual: np.ndarray, weight: np.ndarray) -> float:
    residual = np.asarray(residual, dtype=float)
    weight = np.asarray(weight, dtype=float)

    valid = np.isfinite(residual) & np.isfinite(weight) & (weight > 0.0)
    if not valid.any():
        return np.nan

    return float(np.sqrt(np.average(residual[valid] ** 2, weights=weight[valid])))


def weighted_mae(residual: np.ndarray, weight: np.ndarray) -> float:
    residual = np.asarray(residual, dtype=float)
    weight = np.asarray(weight, dtype=float)

    valid = np.isfinite(residual) & np.isfinite(weight) & (weight > 0.0)
    if not valid.any():
        return np.nan

    return float(np.average(np.abs(residual[valid]), weights=weight[valid]))


def evaluate_svi_fit(theta: np.ndarray, arrays: SVIFitArrays) -> dict[str, float | int | bool]:
    theta = np.asarray(theta, dtype=float)
    fitted_w = raw_svi_total_variance(arrays.k, theta)

    fitted_iv = np.full_like(fitted_w, np.nan, dtype=float)
    valid_iv = np.isfinite(fitted_w) & np.isfinite(arrays.tau) & (fitted_w > 0.0) & (arrays.tau > 0.0)
    fitted_iv[valid_iv] = np.sqrt(fitted_w[valid_iv] / arrays.tau[valid_iv])

    w_residual = fitted_w - arrays.observed_w
    iv_residual = fitted_iv - arrays.observed_iv

    fitted_price = bsm_forward_price_from_total_variance(
        forward=arrays.forward,
        strike=arrays.strike,
        tau=arrays.tau,
        discount_factor=arrays.discount_factor,
        total_variance=fitted_w,
        option_type=arrays.option_type,
    )

    price_residual = fitted_price - arrays.market_price

    grid_diag = svi_slice_grid_diagnostics(arrays.k_grid, theta)
    grid_summary = summarize_svi_grid_diagnostics(grid_diag)

    constraints = raw_svi_parameter_constraints(theta)

    return {
        "rows": int(len(arrays.k)),
        "unique_contracts": int(len(np.unique(arrays.contract_key))),
        "scale_w": float(arrays.scale_w),
        "weighted_rmse_w": weighted_rmse(w_residual, arrays.weight),
        "weighted_mae_w": weighted_mae(w_residual, arrays.weight),
        "weighted_rmse_iv": weighted_rmse(iv_residual, arrays.weight),
        "weighted_mae_iv": weighted_mae(iv_residual, arrays.weight),
        "weighted_rmse_price": weighted_rmse(price_residual, arrays.weight),
        "weighted_mae_price": weighted_mae(price_residual, arrays.weight),
        "max_abs_w_residual": float(np.nanmax(np.abs(w_residual))) if len(w_residual) else np.nan,
        "max_abs_iv_residual": float(np.nanmax(np.abs(iv_residual))) if len(iv_residual) else np.nan,
        "max_abs_price_residual": float(np.nanmax(np.abs(price_residual))) if len(price_residual) else np.nan,
        "analytic_k_min": float(constraints["k_min"]),
        "analytic_w_min": float(constraints["w_min"]),
        "basic_feasible": bool(constraints["basic_feasible"]),
        "positive_total_variance_grid_pass": bool(grid_summary["positive_total_variance_pass"]),
        "butterfly_grid_pass": bool(grid_summary["butterfly_grid_pass"]),
        "grid_points": int(grid_summary["grid_points"]),
        "g_violation_points": int(grid_summary["g_violation_points"]),
        "min_grid_total_variance": float(grid_summary["min_total_variance"]),
        "min_grid_g": float(grid_summary["min_g"]),
        "max_grid_g_violation_amount": float(grid_summary["max_g_violation_amount"]),
    }


# ------------------------------------------------------------
# Single-seed optimizer
# ------------------------------------------------------------

def fit_svi_one_seed(
    arrays: SVIFitArrays,
    seed_row: pd.Series,
) -> dict[str, object]:
    x0 = np.array(
        [
            seed_row["a"],
            seed_row["b"],
            seed_row["rho"],
            seed_row["m"],
            seed_row["sigma"],
        ],
        dtype=float,
    )

    start = time.perf_counter()

    base_record = {
        "mode_name": arrays.mode_name,
        "expiry": arrays.expiry,
        "seed_rank": int(seed_row["seed_rank"]),
        "is_base_seed": bool(seed_row["is_base_seed"]),
        "seed_a": float(seed_row["a"]),
        "seed_b": float(seed_row["b"]),
        "seed_rho": float(seed_row["rho"]),
        "seed_m": float(seed_row["m"]),
        "seed_sigma": float(seed_row["sigma"]),
        "rows": int(len(arrays.k)),
        "unique_contracts": int(len(np.unique(arrays.contract_key))),
        "objective_label": get_objective_config(arrays.mode_name).objective_label,
    }

    try:
        result = least_squares(
            fun=lambda theta: svi_objective_residuals(theta, arrays),
            x0=x0,
            bounds=(SVI_BOUNDS_LOWER, SVI_BOUNDS_UPPER),
            loss=getattr(CONFIG, "loss_function", "soft_l1"),
            f_scale=max(float(getattr(CONFIG, "soft_l1_f_scale", 0.0100)), 0.0100),
            max_nfev=int(getattr(CONFIG, "max_nfev", 20_000)),
            ftol=1.0e-10,
            xtol=1.0e-10,
            gtol=1.0e-10,
            method="trf",
        )

        elapsed = time.perf_counter() - start
        theta_hat = np.asarray(result.x, dtype=float)
        metrics = evaluate_svi_fit(theta_hat, arrays)

        usable_fit = bool(
            np.isfinite(theta_hat).all()
            and metrics["basic_feasible"]
            and metrics["positive_total_variance_grid_pass"]
            and np.isfinite(result.cost)
        )

        record = {
            **base_record,
            "success": bool(result.success),
            "status": int(result.status),
            "message": str(result.message),
            "nfev": int(result.nfev),
            "njev": int(result.njev) if result.njev is not None else np.nan,
            "cost": float(result.cost),
            "optimality": float(result.optimality) if np.isfinite(result.optimality) else np.nan,
            "elapsed_seconds": float(elapsed),
            "fit_a": float(theta_hat[0]),
            "fit_b": float(theta_hat[1]),
            "fit_rho": float(theta_hat[2]),
            "fit_m": float(theta_hat[3]),
            "fit_sigma": float(theta_hat[4]),
            "usable_fit": usable_fit,
            **metrics,
        }

        # Selection score keeps usable fits first; the objective already includes
        # positive-variance and butterfly soft penalties.
        record["selection_score"] = (
            float(result.cost)
            + (0.0 if usable_fit else 1.0e9)
        )

        return record

    except Exception as exc:
        elapsed = time.perf_counter() - start

        return {
            **base_record,
            "success": False,
            "status": -999,
            "message": repr(exc),
            "nfev": 0,
            "njev": np.nan,
            "cost": np.inf,
            "optimality": np.nan,
            "elapsed_seconds": float(elapsed),
            "fit_a": np.nan,
            "fit_b": np.nan,
            "fit_rho": np.nan,
            "fit_m": np.nan,
            "fit_sigma": np.nan,
            "usable_fit": False,
            "rows": int(len(arrays.k)),
            "unique_contracts": int(len(np.unique(arrays.contract_key))),
            "scale_w": arrays.scale_w,
            "weighted_rmse_w": np.nan,
            "weighted_mae_w": np.nan,
            "weighted_rmse_iv": np.nan,
            "weighted_mae_iv": np.nan,
            "weighted_rmse_price": np.nan,
            "weighted_mae_price": np.nan,
            "max_abs_w_residual": np.nan,
            "max_abs_iv_residual": np.nan,
            "max_abs_price_residual": np.nan,
            "analytic_k_min": np.nan,
            "analytic_w_min": np.nan,
            "basic_feasible": False,
            "positive_total_variance_grid_pass": False,
            "butterfly_grid_pass": False,
            "grid_points": np.nan,
            "g_violation_points": np.nan,
            "min_grid_total_variance": np.nan,
            "min_grid_g": np.nan,
            "max_grid_g_violation_amount": np.nan,
            "selection_score": np.inf,
        }


# ------------------------------------------------------------
# Run multistart fits
# ------------------------------------------------------------

fit_records: list[dict[str, object]] = []

eligible_seed_groups = (
    N11_SVI_SEED_TABLE
    .groupby(["mode_name", "expiry"], dropna=False)
    .size()
    .reset_index(name="seed_count")
    .sort_values(["mode_name", "expiry"])
)

total_groups = len(eligible_seed_groups)
total_seeds = len(N11_SVI_SEED_TABLE)

print("Starting multistart SVI fitting.")
print(f"Groups: {total_groups:,}")
print(f"Seeds: {total_seeds:,}")

fit_start_all = time.perf_counter()

for group_index, group_row in eligible_seed_groups.iterrows():
    mode_name = group_row["mode_name"]
    expiry = group_row["expiry"]

    arrays = prepare_svi_fit_arrays(mode_name, expiry)

    seeds = (
        N11_SVI_SEED_TABLE
        .loc[
            (N11_SVI_SEED_TABLE["mode_name"] == mode_name)
            & (N11_SVI_SEED_TABLE["expiry"] == expiry)
        ]
        .sort_values("seed_rank")
        .reset_index(drop=True)
    )

    print(
        f"[{group_index + 1:>2}/{total_groups}] "
        f"mode={mode_name:<18} expiry={pd.Timestamp(expiry).date()} "
        f"rows={len(arrays.k):>4,} seeds={len(seeds):>3}"
    )

    for _, seed_row in seeds.iterrows():
        fit_records.append(fit_svi_one_seed(arrays, seed_row))

fit_elapsed_all = time.perf_counter() - fit_start_all

n11_svi_fit_results = pd.DataFrame(fit_records)

# Defensive sorting even if failures exist.
n11_svi_fit_results = (
    n11_svi_fit_results
    .sort_values(["mode_name", "expiry", "selection_score", "seed_rank"], na_position="last")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Best fit per mode/expiry
# ------------------------------------------------------------

n11_best_svi_fits = (
    n11_svi_fit_results
    .sort_values(["mode_name", "expiry", "selection_score", "seed_rank"], na_position="last")
    .groupby(["mode_name", "expiry"], dropna=False)
    .head(1)
    .reset_index(drop=True)
)

n11_best_svi_fits["fit_rank_within_group"] = 1

# Compact parameter table.
n11_best_svi_parameters = n11_best_svi_fits[
    [
        "mode_name",
        "expiry",
        "fit_rank_within_group",
        "fit_a",
        "fit_b",
        "fit_rho",
        "fit_m",
        "fit_sigma",
        "analytic_k_min",
        "analytic_w_min",
        "rows",
        "unique_contracts",
        "success",
        "usable_fit",
        "cost",
        "weighted_rmse_w",
        "weighted_rmse_iv",
        "weighted_rmse_price",
        "positive_total_variance_grid_pass",
        "butterfly_grid_pass",
        "g_violation_points",
        "min_grid_g",
        "max_grid_g_violation_amount",
    ]
].copy()


# ------------------------------------------------------------
# Fitted contract-level points and grid diagnostics for best fits
# ------------------------------------------------------------

best_contract_frames = []
best_grid_frames = []

for _, row in n11_best_svi_fits.iterrows():
    mode_name = row["mode_name"]
    expiry = row["expiry"]
    theta = np.array(
        [
            row["fit_a"],
            row["fit_b"],
            row["fit_rho"],
            row["fit_m"],
            row["fit_sigma"],
        ],
        dtype=float,
    )

    arrays = prepare_svi_fit_arrays(mode_name, expiry)

    fitted_w = raw_svi_total_variance(arrays.k, theta)
    fitted_iv = np.full_like(fitted_w, np.nan, dtype=float)

    valid_iv = np.isfinite(fitted_w) & np.isfinite(arrays.tau) & (fitted_w > 0.0) & (arrays.tau > 0.0)
    fitted_iv[valid_iv] = np.sqrt(fitted_w[valid_iv] / arrays.tau[valid_iv])

    fitted_price = bsm_forward_price_from_total_variance(
        forward=arrays.forward,
        strike=arrays.strike,
        tau=arrays.tau,
        discount_factor=arrays.discount_factor,
        total_variance=fitted_w,
        option_type=arrays.option_type,
    )

    contract_df = pd.DataFrame(
        {
            "mode_name": mode_name,
            "expiry": expiry,
            "contract_key": arrays.contract_key,
            "k": arrays.k,
            "tau_years": arrays.tau,
            "strike": arrays.strike,
            "forward": arrays.forward,
            "discount_factor": arrays.discount_factor,
            "option_type": arrays.option_type,
            "observed_total_variance": arrays.observed_w,
            "fitted_total_variance": fitted_w,
            "total_variance_residual": fitted_w - arrays.observed_w,
            "observed_iv": arrays.observed_iv,
            "fitted_iv": fitted_iv,
            "iv_residual": fitted_iv - arrays.observed_iv,
            "market_price": arrays.market_price,
            "fitted_bsm_price": fitted_price,
            "price_residual": fitted_price - arrays.market_price,
            "fit_weight": arrays.weight,
            "normalized_weight": arrays.normalized_weight,
            "fit_a": row["fit_a"],
            "fit_b": row["fit_b"],
            "fit_rho": row["fit_rho"],
            "fit_m": row["fit_m"],
            "fit_sigma": row["fit_sigma"],
        }
    )

    best_contract_frames.append(contract_df)

    grid_df = svi_slice_grid_diagnostics(arrays.k_grid, theta)
    grid_df.insert(0, "mode_name", mode_name)
    grid_df.insert(1, "expiry", expiry)
    grid_df["fit_a"] = row["fit_a"]
    grid_df["fit_b"] = row["fit_b"]
    grid_df["fit_rho"] = row["fit_rho"]
    grid_df["fit_m"] = row["fit_m"]
    grid_df["fit_sigma"] = row["fit_sigma"]

    best_grid_frames.append(grid_df)

n11_best_svi_contract_points = (
    pd.concat(best_contract_frames, ignore_index=True)
    if best_contract_frames
    else pd.DataFrame()
)

n11_best_svi_grid_diagnostics = (
    pd.concat(best_grid_frames, ignore_index=True)
    if best_grid_frames
    else pd.DataFrame()
)


# ------------------------------------------------------------
# Fit summary and gate
# ------------------------------------------------------------

n11_svi_fit_summary_by_mode = (
    n11_svi_fit_results
    .groupby("mode_name", dropna=False)
    .agg(
        attempted_fits=("seed_rank", "count"),
        successful_optimizer_fits=("success", "sum"),
        usable_fits=("usable_fit", "sum"),
        best_cost=("cost", "min"),
        median_cost=("cost", "median"),
        median_elapsed_seconds=("elapsed_seconds", "median"),
        total_elapsed_seconds=("elapsed_seconds", "sum"),
        median_rmse_iv=("weighted_rmse_iv", "median"),
        median_rmse_price=("weighted_rmse_price", "median"),
        best_fit_expiries=("expiry", "nunique"),
    )
    .reset_index()
)

n11_svi_best_summary_by_mode = (
    n11_best_svi_fits
    .groupby("mode_name", dropna=False)
    .agg(
        selected_expiries=("expiry", "nunique"),
        selected_usable_fits=("usable_fit", "sum"),
        selected_successful_optimizer_fits=("success", "sum"),
        positive_variance_grid_passes=("positive_total_variance_grid_pass", "sum"),
        butterfly_grid_passes=("butterfly_grid_pass", "sum"),
        total_g_violation_points=("g_violation_points", "sum"),
        median_best_rmse_w=("weighted_rmse_w", "median"),
        median_best_rmse_iv=("weighted_rmse_iv", "median"),
        median_best_rmse_price=("weighted_rmse_price", "median"),
        max_best_abs_iv_residual=("max_abs_iv_residual", "max"),
        max_best_g_violation_amount=("max_grid_g_violation_amount", "max"),
    )
    .reset_index()
)

expected_fit_groups = len(eligible_seed_groups)
actual_best_groups = len(n11_best_svi_fits)

weighted_broad_best_count = int(
    n11_best_svi_fits.query("mode_name == 'weighted_broad_svi'")["expiry"].nunique()
)

weighted_broad_positive_variance_pass_count = int(
    n11_best_svi_fits.query("mode_name == 'weighted_broad_svi'")["positive_total_variance_grid_pass"].sum()
)

weighted_broad_usable_count = int(
    n11_best_svi_fits.query("mode_name == 'weighted_broad_svi'")["usable_fit"].sum()
)

any_mode_full_butterfly_pass = bool(
    (
        n11_svi_best_summary_by_mode["butterfly_grid_passes"]
        == n11_svi_best_summary_by_mode["selected_expiries"]
    ).any()
)

n11_svi_fit_gate = pd.DataFrame(
    [
        {
            "check_name": "all_seed_fits_attempted",
            "check_type": "blocking",
            "passes": len(n11_svi_fit_results) == len(N11_SVI_SEED_TABLE),
            "observed_value": float(len(n11_svi_fit_results)),
            "threshold_or_requirement": f"== {len(N11_SVI_SEED_TABLE)}",
        },
        {
            "check_name": "best_fit_per_group_available",
            "check_type": "blocking",
            "passes": actual_best_groups == expected_fit_groups,
            "observed_value": float(actual_best_groups),
            "threshold_or_requirement": f"== {expected_fit_groups}",
        },
        {
            "check_name": "weighted_broad_best_expiry_count",
            "check_type": "blocking",
            "passes": weighted_broad_best_count >= CONFIG.min_primary_expiries,
            "observed_value": float(weighted_broad_best_count),
            "threshold_or_requirement": f">= {CONFIG.min_primary_expiries}",
        },
        {
            "check_name": "weighted_broad_usable_fit_count",
            "check_type": "blocking",
            "passes": weighted_broad_usable_count >= CONFIG.min_primary_expiries,
            "observed_value": float(weighted_broad_usable_count),
            "threshold_or_requirement": f">= {CONFIG.min_primary_expiries}",
        },
        {
            "check_name": "weighted_broad_positive_variance_grid_pass_count",
            "check_type": "blocking",
            "passes": weighted_broad_positive_variance_pass_count >= CONFIG.min_primary_expiries,
            "observed_value": float(weighted_broad_positive_variance_pass_count),
            "threshold_or_requirement": f">= {CONFIG.min_primary_expiries}",
        },
        {
            "check_name": "best_parameter_table_nonempty",
            "check_type": "blocking",
            "passes": len(n11_best_svi_parameters) > 0,
            "observed_value": float(len(n11_best_svi_parameters)),
            "threshold_or_requirement": "> 0",
        },
        {
            "check_name": "best_contract_points_available",
            "check_type": "blocking",
            "passes": len(n11_best_svi_contract_points) > 0,
            "observed_value": float(len(n11_best_svi_contract_points)),
            "threshold_or_requirement": "> 0",
        },
        {
            "check_name": "best_grid_diagnostics_available",
            "check_type": "blocking",
            "passes": len(n11_best_svi_grid_diagnostics) > 0,
            "observed_value": float(len(n11_best_svi_grid_diagnostics)),
            "threshold_or_requirement": "> 0",
        },
        {
            "check_name": "any_mode_full_butterfly_grid_pass",
            "check_type": "warning",
            "passes": any_mode_full_butterfly_pass,
            "observed_value": float(any_mode_full_butterfly_pass),
            "threshold_or_requirement": "preferred True; repair layer can address False",
        },
    ]
)

N11_SVI_FIT_PASS = bool(
    n11_svi_fit_gate
    .query("check_type == 'blocking'")["passes"]
    .all()
)


# ------------------------------------------------------------
# Persist outputs
# ------------------------------------------------------------

_write_table_pair(
    n11_svi_objective_config_summary,
    "n11_svi_objective_config_summary",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_svi_fit_results,
    "n11_svi_fit_results",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_best_svi_fits,
    "n11_best_svi_fits",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_best_svi_parameters,
    "n11_best_svi_parameters",
    N11_PARAM_DIR,
)

_write_table_pair(
    n11_best_svi_contract_points,
    "n11_best_svi_contract_points",
    N11_HANDOFF_DIR,
)

_write_table_pair(
    n11_best_svi_grid_diagnostics,
    "n11_best_svi_grid_diagnostics",
    N11_DIAGNOSTIC_DIR,
)

_write_table_pair(
    n11_svi_fit_summary_by_mode,
    "n11_svi_fit_summary_by_mode",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_svi_best_summary_by_mode,
    "n11_svi_best_summary_by_mode",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_svi_fit_gate,
    "n11_svi_fit_gate",
    N11_TABLE_DIR,
)


# ------------------------------------------------------------
# Console summary
# ------------------------------------------------------------

print("Notebook 11 multistart SVI fitting complete.")
print(f"SVI fit pass: {N11_SVI_FIT_PASS}")
print(f"Attempted fits: {len(n11_svi_fit_results):,}")
print(f"Best fit groups: {len(n11_best_svi_fits):,}")
print(f"Total fitting elapsed seconds: {fit_elapsed_all:,.2f}")

display(n11_svi_fit_summary_by_mode)
display(n11_svi_best_summary_by_mode)
display(n11_best_svi_parameters)
display(n11_svi_fit_gate)

if not N11_SVI_FIT_PASS:
    raise RuntimeError(
        "Notebook 11 SVI fitting gate failed. "
        "Review n11_svi_fit_gate and n11_best_svi_parameters before proceeding."
    )

Starting multistart SVI fitting.
Groups: 20
Seeds: 1,280
[ 1/20] mode=repair_aware_svi   expiry=2026-07-10 rows=  97 seeds= 64
[ 2/20] mode=repair_aware_svi   expiry=2026-07-17 rows= 130 seeds= 64
[ 3/20] mode=repair_aware_svi   expiry=2026-07-24 rows= 152 seeds= 64
[ 4/20] mode=repair_aware_svi   expiry=2026-07-31 rows= 175 seeds= 64
[ 5/20] mode=repair_aware_svi   expiry=2026-08-07 rows= 131 seeds= 64
[ 6/20] mode=repair_aware_svi   expiry=2026-08-21 rows= 173 seeds= 64
[ 7/20] mode=repair_aware_svi   expiry=2026-08-31 rows= 187 seeds= 64
[ 8/20] mode=repair_aware_svi   expiry=2026-09-18 rows= 215 seeds= 64
[ 9/20] mode=strict_svi         expiry=2026-07-24 rows=  29 seeds= 64
[10/20] mode=strict_svi         expiry=2026-07-31 rows=  45 seeds= 64
[11/20] mode=strict_svi         expiry=2026-08-21 rows=  33 seeds= 64
[12/20] mode=strict_svi         expiry=2026-08-31 rows=  33 seeds= 64
[13/20] mode=weighted_broad_svi expiry=2026-07-10 rows=  97 seeds= 64
[14/20] mode=weighted_broad_svi e

,mode_name,attempted_fits,successful_optimizer_fits,usable_fits,best_cost,median_cost,median_elapsed_seconds,total_elapsed_seconds,median_rmse_iv,median_rmse_price,best_fit_expiries
0,repair_aware_svi,512,512,448,0.0015719456,0.0056241842,1.8223911000,988.5281617000,0.0018576256,0.0620438938,8
1,strict_svi,256,256,192,0.0000659533,0.0001159406,3.9439139000,"1,058.1855895000",0.0002039166,0.0169861773,4
2,weighted_broad_svi,512,512,448,0.0017333675,0.0058409269,2.0518623000,945.1234803000,0.0019403544,0.0555898741,8


,mode_name,selected_expiries,selected_usable_fits,selected_successful_optimizer_fits,positive_variance_grid_passes,butterfly_grid_passes,total_g_violation_points,median_best_rmse_w,median_best_rmse_iv,median_best_rmse_price,max_best_abs_iv_residual,max_best_g_violation_amount
0,repair_aware_svi,8,7,8,8,8,0,0.0000286313,0.0018575933,0.0546484897,0.0384506639,0.0000000000
1,strict_svi,4,3,4,4,4,0,0.0000065697,0.0002038855,0.0166116598,0.0008776236,0.0000000000
2,weighted_broad_svi,8,7,8,8,8,0,0.0000299764,0.0019403535,0.0555896811,0.0386599694,0.0000000000


,mode_name,expiry,fit_rank_within_group,fit_a,fit_b,fit_rho,fit_m,fit_sigma,analytic_k_min,analytic_w_min,rows,unique_contracts,success,usable_fit,cost,weighted_rmse_w,weighted_rmse_iv,weighted_rmse_price,positive_total_variance_grid_pass,butterfly_grid_pass,g_violation_points,min_grid_g,max_grid_g_violation_amount
0,repair_aware_svi,2026-07-10 00:00:00+00:00,1,-0.0137280593,0.3251875794,-0.9500519497,-0.3505727952,0.1361026680,0.0637431346,0.0000847796,97,97,True,True,0.0057884914,0.0000069641,0.0021055710,0.0271761159,True,True,0,0.0816478622,0.0000000000
1,repair_aware_svi,2026-07-17 00:00:00+00:00,1,-0.0023267408,0.0330002064,-0.8767642397,-0.1040652274,0.1466078916,0.1632150676,-0.0000000042,130,130,True,False,0.0058502233,0.0000168706,0.0021767419,0.0435958167,True,True,0,0.0250911365,0.0000000000
2,repair_aware_svi,2026-07-24 00:00:00+00:00,1,-0.0049286098,0.0426051630,-0.7936173177,-0.1094447422,0.1966843639,0.1471096481,0.0001697865,152,152,True,True,0.0080883592,0.0000269633,0.0023156507,0.0672178479,True,True,0,0.0404922521,0.0000000000
3,repair_aware_svi,2026-07-31 00:00:00+00:00,1,-0.1196754062,0.1524856341,-0.1413599508,-0.0277397022,0.7976345916,0.0861576111,0.0007310558,175,175,True,True,0.0054598769,0.0000302992,0.0016158614,0.0630178899,True,True,0,0.0406955305,0.0000000000
4,repair_aware_svi,2026-08-07 00:00:00+00:00,1,-0.9999999999,0.7786228050,0.6234580909,1.3999402725,1.6442447456,0.0888079408,0.0009693590,131,131,True,True,0.0076716657,0.0000458975,0.0020993253,0.0694071336,True,True,0,0.0723498377,0.0000000000
5,repair_aware_svi,2026-08-21 00:00:00+00:00,1,-0.3019621739,2.0043662396,0.9183374065,0.9693722770,0.3827244116,0.0813694326,0.0016626843,173,173,True,True,0.0033438392,0.0000366388,0.0009961876,0.0472530179,True,True,0,0.0798722033,0.0000000000
6,repair_aware_svi,2026-08-31 00:00:00+00:00,1,-0.1966173480,2.0037327985,0.9278768703,0.7415665018,0.2660155329,0.0796237722,0.0021402465,187,187,True,True,0.0015719456,0.0000261118,0.0004970035,0.0366726822,True,True,0,0.1127155151,0.0000000000
7,repair_aware_svi,2026-09-18 00:00:00+00:00,1,-0.1883277058,2.0005762056,0.9231144862,0.6844484870,0.2487952455,0.0871755833,0.0030635120,215,215,True,True,0.0018163553,0.0000405568,0.0005660184,0.0620439616,True,True,0,0.1250041994,0.0000000000
8,strict_svi,2026-07-24 00:00:00+00:00,1,-0.0136534410,0.6970420525,0.9500004119,0.2610527704,0.0647129954,0.0641668176,0.0004313708,29,29,True,True,0.0000659533,0.0000022864,0.0001562562,0.0090144721,True,True,0,0.1950406061,0.0000000000
9,strict_svi,2026-07-31 00:00:00+00:00,1,-0.0376785224,0.5004347376,-0.9500241248,-0.5471001669,0.2411826748,0.1868736117,-0.0000000097,45,45,True,False,0.0001651678,0.0000056530,0.0002559387,0.0162464802,True,True,0,0.1274653830,0.0000000000


,check_name,check_type,passes,observed_value,threshold_or_requirement
0,all_seed_fits_attempted,blocking,True,"1,280.0000000000",== 1280
1,best_fit_per_group_available,blocking,True,20.0000000000,== 20
2,weighted_broad_best_expiry_count,blocking,True,8.0000000000,>= 4
3,weighted_broad_usable_fit_count,blocking,True,7.0000000000,>= 4
4,weighted_broad_positive_variance_grid_pass_count,blocking,True,8.0000000000,>= 4
5,best_parameter_table_nonempty,blocking,True,20.0000000000,> 0
6,best_contract_points_available,blocking,True,"2,660.0000000000",> 0
7,best_grid_diagnostics_available,blocking,True,"3,620.0000000000",> 0
8,any_mode_full_butterfly_grid_pass,warning,True,1.0000000000,preferred True; repair layer can address False


In [7]:
# ------------------------------------------------------------
# Notebook 11 cell 07: Pre-repair surface audit, calendar checks, and mode ranking
# ------------------------------------------------------------
# This cell audits fitted SVI slices before selecting or repairing a surface.
#
# It checks:
#
#   1. parameter-boundary / degeneracy risk
#   2. contract-level fit residuals
#   3. butterfly pass already achieved by fitted slices
#   4. calendar monotonicity across fitted expiries on common k-grids
#   5. candidate-mode ranking before repair
#
# It does NOT repair the surface yet. It decides which fitted candidate should
# be handed into the repair layer.

from __future__ import annotations

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Pre-flight checks
# ------------------------------------------------------------

required_globals_for_audit = [
    "N11_SVI_FIT_PASS",
    "n11_best_svi_parameters",
    "n11_best_svi_contract_points",
    "n11_best_svi_grid_diagnostics",
    "n11_best_svi_fits",
    "N11_COMMON_K_GRID",
    "N11_COMMON_K_GRID_INNER",
    "raw_svi_total_variance",
    "svi_butterfly_g",
    "svi_slice_grid_diagnostics",
    "summarize_svi_grid_diagnostics",
    "CONFIG",
]

missing_audit_globals = [name for name in required_globals_for_audit if name not in globals()]
if missing_audit_globals:
    raise NameError(f"Missing required objects before pre-repair audit: {missing_audit_globals}")

if not bool(N11_SVI_FIT_PASS):
    raise RuntimeError("SVI fitting did not pass. Run/fix the previous cell first.")


# ------------------------------------------------------------
# Parameter-boundary and degeneracy diagnostics
# ------------------------------------------------------------

def relative_boundary_distance(value: float, lower: float, upper: float) -> float:
    value = float(value)
    lower = float(lower)
    upper = float(upper)

    width = max(upper - lower, N11_TINY)
    return float(min(value - lower, upper - value) / width)


def parameter_boundary_diagnostics(params: pd.DataFrame) -> pd.DataFrame:
    out = params.copy()

    out["a_boundary_distance"] = out["fit_a"].apply(
        lambda x: relative_boundary_distance(x, CONFIG.svi_a_lower, CONFIG.svi_a_upper)
    )
    out["b_boundary_distance"] = out["fit_b"].apply(
        lambda x: relative_boundary_distance(x, CONFIG.svi_b_lower, CONFIG.svi_b_upper)
    )
    out["rho_boundary_distance"] = out["fit_rho"].apply(
        lambda x: relative_boundary_distance(x, CONFIG.svi_rho_lower, CONFIG.svi_rho_upper)
    )
    out["m_boundary_distance"] = out["fit_m"].apply(
        lambda x: relative_boundary_distance(x, CONFIG.svi_m_lower, CONFIG.svi_m_upper)
    )
    out["sigma_boundary_distance"] = out["fit_sigma"].apply(
        lambda x: relative_boundary_distance(x, CONFIG.svi_sigma_lower, CONFIG.svi_sigma_upper)
    )

    boundary_cols = [
        "a_boundary_distance",
        "b_boundary_distance",
        "rho_boundary_distance",
        "m_boundary_distance",
        "sigma_boundary_distance",
    ]

    out["min_parameter_boundary_distance"] = out[boundary_cols].min(axis=1)

    # Hard boundary means the optimizer is effectively using a constraint edge.
    out["a_near_boundary"] = out["a_boundary_distance"] <= 0.005
    out["b_near_boundary"] = out["b_boundary_distance"] <= 0.005
    out["rho_near_boundary"] = out["rho_boundary_distance"] <= 0.025
    out["m_near_boundary"] = out["m_boundary_distance"] <= 0.005
    out["sigma_near_boundary"] = out["sigma_boundary_distance"] <= 0.005

    # Shape-degeneracy flags are softer than boundary flags.
    out["rho_extreme_abs_ge_090"] = out["fit_rho"].abs() >= 0.90
    out["rho_extreme_abs_ge_095"] = out["fit_rho"].abs() >= 0.95
    out["a_large_negative"] = out["fit_a"] <= -0.1000
    out["b_large"] = out["fit_b"] >= 1.0000
    out["sigma_large"] = out["fit_sigma"] >= 1.0000

    out["parameter_boundary_flag_count"] = out[
        [
            "a_near_boundary",
            "b_near_boundary",
            "rho_near_boundary",
            "m_near_boundary",
            "sigma_near_boundary",
        ]
    ].sum(axis=1)

    out["parameter_degeneracy_flag_count"] = out[
        [
            "rho_extreme_abs_ge_090",
            "rho_extreme_abs_ge_095",
            "a_large_negative",
            "b_large",
            "sigma_large",
        ]
    ].sum(axis=1)

    out["parameter_risk_score"] = (
        1.00 * out["parameter_boundary_flag_count"]
        + 0.50 * out["parameter_degeneracy_flag_count"]
        + 2.00 * out["rho_extreme_abs_ge_095"].astype(float)
        + 1.50 * out["a_near_boundary"].astype(float)
    )

    out["parameter_risk_label"] = np.select(
        [
            out["parameter_risk_score"] >= 5.0,
            out["parameter_risk_score"] >= 2.5,
            out["parameter_risk_score"] > 0.0,
        ],
        [
            "high",
            "moderate",
            "low",
        ],
        default="none",
    )

    return out


n11_pre_repair_parameter_audit = parameter_boundary_diagnostics(n11_best_svi_parameters)


n11_pre_repair_parameter_summary = (
    n11_pre_repair_parameter_audit
    .groupby("mode_name", dropna=False)
    .agg(
        fitted_expiries=("expiry", "nunique"),
        usable_fits=("usable_fit", "sum"),
        success_fits=("success", "sum"),
        near_boundary_slices=("parameter_boundary_flag_count", lambda s: int((s > 0).sum())),
        degenerate_slices=("parameter_degeneracy_flag_count", lambda s: int((s > 0).sum())),
        high_parameter_risk_slices=("parameter_risk_label", lambda s: int((s == "high").sum())),
        median_parameter_risk_score=("parameter_risk_score", "median"),
        max_parameter_risk_score=("parameter_risk_score", "max"),
        min_analytic_w_min=("analytic_w_min", "min"),
        median_analytic_w_min=("analytic_w_min", "median"),
        median_abs_rho=("fit_rho", lambda s: float(np.nanmedian(np.abs(s)))),
        max_abs_rho=("fit_rho", lambda s: float(np.nanmax(np.abs(s)))),
        median_b=("fit_b", "median"),
        max_b=("fit_b", "max"),
        median_sigma=("fit_sigma", "median"),
        max_sigma=("fit_sigma", "max"),
    )
    .reset_index()
)


# ------------------------------------------------------------
# Contract-level residual diagnostics
# ------------------------------------------------------------

def contract_fit_residual_summary(contract_points: pd.DataFrame) -> pd.DataFrame:
    df = contract_points.copy()

    df["abs_w_residual"] = pd.to_numeric(df["total_variance_residual"], errors="coerce").abs()
    df["abs_iv_residual"] = pd.to_numeric(df["iv_residual"], errors="coerce").abs()
    df["abs_price_residual"] = pd.to_numeric(df["price_residual"], errors="coerce").abs()

    # ATM-ish subset for a more relevant center-of-smile fit diagnostic.
    df["is_atm_band"] = pd.to_numeric(df["k"], errors="coerce").abs() <= 0.0500
    df["is_inner_band"] = pd.to_numeric(df["k"], errors="coerce").abs() <= 0.1500

    rows = []

    for mode_name, mode_df in df.groupby("mode_name", dropna=False):
        weight = pd.to_numeric(mode_df["fit_weight"], errors="coerce").fillna(0.0).to_numpy(dtype=float)

        for band_name, band_mask in {
            "all": pd.Series(True, index=mode_df.index),
            "inner_abs_k_le_015": mode_df["is_inner_band"],
            "atm_abs_k_le_005": mode_df["is_atm_band"],
        }.items():
            sub = mode_df.loc[band_mask].copy()

            if len(sub) == 0:
                rows.append(
                    {
                        "mode_name": mode_name,
                        "band": band_name,
                        "rows": 0,
                        "unique_expiries": 0,
                        "weighted_rmse_iv": np.nan,
                        "weighted_mae_iv": np.nan,
                        "median_abs_iv_residual": np.nan,
                        "p95_abs_iv_residual": np.nan,
                        "max_abs_iv_residual": np.nan,
                        "weighted_rmse_price": np.nan,
                        "weighted_mae_price": np.nan,
                        "median_abs_price_residual": np.nan,
                        "p95_abs_price_residual": np.nan,
                        "max_abs_price_residual": np.nan,
                    }
                )
                continue

            w_sub = pd.to_numeric(sub["fit_weight"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
            iv_res = pd.to_numeric(sub["iv_residual"], errors="coerce").to_numpy(dtype=float)
            px_res = pd.to_numeric(sub["price_residual"], errors="coerce").to_numpy(dtype=float)
            abs_iv = np.abs(iv_res)
            abs_px = np.abs(px_res)

            rows.append(
                {
                    "mode_name": mode_name,
                    "band": band_name,
                    "rows": int(len(sub)),
                    "unique_expiries": int(sub["expiry"].nunique()),
                    "weighted_rmse_iv": weighted_rmse(iv_res, w_sub),
                    "weighted_mae_iv": weighted_mae(iv_res, w_sub),
                    "median_abs_iv_residual": float(np.nanmedian(abs_iv)),
                    "p95_abs_iv_residual": float(np.nanpercentile(abs_iv, 95)),
                    "max_abs_iv_residual": float(np.nanmax(abs_iv)),
                    "weighted_rmse_price": weighted_rmse(px_res, w_sub),
                    "weighted_mae_price": weighted_mae(px_res, w_sub),
                    "median_abs_price_residual": float(np.nanmedian(abs_px)),
                    "p95_abs_price_residual": float(np.nanpercentile(abs_px, 95)),
                    "max_abs_price_residual": float(np.nanmax(abs_px)),
                }
            )

    return pd.DataFrame(rows)


n11_pre_repair_contract_residual_summary = contract_fit_residual_summary(n11_best_svi_contract_points)


# ------------------------------------------------------------
# Build common-grid fitted surfaces by mode
# ------------------------------------------------------------

def build_mode_surface_grid(
    best_params: pd.DataFrame,
    k_grid: np.ndarray,
    *,
    grid_name: str,
) -> pd.DataFrame:
    rows = []

    k_grid = np.asarray(k_grid, dtype=float)

    for _, row in best_params.iterrows():
        theta = np.array(
            [
                row["fit_a"],
                row["fit_b"],
                row["fit_rho"],
                row["fit_m"],
                row["fit_sigma"],
            ],
            dtype=float,
        )

        w_grid = raw_svi_total_variance(k_grid, theta)
        g_grid = svi_butterfly_g(k_grid, theta)

        tau = np.nan
        if "expiry" in row.index:
            matching_tau = n11_best_svi_contract_points.loc[
                (n11_best_svi_contract_points["mode_name"] == row["mode_name"])
                & (n11_best_svi_contract_points["expiry"] == row["expiry"]),
                "tau_years",
            ]
            if len(matching_tau) > 0:
                tau = float(pd.to_numeric(matching_tau, errors="coerce").dropna().median())

        iv_grid = np.full_like(w_grid, np.nan, dtype=float)
        valid_iv = np.isfinite(w_grid) & np.isfinite(tau) & (w_grid > 0.0) & (tau > 0.0)
        iv_grid[valid_iv] = np.sqrt(w_grid[valid_iv] / tau)

        for k_value, w_value, iv_value, g_value in zip(k_grid, w_grid, iv_grid, g_grid):
            rows.append(
                {
                    "grid_name": grid_name,
                    "mode_name": row["mode_name"],
                    "expiry": row["expiry"],
                    "tau_years": tau,
                    "k": float(k_value),
                    "svi_total_variance": float(w_value),
                    "svi_iv": float(iv_value) if np.isfinite(iv_value) else np.nan,
                    "svi_g": float(g_value),
                    "fit_a": float(row["fit_a"]),
                    "fit_b": float(row["fit_b"]),
                    "fit_rho": float(row["fit_rho"]),
                    "fit_m": float(row["fit_m"]),
                    "fit_sigma": float(row["fit_sigma"]),
                }
            )

    return pd.DataFrame(rows)


n11_pre_repair_surface_grid_full = build_mode_surface_grid(
    n11_best_svi_parameters,
    N11_COMMON_K_GRID,
    grid_name="common_full_support",
)

n11_pre_repair_surface_grid_inner = build_mode_surface_grid(
    n11_best_svi_parameters,
    N11_COMMON_K_GRID_INNER,
    grid_name="common_inner_95pct_support",
)

n11_pre_repair_surface_grid = pd.concat(
    [
        n11_pre_repair_surface_grid_full,
        n11_pre_repair_surface_grid_inner,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# Calendar diagnostics across fitted expiries
# ------------------------------------------------------------

def calendar_diagnostics_for_grid(surface_grid: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    detailed_rows = []
    summary_rows = []

    for (grid_name, mode_name), mode_df in surface_grid.groupby(["grid_name", "mode_name"], dropna=False):
        mode_df = mode_df.copy()
        mode_df["tau_years"] = pd.to_numeric(mode_df["tau_years"], errors="coerce")
        mode_df["svi_total_variance"] = pd.to_numeric(mode_df["svi_total_variance"], errors="coerce")
        mode_df["k"] = pd.to_numeric(mode_df["k"], errors="coerce")

        expiries = (
            mode_df[["expiry", "tau_years"]]
            .drop_duplicates()
            .sort_values("tau_years")
            .reset_index(drop=True)
        )

        if len(expiries) < 2:
            summary_rows.append(
                {
                    "grid_name": grid_name,
                    "mode_name": mode_name,
                    "expiry_pairs": 0,
                    "k_points": int(mode_df["k"].nunique()),
                    "calendar_checks": 0,
                    "calendar_violation_count": 0,
                    "calendar_violation_rate": np.nan,
                    "max_calendar_violation_amount": 0.0,
                    "min_calendar_diff": np.nan,
                    "calendar_pass": False,
                }
            )
            continue

        for i in range(len(expiries) - 1):
            e1 = expiries.loc[i, "expiry"]
            e2 = expiries.loc[i + 1, "expiry"]
            t1 = float(expiries.loc[i, "tau_years"])
            t2 = float(expiries.loc[i + 1, "tau_years"])

            slice1 = (
                mode_df.loc[mode_df["expiry"] == e1, ["k", "svi_total_variance"]]
                .rename(columns={"svi_total_variance": "w_near"})
            )
            slice2 = (
                mode_df.loc[mode_df["expiry"] == e2, ["k", "svi_total_variance"]]
                .rename(columns={"svi_total_variance": "w_far"})
            )

            pair = slice1.merge(slice2, on="k", how="inner")
            pair["grid_name"] = grid_name
            pair["mode_name"] = mode_name
            pair["near_expiry"] = e1
            pair["far_expiry"] = e2
            pair["near_tau"] = t1
            pair["far_tau"] = t2
            pair["calendar_diff"] = pair["w_far"] - pair["w_near"]
            pair["calendar_violation"] = pair["calendar_diff"] < CONFIG.calendar_tolerance
            pair["calendar_violation_amount"] = np.maximum(
                0.0,
                CONFIG.calendar_tolerance - pair["calendar_diff"],
            )

            detailed_rows.append(pair)

        if detailed_rows:
            current_detail = pd.concat(detailed_rows, ignore_index=True)
            current_detail = current_detail.loc[
                (current_detail["grid_name"] == grid_name)
                & (current_detail["mode_name"] == mode_name)
            ].copy()
        else:
            current_detail = pd.DataFrame()

        if len(current_detail) == 0:
            summary_rows.append(
                {
                    "grid_name": grid_name,
                    "mode_name": mode_name,
                    "expiry_pairs": int(len(expiries) - 1),
                    "k_points": int(mode_df["k"].nunique()),
                    "calendar_checks": 0,
                    "calendar_violation_count": 0,
                    "calendar_violation_rate": np.nan,
                    "max_calendar_violation_amount": 0.0,
                    "min_calendar_diff": np.nan,
                    "calendar_pass": False,
                }
            )
        else:
            violation_count = int(current_detail["calendar_violation"].sum())
            check_count = int(len(current_detail))

            summary_rows.append(
                {
                    "grid_name": grid_name,
                    "mode_name": mode_name,
                    "expiry_pairs": int(len(expiries) - 1),
                    "k_points": int(mode_df["k"].nunique()),
                    "calendar_checks": check_count,
                    "calendar_violation_count": violation_count,
                    "calendar_violation_rate": float(violation_count / max(check_count, 1)),
                    "max_calendar_violation_amount": float(current_detail["calendar_violation_amount"].max()),
                    "min_calendar_diff": float(current_detail["calendar_diff"].min()),
                    "calendar_pass": bool(violation_count == 0),
                }
            )

    detailed = pd.concat(detailed_rows, ignore_index=True) if detailed_rows else pd.DataFrame()
    summary = pd.DataFrame(summary_rows)

    return detailed, summary


n11_pre_repair_calendar_detail, n11_pre_repair_calendar_summary = calendar_diagnostics_for_grid(
    n11_pre_repair_surface_grid
)


# ------------------------------------------------------------
# Candidate mode ranking before repair
# ------------------------------------------------------------

mode_fit_core = (
    n11_svi_best_summary_by_mode
    .merge(
        n11_pre_repair_parameter_summary,
        on="mode_name",
        how="left",
        suffixes=("", "_param"),
    )
)

calendar_wide = (
    n11_pre_repair_calendar_summary
    .pivot_table(
        index="mode_name",
        columns="grid_name",
        values=[
            "calendar_pass",
            "calendar_violation_count",
            "calendar_violation_rate",
            "max_calendar_violation_amount",
            "min_calendar_diff",
        ],
        aggfunc="first",
    )
)

calendar_wide.columns = [
    f"{metric}_{grid_name}"
    for metric, grid_name in calendar_wide.columns
]
calendar_wide = calendar_wide.reset_index()

residual_all = (
    n11_pre_repair_contract_residual_summary
    .query("band == 'all'")
    .drop(columns=["band"])
    .rename(
        columns={
            "weighted_rmse_iv": "all_weighted_rmse_iv",
            "weighted_mae_iv": "all_weighted_mae_iv",
            "median_abs_iv_residual": "all_median_abs_iv_residual",
            "p95_abs_iv_residual": "all_p95_abs_iv_residual",
            "max_abs_iv_residual": "all_max_abs_iv_residual",
            "weighted_rmse_price": "all_weighted_rmse_price",
            "weighted_mae_price": "all_weighted_mae_price",
            "median_abs_price_residual": "all_median_abs_price_residual",
            "p95_abs_price_residual": "all_p95_abs_price_residual",
            "max_abs_price_residual": "all_max_abs_price_residual",
        }
    )
)

residual_atm = (
    n11_pre_repair_contract_residual_summary
    .query("band == 'atm_abs_k_le_005'")
    .drop(columns=["band"])
    .rename(
        columns={
            "rows": "atm_rows",
            "unique_expiries": "atm_unique_expiries",
            "weighted_rmse_iv": "atm_weighted_rmse_iv",
            "weighted_mae_iv": "atm_weighted_mae_iv",
            "median_abs_iv_residual": "atm_median_abs_iv_residual",
            "p95_abs_iv_residual": "atm_p95_abs_iv_residual",
            "max_abs_iv_residual": "atm_max_abs_iv_residual",
            "weighted_rmse_price": "atm_weighted_rmse_price",
            "weighted_mae_price": "atm_weighted_mae_price",
            "median_abs_price_residual": "atm_median_abs_price_residual",
            "p95_abs_price_residual": "atm_p95_abs_price_residual",
            "max_abs_price_residual": "atm_max_abs_price_residual",
        }
    )
)

n11_pre_repair_mode_ranking = (
    mode_fit_core
    .merge(calendar_wide, on="mode_name", how="left")
    .merge(residual_all, on="mode_name", how="left")
    .merge(residual_atm, on="mode_name", how="left")
)

# Fill boolean calendar pass fields that may be object dtype.
for col in n11_pre_repair_mode_ranking.columns:
    if col.startswith("calendar_pass_"):
        n11_pre_repair_mode_ranking[col] = n11_pre_repair_mode_ranking[col].fillna(False).astype(bool)

# Ranking score: lower is better.
# This is intentionally not RMSE-only. It penalizes static-arb violations and parameter degeneracy.
n11_pre_repair_mode_ranking["pre_repair_score"] = (
    1_000.0 * n11_pre_repair_mode_ranking["median_best_rmse_iv"].fillna(1.0)
    + 100.0 * n11_pre_repair_mode_ranking["median_best_rmse_price"].fillna(1.0)
    + 50.0 * n11_pre_repair_mode_ranking.get("max_calendar_violation_amount_common_inner_95pct_support", 0.0).fillna(0.0)
    + 10.0 * n11_pre_repair_mode_ranking.get("calendar_violation_rate_common_inner_95pct_support", 1.0).fillna(1.0)
    + 5.0 * n11_pre_repair_mode_ranking["max_parameter_risk_score"].fillna(10.0)
    + 2.0 * n11_pre_repair_mode_ranking["high_parameter_risk_slices"].fillna(10.0)
    + 1.0 * n11_pre_repair_mode_ranking["total_g_violation_points"].fillna(999.0)
)

# Preference discipline:
#   - broad/repair-aware modes have all 8 expiries
#   - strict has excellent local fit but only 4 expiries, so do not let it win
#     merely because its subset is easier to fit.
n11_pre_repair_mode_ranking["coverage_penalty"] = np.maximum(
    0,
    CONFIG.min_primary_expiries - n11_pre_repair_mode_ranking["selected_expiries"].fillna(0),
) * 100.0

n11_pre_repair_mode_ranking["full_expiry_coverage_bonus"] = np.where(
    n11_pre_repair_mode_ranking["selected_expiries"] >= n11_canonical_surface["expiry"].nunique(),
    -5.0,
    0.0,
)

n11_pre_repair_mode_ranking["pre_repair_selection_score"] = (
    n11_pre_repair_mode_ranking["pre_repair_score"]
    + n11_pre_repair_mode_ranking["coverage_penalty"]
    + n11_pre_repair_mode_ranking["full_expiry_coverage_bonus"]
)

n11_pre_repair_mode_ranking = (
    n11_pre_repair_mode_ranking
    .sort_values("pre_repair_selection_score")
    .reset_index(drop=True)
)

N11_PRE_REPAIR_SELECTED_MODE = str(n11_pre_repair_mode_ranking.loc[0, "mode_name"])

# Override guard: prefer weighted_broad_svi over repair_aware_svi when their scores are close,
# because weighted_broad_svi is the intended default market-information candidate.
if "weighted_broad_svi" in set(n11_pre_repair_mode_ranking["mode_name"]):
    best_score = float(n11_pre_repair_mode_ranking["pre_repair_selection_score"].min())
    wb_score = float(
        n11_pre_repair_mode_ranking
        .loc[n11_pre_repair_mode_ranking["mode_name"] == "weighted_broad_svi", "pre_repair_selection_score"]
        .iloc[0]
    )
    if wb_score <= best_score * 1.10 + 1.0e-12:
        N11_PRE_REPAIR_SELECTED_MODE = "weighted_broad_svi"


n11_pre_repair_selected_parameters = (
    n11_best_svi_parameters
    .loc[n11_best_svi_parameters["mode_name"] == N11_PRE_REPAIR_SELECTED_MODE]
    .copy()
    .sort_values("expiry")
    .reset_index(drop=True)
)

n11_pre_repair_selected_contract_points = (
    n11_best_svi_contract_points
    .loc[n11_best_svi_contract_points["mode_name"] == N11_PRE_REPAIR_SELECTED_MODE]
    .copy()
    .sort_values(["expiry", "k", "strike"])
    .reset_index(drop=True)
)

n11_pre_repair_selected_grid = (
    n11_pre_repair_surface_grid
    .loc[n11_pre_repair_surface_grid["mode_name"] == N11_PRE_REPAIR_SELECTED_MODE]
    .copy()
    .sort_values(["grid_name", "tau_years", "k"])
    .reset_index(drop=True)
)

n11_pre_repair_selected_calendar_summary = (
    n11_pre_repair_calendar_summary
    .loc[n11_pre_repair_calendar_summary["mode_name"] == N11_PRE_REPAIR_SELECTED_MODE]
    .copy()
    .reset_index(drop=True)
)

n11_pre_repair_selected_parameter_audit = (
    n11_pre_repair_parameter_audit
    .loc[n11_pre_repair_parameter_audit["mode_name"] == N11_PRE_REPAIR_SELECTED_MODE]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Audit gate
# ------------------------------------------------------------

selected_inner_calendar_pass = bool(
    n11_pre_repair_selected_calendar_summary
    .query("grid_name == 'common_inner_95pct_support'")["calendar_pass"]
    .iloc[0]
) if len(n11_pre_repair_selected_calendar_summary.query("grid_name == 'common_inner_95pct_support'")) else False

selected_full_calendar_pass = bool(
    n11_pre_repair_selected_calendar_summary
    .query("grid_name == 'common_full_support'")["calendar_pass"]
    .iloc[0]
) if len(n11_pre_repair_selected_calendar_summary.query("grid_name == 'common_full_support'")) else False

selected_butterfly_pass_count = int(
    n11_pre_repair_selected_parameters["butterfly_grid_pass"].sum()
)

selected_expiry_count = int(n11_pre_repair_selected_parameters["expiry"].nunique())

selected_high_parameter_risk_slices = int(
    (n11_pre_repair_selected_parameter_audit["parameter_risk_label"] == "high").sum()
)

n11_pre_repair_audit_gate = pd.DataFrame(
    [
        {
            "check_name": "parameter_audit_available",
            "check_type": "blocking",
            "passes": len(n11_pre_repair_parameter_audit) > 0,
            "observed_value": float(len(n11_pre_repair_parameter_audit)),
            "threshold_or_requirement": "> 0",
        },
        {
            "check_name": "contract_residual_summary_available",
            "check_type": "blocking",
            "passes": len(n11_pre_repair_contract_residual_summary) > 0,
            "observed_value": float(len(n11_pre_repair_contract_residual_summary)),
            "threshold_or_requirement": "> 0",
        },
        {
            "check_name": "calendar_diagnostics_available",
            "check_type": "blocking",
            "passes": len(n11_pre_repair_calendar_summary) > 0,
            "observed_value": float(len(n11_pre_repair_calendar_summary)),
            "threshold_or_requirement": "> 0",
        },
        {
            "check_name": "mode_ranking_available",
            "check_type": "blocking",
            "passes": len(n11_pre_repair_mode_ranking) > 0,
            "observed_value": float(len(n11_pre_repair_mode_ranking)),
            "threshold_or_requirement": "> 0",
        },
        {
            "check_name": "selected_mode_has_min_expiry_coverage",
            "check_type": "blocking",
            "passes": selected_expiry_count >= CONFIG.min_primary_expiries,
            "observed_value": float(selected_expiry_count),
            "threshold_or_requirement": f">= {CONFIG.min_primary_expiries}",
        },
        {
            "check_name": "selected_mode_butterfly_grid_passes",
            "check_type": "blocking",
            "passes": selected_butterfly_pass_count == selected_expiry_count,
            "observed_value": float(selected_butterfly_pass_count),
            "threshold_or_requirement": "all selected expiries",
        },
        {
            "check_name": "selected_mode_inner_calendar_pass",
            "check_type": "warning",
            "passes": selected_inner_calendar_pass,
            "observed_value": float(selected_inner_calendar_pass),
            "threshold_or_requirement": "preferred True; repair layer can address False",
        },
        {
            "check_name": "selected_mode_full_calendar_pass",
            "check_type": "warning",
            "passes": selected_full_calendar_pass,
            "observed_value": float(selected_full_calendar_pass),
            "threshold_or_requirement": "preferred True; repair layer can address False",
        },
        {
            "check_name": "selected_mode_high_parameter_risk_slices",
            "check_type": "warning",
            "passes": selected_high_parameter_risk_slices == 0,
            "observed_value": float(selected_high_parameter_risk_slices),
            "threshold_or_requirement": "preferred 0; document if > 0",
        },
    ]
)

N11_PRE_REPAIR_AUDIT_PASS = bool(
    n11_pre_repair_audit_gate
    .query("check_type == 'blocking'")["passes"]
    .all()
)


# ------------------------------------------------------------
# Persist outputs
# ------------------------------------------------------------

_write_table_pair(
    n11_pre_repair_parameter_audit,
    "n11_pre_repair_parameter_audit",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_pre_repair_parameter_summary,
    "n11_pre_repair_parameter_summary",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_pre_repair_contract_residual_summary,
    "n11_pre_repair_contract_residual_summary",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_pre_repair_surface_grid,
    "n11_pre_repair_surface_grid",
    N11_GRID_DIR,
)

_write_table_pair(
    n11_pre_repair_calendar_detail,
    "n11_pre_repair_calendar_detail",
    N11_DIAGNOSTIC_DIR,
)

_write_table_pair(
    n11_pre_repair_calendar_summary,
    "n11_pre_repair_calendar_summary",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_pre_repair_mode_ranking,
    "n11_pre_repair_mode_ranking",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_pre_repair_selected_parameters,
    "n11_pre_repair_selected_parameters",
    N11_PARAM_DIR,
)

_write_table_pair(
    n11_pre_repair_selected_contract_points,
    "n11_pre_repair_selected_contract_points",
    N11_HANDOFF_DIR,
)

_write_table_pair(
    n11_pre_repair_selected_grid,
    "n11_pre_repair_selected_grid",
    N11_GRID_DIR,
)

_write_table_pair(
    n11_pre_repair_selected_calendar_summary,
    "n11_pre_repair_selected_calendar_summary",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_pre_repair_audit_gate,
    "n11_pre_repair_audit_gate",
    N11_TABLE_DIR,
)


# ------------------------------------------------------------
# Console summary
# ------------------------------------------------------------

print("Notebook 11 pre-repair surface audit complete.")
print(f"Pre-repair audit pass: {N11_PRE_REPAIR_AUDIT_PASS}")
print(f"Selected pre-repair mode: {N11_PRE_REPAIR_SELECTED_MODE}")
print(f"Selected expiries: {selected_expiry_count:,}")
print(f"Selected butterfly pass count: {selected_butterfly_pass_count:,}/{selected_expiry_count:,}")
print(f"Selected inner calendar pass: {selected_inner_calendar_pass}")
print(f"Selected full calendar pass: {selected_full_calendar_pass}")
print(f"Selected high-parameter-risk slices: {selected_high_parameter_risk_slices:,}")

display(n11_pre_repair_parameter_summary)
display(n11_pre_repair_contract_residual_summary)
display(n11_pre_repair_calendar_summary)
display(n11_pre_repair_mode_ranking)
display(n11_pre_repair_selected_parameter_audit)
display(n11_pre_repair_audit_gate)

if not N11_PRE_REPAIR_AUDIT_PASS:
    raise RuntimeError(
        "Notebook 11 pre-repair audit gate failed. "
        "Do not proceed to repair until blocking audit checks pass."
    )

Notebook 11 pre-repair surface audit complete.
Pre-repair audit pass: True
Selected pre-repair mode: weighted_broad_svi
Selected expiries: 8
Selected butterfly pass count: 8/8
Selected inner calendar pass: False
Selected full calendar pass: False
Selected high-parameter-risk slices: 0


,mode_name,fitted_expiries,usable_fits,success_fits,near_boundary_slices,degenerate_slices,high_parameter_risk_slices,median_parameter_risk_score,max_parameter_risk_score,min_analytic_w_min,median_analytic_w_min,median_abs_rho,max_abs_rho,median_b,max_b,median_sigma,max_sigma
0,repair_aware_svi,8,7,8,2,6,0,1.5000000000,4.0000000000,-0.0000000042,0.0008502074,0.8975508231,0.9500519497,0.5519051922,2.0043662396,0.2574053892,1.6442447456
1,strict_svi,4,3,4,3,3,0,3.7500000000,4.0000000000,-0.0000000097,0.0008590739,0.7452728010,0.9500241248,0.5942763177,0.6970420525,0.2113922680,1.7295654638
2,weighted_broad_svi,8,7,8,2,5,0,1.5000000000,4.0000000000,-0.0000000078,0.0008211565,0.8992060431,0.9501047742,0.5036512868,2.0086054221,0.2690865887,1.7519163663


,mode_name,band,rows,unique_expiries,weighted_rmse_iv,weighted_mae_iv,median_abs_iv_residual,p95_abs_iv_residual,max_abs_iv_residual,weighted_rmse_price,weighted_mae_price,median_abs_price_residual,p95_abs_price_residual,max_abs_price_residual
0,repair_aware_svi,all,1260,8,0.0016699445,0.0008430893,0.0006372968,0.0032721343,0.0384506639,0.0532769434,0.0414835926,0.0217432977,0.0912086215,0.1505369458
1,repair_aware_svi,inner_abs_k_le_015,1026,8,0.0016789910,0.0008441089,0.0005952689,0.0035610349,0.0384506639,0.0537089300,0.0419512462,0.0261470930,0.0981492658,0.1505369458
2,repair_aware_svi,atm_abs_k_le_005,522,8,0.0013091993,0.0008204648,0.0006887457,0.0024424421,0.0191224028,0.0579475878,0.0464388783,0.0420447968,0.1188916239,0.1505369458
3,strict_svi,all,140,4,0.0002107590,0.0001602315,0.0001289475,0.0003876107,0.0008776236,0.0178262772,0.0129649814,0.0071625660,0.0317448314,0.0656427282
4,strict_svi,inner_abs_k_le_015,133,4,0.0002127255,0.0001625789,0.0001380187,0.0004160422,0.0008776236,0.0180003089,0.0131955824,0.0078283985,0.0319631475,0.0656427282
5,strict_svi,atm_abs_k_le_005,104,4,0.0002087349,0.0001578525,0.0001340207,0.0003835394,0.0008776236,0.0186153821,0.0139458751,0.0110181219,0.0322438396,0.0656427282
6,weighted_broad_svi,all,1260,8,0.0017234039,0.0008593673,0.0006304589,0.0032142563,0.0386599694,0.0546537251,0.0420469790,0.0215653395,0.0927949973,0.1545535971
7,weighted_broad_svi,inner_abs_k_le_015,1026,8,0.0017378878,0.0008634932,0.0006201237,0.0038288909,0.0386599694,0.0552870937,0.0427663898,0.0277088392,0.1014628644,0.1545535971
8,weighted_broad_svi,atm_abs_k_le_005,522,8,0.0013343947,0.0008422841,0.0007031313,0.0024379854,0.0192816658,0.0599872027,0.0477667649,0.0435090211,0.1231439751,0.1545535971


,grid_name,mode_name,expiry_pairs,k_points,calendar_checks,calendar_violation_count,calendar_violation_rate,max_calendar_violation_amount,min_calendar_diff,calendar_pass
0,common_full_support,repair_aware_svi,7,181,1267,110,0.0868192581,0.0918509750,-0.0918509850,False
1,common_full_support,strict_svi,3,181,543,135,0.2486187845,0.0341174411,-0.0341174511,False
2,common_full_support,weighted_broad_svi,7,181,1267,109,0.0860299921,0.0917503099,-0.0917503199,False
3,common_inner_95pct_support,repair_aware_svi,7,181,1267,64,0.0505130229,0.0192566582,-0.0192566682,False
4,common_inner_95pct_support,strict_svi,3,181,543,33,0.0607734807,0.0019412849,-0.0019412949,False
5,common_inner_95pct_support,weighted_broad_svi,7,181,1267,64,0.0505130229,0.0197909836,-0.0197909936,False


,mode_name,selected_expiries,selected_usable_fits,selected_successful_optimizer_fits,positive_variance_grid_passes,butterfly_grid_passes,total_g_violation_points,median_best_rmse_w,median_best_rmse_iv,median_best_rmse_price,max_best_abs_iv_residual,max_best_g_violation_amount,fitted_expiries,usable_fits,success_fits,near_boundary_slices,degenerate_slices,high_parameter_risk_slices,median_parameter_risk_score,max_parameter_risk_score,min_analytic_w_min,median_analytic_w_min,median_abs_rho,max_abs_rho,median_b,max_b,median_sigma,max_sigma,calendar_pass_common_full_support,calendar_pass_common_inner_95pct_support,calendar_violation_count_common_full_support,calendar_violation_count_common_inner_95pct_support,calendar_violation_rate_common_full_support,calendar_violation_rate_common_inner_95pct_support,max_calendar_violation_amount_common_full_support,max_calendar_violation_amount_common_inner_95pct_support,min_calendar_diff_common_full_support,min_calendar_diff_common_inner_95pct_support,rows,unique_expiries,all_weighted_rmse_iv,all_weighted_mae_iv,all_median_abs_iv_residual,all_p95_abs_iv_residual,all_max_abs_iv_residual,all_weighted_rmse_price,all_weighted_mae_price,all_median_abs_price_residual,all_p95_abs_price_residual,all_max_abs_price_residual,atm_rows,atm_unique_expiries,atm_weighted_rmse_iv,atm_weighted_mae_iv,atm_median_abs_iv_residual,atm_p95_abs_iv_residual,atm_max_abs_iv_residual,atm_weighted_rmse_price,atm_weighted_mae_price,atm_median_abs_price_residual,atm_p95_abs_price_residual,atm_max_abs_price_residual,pre_repair_score,coverage_penalty,full_expiry_coverage_bonus,pre_repair_selection_score
0,strict_svi,4,3,4,4,4,0,0.0000065697,0.0002038855,0.0166116598,0.0008776236,0.0000000000,4,3,4,3,3,0,3.7500000000,4.0000000000,-0.0000000097,0.0008590739,0.7452728010,0.9500241248,0.5942763177,0.6970420525,0.2113922680,1.7295654638,False,False,135,33,0.2486187845,0.0607734807,0.0341174411,0.0019412849,-0.0341174511,-0.0019412949,140,4,0.0002107590,0.0001602315,0.0001289475,0.0003876107,0.0008776236,0.0178262772,0.0129649814,0.0071625660,0.0317448314,0.0656427282,104,4,0.0002087349,0.0001578525,0.0001340207,0.0003835394,0.0008776236,0.0186153821,0.0139458751,0.0110181219,0.0322438396,0.0656427282,22.5698504986,0.0000000000,0.0000000000,22.5698504986
1,repair_aware_svi,8,7,8,8,8,0,0.0000286313,0.0018575933,0.0546484897,0.0384506639,0.0000000000,8,7,8,2,6,0,1.5000000000,4.0000000000,-0.0000000042,0.0008502074,0.8975508231,0.9500519497,0.5519051922,2.0043662396,0.2574053892,1.6442447456,False,False,110,64,0.0868192581,0.0505130229,0.0918509750,0.0192566582,-0.0918509850,-0.0192566682,1260,8,0.0016699445,0.0008430893,0.0006372968,0.0032721343,0.0384506639,0.0532769434,0.0414835926,0.0217432977,0.0912086215,0.1505369458,522,8,0.0013091993,0.0008204648,0.0006887457,0.0024424421,0.0191224028,0.0579475878,0.0464388783,0.0420447968,0.1188916239,0.1505369458,28.7904054359,0.0000000000,-5.0000000000,23.7904054359
2,weighted_broad_svi,8,7,8,8,8,0,0.0000299764,0.0019403535,0.0555896811,0.0386599694,0.0000000000,8,7,8,2,5,0,1.5000000000,4.0000000000,-0.0000000078,0.0008211565,0.8992060431,0.9501047742,0.5036512868,2.0086054221,0.2690865887,1.7519163663,False,False,109,64,0.0860299921,0.0505130229,0.0917503099,0.0197909836,-0.0917503199,-0.0197909936,1260,8,0.0017234039,0.0008593673,0.0006304589,0.0032142563,0.0386599694,0.0546537251,0.0420469790,0.0215653395,0.0927949973,0.1545535971,522,8,0.0013343947,0.0008422841,0.0007031313,0.0024379854,0.0192816658,0.0599872027,0.0477667649,0.0435090211,0.1231439751,0.1545535971,28.9940010220,0.0000000000,-5.0000000000,23.9940010220


,mode_name,expiry,fit_rank_within_group,fit_a,fit_b,fit_rho,fit_m,fit_sigma,analytic_k_min,analytic_w_min,rows,unique_contracts,success,usable_fit,cost,weighted_rmse_w,weighted_rmse_iv,weighted_rmse_price,positive_total_variance_grid_pass,butterfly_grid_pass,g_violation_points,min_grid_g,max_grid_g_violation_amount,a_boundary_distance,b_boundary_distance,rho_boundary_distance,m_boundary_distance,sigma_boundary_distance,min_parameter_boundary_distance,a_near_boundary,b_near_boundary,rho_near_boundary,m_near_boundary,sigma_near_boundary,rho_extreme_abs_ge_090,rho_extreme_abs_ge_095,a_large_negative,b_large,sigma_large,parameter_boundary_flag_count,parameter_degeneracy_flag_count,parameter_risk_score,parameter_risk_label
0,weighted_broad_svi,2026-07-10 00:00:00+00:00,1,-0.0132376001,0.3171162592,-0.9501047742,-0.3456910225,0.1346628658,0.0644761361,0.0000830320,97,97,True,True,0.0059561486,0.0000070266,0.0021138917,0.0272367644,True,True,0,0.0825578413,0.0000000000,0.1644604000,0.0634232500,0.0244720850,0.4423848296,0.0269323785,0.0244720850,False,False,True,False,False,True,True,False,False,False,1,2,4.0000000000,moderate
1,weighted_broad_svi,2026-07-17 00:00:00+00:00,1,-0.0021808720,0.0323747608,-0.8820614519,-0.1016986910,0.1429806535,0.1659908063,-0.0000000078,130,130,True,False,0.0064396316,0.0000178392,0.0022146115,0.0450652924,True,True,0,0.0265358284,0.0000000000,0.1663031880,0.0064749502,0.0585278019,0.4830502182,0.0285959364,0.0064749502,False,False,False,False,False,False,False,False,False,False,0,0,0.0000000000,none
2,weighted_broad_svi,2026-07-24 00:00:00+00:00,1,-0.0043429766,0.0410657455,-0.8129848661,-0.1069495437,0.1877883831,0.1552402115,0.0001474117,152,152,True,True,0.0084362707,0.0000279496,0.0024099626,0.0684267074,True,True,0,0.0425300781,0.0000000000,0.1659428372,0.0082131471,0.0931006676,0.4821750760,0.0375574841,0.0082131471,False,False,False,False,False,False,False,False,False,False,0,0,0.0000000000,none
3,weighted_broad_svi,2026-07-31 00:00:00+00:00,1,-0.0296668335,0.0803753758,-0.3983723969,-0.0825182159,0.4115933172,0.0962466861,0.0006767333,175,175,True,True,0.0057257051,0.0000320032,0.0017668153,0.0659135738,True,True,0,0.0460682327,0.0000000000,0.1617221944,0.0160750732,0.3006144159,0.4862469640,0.0823184799,0.0160750732,False,False,False,False,False,False,False,False,False,False,0,0,0.0000000000,none
4,weighted_broad_svi,2026-08-07 00:00:00+00:00,1,-0.9999999998,0.6901863143,0.5609839150,1.2767016202,1.7519163663,0.0895003612,0.0009655796,131,131,True,True,0.0078747243,0.0000466639,0.0021304039,0.0699006395,True,True,0,0.0715345985,0.0000000000,0.0000000000,0.1380372611,0.2192272698,0.2872163966,0.3503831433,0.0000000000,True,False,False,False,False,False,False,True,False,True,1,2,3.5000000000,moderate
5,weighted_broad_svi,2026-08-21 00:00:00+00:00,1,-0.3219013963,2.0086054221,0.9163506343,1.0029182636,0.4023367414,0.0820816784,0.0016573458,173,173,True,True,0.0035377582,0.0000383527,0.0010559864,0.0487660513,True,True,0,0.0781726270,0.0000000000,0.1130164340,0.4017210832,0.0413660489,0.3328469561,0.0804671644,0.0413660489,False,False,False,False,False,True,False,True,True,False,0,3,1.5000000000,low
6,weighted_broad_svi,2026-08-31 00:00:00+00:00,1,-0.2127906809,2.0080390745,0.9257707413,0.7737488902,0.2830865785,0.0805898743,0.0021315266,187,187,True,True,0.0017333675,0.0000275093,0.0005442672,0.0382829960,True,True,0,0.1105895831,0.0000000000,0.1312015532,0.4016078137,0.0366512806,0.3710418516,0.0566171270,0.0366512806,False,False,False,False,False,True,False,True,True,False,0,3,1.5000000000,low
7,weighted_broad_svi,2026-09-18 00:00:00+00:00,1,-0.1941896287,2.0021284095,0.9224089294,0.6971172718,0.2550865989,0.0878857272,0.0030564287,215,215,True,True,0.0020478271,0.0000433440,0.0005959005,0.0624133110,True,True,0,0.1243285038,0.0000000000,0.1343017285,0.4004256807,0.0383338692,0.3838137880,0.0510171300,0.0383338692,False,False,False,False,False,True,False,True,True,False,0,3,1.5000000000,low


,check_name,check_type,passes,observed_value,threshold_or_requirement
0,parameter_audit_available,blocking,True,20.0000000000,> 0
1,contract_residual_summary_available,blocking,True,9.0000000000,> 0
2,calendar_diagnostics_available,blocking,True,6.0000000000,> 0
3,mode_ranking_available,blocking,True,3.0000000000,> 0
4,selected_mode_has_min_expiry_coverage,blocking,True,8.0000000000,>= 4
5,selected_mode_butterfly_grid_passes,blocking,True,8.0000000000,all selected expiries
6,selected_mode_inner_calendar_pass,warning,False,0.0000000000,preferred True; repair layer can address False
7,selected_mode_full_calendar_pass,warning,False,0.0000000000,preferred True; repair layer can address False
8,selected_mode_high_parameter_risk_slices,warning,True,0.0000000000,preferred 0; document if > 0


In [8]:
# ------------------------------------------------------------
# Notebook 11 cell 08: Calendar repair layer on selected SVI surface
# ------------------------------------------------------------
# The selected pre-repair surface already passes butterfly diagnostics slice-by-slice,
# but it fails calendar monotonicity:
#
#     w(k, T_2) >= w(k, T_1)   whenever   T_2 > T_1
#
# This cell repairs calendar violations across maturity at fixed k.
#
# Two repair candidates are built:
#
#   1. calendar_pava
#      Weighted isotonic regression across maturities.
#      Minimal weighted L2 adjustment, but may move both near and far maturities.
#
#   2. calendar_cummax
#      Upward-only cumulative maximum repair.
#      Never lowers a fitted total variance, but can create larger upward shifts.
#
# The cell ranks both repair candidates, selects one, and creates a contract-level
# repaired handoff candidate. It does not yet declare the final Notebook 12 handoff.

from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.stats import norm


# ------------------------------------------------------------
# Pre-flight checks
# ------------------------------------------------------------

required_globals_for_calendar_repair = [
    "N11_PRE_REPAIR_AUDIT_PASS",
    "N11_PRE_REPAIR_SELECTED_MODE",
    "n11_pre_repair_selected_grid",
    "n11_pre_repair_selected_contract_points",
    "calendar_diagnostics_for_grid",
    "bsm_forward_price_from_total_variance",
    "CONFIG",
]

missing_calendar_repair_globals = [
    name for name in required_globals_for_calendar_repair if name not in globals()
]

if missing_calendar_repair_globals:
    raise NameError(
        f"Missing required objects before calendar repair: {missing_calendar_repair_globals}"
    )

if not bool(N11_PRE_REPAIR_AUDIT_PASS):
    raise RuntimeError("Pre-repair audit did not pass. Run/fix the previous cell first.")


# ------------------------------------------------------------
# Weighted isotonic regression primitive
# ------------------------------------------------------------

def weighted_pava_increasing(y: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """
    Weighted pool-adjacent-violators algorithm for an increasing sequence.

    Solves:

        min_x sum_i weights_i * (x_i - y_i)^2

    subject to:

        x_1 <= x_2 <= ... <= x_n
    """
    y = np.asarray(y, dtype=float)
    weights = np.asarray(weights, dtype=float)

    if y.ndim != 1:
        raise ValueError("y must be one-dimensional.")

    if len(y) != len(weights):
        raise ValueError("y and weights must have the same length.")

    valid = np.isfinite(y) & np.isfinite(weights) & (weights > 0.0)

    if not valid.all():
        # Repair only finite positions, then restore original for invalid points.
        out = y.copy()
        idx = np.flatnonzero(valid)
        if len(idx) > 0:
            out[idx] = weighted_pava_increasing(y[idx], weights[idx])
        return out

    levels: list[float] = []
    block_weights: list[float] = []
    block_indices: list[list[int]] = []

    for i, (value, weight) in enumerate(zip(y, weights)):
        levels.append(float(value))
        block_weights.append(float(weight))
        block_indices.append([i])

        while len(levels) >= 2 and levels[-2] > levels[-1]:
            w_left = block_weights[-2]
            w_right = block_weights[-1]
            merged_weight = w_left + w_right

            merged_level = (
                levels[-2] * w_left
                + levels[-1] * w_right
            ) / max(merged_weight, N11_TINY)

            merged_indices = block_indices[-2] + block_indices[-1]

            levels[-2] = float(merged_level)
            block_weights[-2] = float(merged_weight)
            block_indices[-2] = merged_indices

            levels.pop()
            block_weights.pop()
            block_indices.pop()

    out = np.empty_like(y, dtype=float)

    for level, indices in zip(levels, block_indices):
        out[indices] = level

    return out


def cumulative_max_repair(y: np.ndarray) -> np.ndarray:
    """
    Upward-only calendar repair.

        repaired_w_i = max(w_1, ..., w_i)
    """
    y = np.asarray(y, dtype=float)
    return np.maximum.accumulate(y)


# ------------------------------------------------------------
# Repair candidates on common k-grids
# ------------------------------------------------------------

expiry_weight_map = (
    n11_pre_repair_selected_contract_points
    .groupby("expiry", dropna=False)["fit_weight"]
    .sum()
    .to_dict()
)

median_expiry_weight = float(
    np.nanmedian(list(expiry_weight_map.values()))
) if len(expiry_weight_map) else 1.0


def repair_one_grid_name(
    selected_grid: pd.DataFrame,
    *,
    grid_name: str,
    repair_method: str,
) -> pd.DataFrame:
    grid = (
        selected_grid
        .loc[selected_grid["grid_name"] == grid_name]
        .copy()
        .sort_values(["k", "tau_years", "expiry"])
        .reset_index(drop=True)
    )

    if len(grid) == 0:
        raise ValueError(f"No selected grid rows found for grid_name={grid_name}")

    repaired_frames = []

    for k_value, k_grp in grid.groupby("k", dropna=False):
        k_grp = k_grp.sort_values(["tau_years", "expiry"]).copy()

        raw_w = pd.to_numeric(k_grp["svi_total_variance"], errors="coerce").to_numpy(dtype=float)

        weights = np.array(
            [
                float(expiry_weight_map.get(expiry, median_expiry_weight))
                for expiry in k_grp["expiry"]
            ],
            dtype=float,
        )

        weights = np.where(np.isfinite(weights) & (weights > 0.0), weights, median_expiry_weight)

        if repair_method == "calendar_pava":
            repaired_w = weighted_pava_increasing(raw_w, weights)
        elif repair_method == "calendar_cummax":
            repaired_w = cumulative_max_repair(raw_w)
        else:
            raise ValueError(f"Unknown repair_method={repair_method}")

        repaired_w = np.maximum(repaired_w, CONFIG.positive_total_variance_floor)

        k_grp["source_mode_name"] = N11_PRE_REPAIR_SELECTED_MODE
        k_grp["mode_name"] = repair_method
        k_grp["repair_method"] = repair_method
        k_grp["pre_repair_total_variance"] = raw_w
        k_grp["svi_total_variance"] = repaired_w
        k_grp["repaired_total_variance"] = repaired_w
        k_grp["total_variance_repair"] = repaired_w - raw_w
        k_grp["abs_total_variance_repair"] = np.abs(k_grp["total_variance_repair"])

        tau = pd.to_numeric(k_grp["tau_years"], errors="coerce").to_numpy(dtype=float)
        pre_iv = np.full_like(raw_w, np.nan, dtype=float)
        repaired_iv = np.full_like(repaired_w, np.nan, dtype=float)

        pre_ok = np.isfinite(raw_w) & np.isfinite(tau) & (raw_w > 0.0) & (tau > 0.0)
        rep_ok = np.isfinite(repaired_w) & np.isfinite(tau) & (repaired_w > 0.0) & (tau > 0.0)

        pre_iv[pre_ok] = np.sqrt(raw_w[pre_ok] / tau[pre_ok])
        repaired_iv[rep_ok] = np.sqrt(repaired_w[rep_ok] / tau[rep_ok])

        k_grp["pre_repair_iv"] = pre_iv
        k_grp["svi_iv"] = repaired_iv
        k_grp["repaired_iv"] = repaired_iv
        k_grp["iv_repair"] = repaired_iv - pre_iv
        k_grp["abs_iv_repair"] = np.abs(k_grp["iv_repair"])

        repaired_frames.append(k_grp)

    repaired = pd.concat(repaired_frames, ignore_index=True)

    return repaired.sort_values(["grid_name", "mode_name", "tau_years", "k"]).reset_index(drop=True)


repair_grid_frames = []

for method in ["calendar_pava", "calendar_cummax"]:
    for grid_name in sorted(n11_pre_repair_selected_grid["grid_name"].unique()):
        repair_grid_frames.append(
            repair_one_grid_name(
                n11_pre_repair_selected_grid,
                grid_name=grid_name,
                repair_method=method,
            )
        )

n11_calendar_repair_grid_candidates = pd.concat(repair_grid_frames, ignore_index=True)


# ------------------------------------------------------------
# Calendar diagnostics after repair
# ------------------------------------------------------------

(
    n11_calendar_repair_detail,
    n11_calendar_repair_summary,
) = calendar_diagnostics_for_grid(n11_calendar_repair_grid_candidates)


# ------------------------------------------------------------
# Finite-difference static diagnostics on repaired grids
# ------------------------------------------------------------

def normalized_forward_call_from_total_variance(k: np.ndarray, total_variance: np.ndarray) -> np.ndarray:
    """
    Undiscounted forward-normalized Black-Scholes call price:

        c(k, w) = N(d_+) - exp(k) N(d_-)

    with:

        d_+ = -k / sqrt(w) + sqrt(w) / 2
        d_- = -k / sqrt(w) - sqrt(w) / 2
    """
    k = np.asarray(k, dtype=float)
    w = np.maximum(np.asarray(total_variance, dtype=float), N11_TINY)

    sqrt_w = np.sqrt(w)
    d_plus = -k / sqrt_w + 0.5 * sqrt_w
    d_minus = d_plus - sqrt_w

    return norm.cdf(d_plus) - np.exp(k) * norm.cdf(d_minus)


def repaired_grid_static_proxy_diagnostics(repaired_grid: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    detail_rows = []
    summary_rows = []

    group_cols = ["repair_method", "grid_name", "expiry"]

    for (repair_method, grid_name, expiry), grp in repaired_grid.groupby(group_cols, dropna=False):
        grp = grp.sort_values("k").copy()

        k = pd.to_numeric(grp["k"], errors="coerce").to_numpy(dtype=float)
        x = np.exp(k)
        w = pd.to_numeric(grp["repaired_total_variance"], errors="coerce").to_numpy(dtype=float)

        c = normalized_forward_call_from_total_variance(k, w)

        dc = np.diff(c)
        dx = np.diff(x)

        valid_first = np.isfinite(dc) & np.isfinite(dx) & (dx > 0.0)

        # Call price must be decreasing in strike.
        monotonicity_violation = np.zeros_like(dc, dtype=bool)
        monotonicity_violation[valid_first] = dc[valid_first] > abs(CONFIG.convexity_tolerance)

        slopes = np.full_like(dc, np.nan, dtype=float)
        slopes[valid_first] = dc[valid_first] / dx[valid_first]

        slope_diff = np.diff(slopes)
        valid_second = np.isfinite(slope_diff)

        # Convexity in strike means slope should be nondecreasing.
        convexity_violation = np.zeros_like(slope_diff, dtype=bool)
        convexity_violation[valid_second] = slope_diff[valid_second] < CONFIG.convexity_tolerance

        for i in range(len(dc)):
            detail_rows.append(
                {
                    "repair_method": repair_method,
                    "grid_name": grid_name,
                    "expiry": expiry,
                    "diagnostic_type": "monotonicity",
                    "k_left": float(k[i]),
                    "k_right": float(k[i + 1]),
                    "x_left": float(x[i]),
                    "x_right": float(x[i + 1]),
                    "call_left": float(c[i]),
                    "call_right": float(c[i + 1]),
                    "diagnostic_value": float(dc[i]),
                    "violation": bool(monotonicity_violation[i]),
                }
            )

        for i in range(len(slope_diff)):
            detail_rows.append(
                {
                    "repair_method": repair_method,
                    "grid_name": grid_name,
                    "expiry": expiry,
                    "diagnostic_type": "convexity",
                    "k_left": float(k[i]),
                    "k_right": float(k[i + 2]),
                    "x_left": float(x[i]),
                    "x_right": float(x[i + 2]),
                    "call_left": float(c[i]),
                    "call_right": float(c[i + 2]),
                    "diagnostic_value": float(slope_diff[i]),
                    "violation": bool(convexity_violation[i]),
                }
            )

        monotonicity_violation_count = int(monotonicity_violation.sum())
        convexity_violation_count = int(convexity_violation.sum())

        summary_rows.append(
            {
                "repair_method": repair_method,
                "grid_name": grid_name,
                "expiry": expiry,
                "grid_points": int(len(grp)),
                "monotonicity_checks": int(len(monotonicity_violation)),
                "monotonicity_violation_count": monotonicity_violation_count,
                "convexity_checks": int(len(convexity_violation)),
                "convexity_violation_count": convexity_violation_count,
                "total_static_proxy_violation_count": monotonicity_violation_count + convexity_violation_count,
                "min_call_price": float(np.nanmin(c)),
                "max_call_price": float(np.nanmax(c)),
                "min_slope_diff": float(np.nanmin(slope_diff)) if len(slope_diff) else np.nan,
                "static_proxy_pass": bool(
                    monotonicity_violation_count == 0
                    and convexity_violation_count == 0
                ),
            }
        )

    detail = pd.DataFrame(detail_rows)
    summary = pd.DataFrame(summary_rows)

    return detail, summary


(
    n11_calendar_repair_static_proxy_detail,
    n11_calendar_repair_static_proxy_summary,
) = repaired_grid_static_proxy_diagnostics(n11_calendar_repair_grid_candidates)


n11_calendar_repair_static_proxy_by_method = (
    n11_calendar_repair_static_proxy_summary
    .groupby(["repair_method", "grid_name"], dropna=False)
    .agg(
        expiries=("expiry", "nunique"),
        total_monotonicity_violations=("monotonicity_violation_count", "sum"),
        total_convexity_violations=("convexity_violation_count", "sum"),
        total_static_proxy_violations=("total_static_proxy_violation_count", "sum"),
        static_proxy_passes=("static_proxy_pass", "sum"),
        min_slope_diff=("min_slope_diff", "min"),
    )
    .reset_index()
)


# ------------------------------------------------------------
# Repair-bias summaries on grid
# ------------------------------------------------------------

n11_calendar_repair_grid_bias_summary = (
    n11_calendar_repair_grid_candidates
    .groupby(["repair_method", "grid_name"], dropna=False)
    .agg(
        rows=("k", "size"),
        expiries=("expiry", "nunique"),
        k_points=("k", "nunique"),
        mean_total_variance_repair=("total_variance_repair", "mean"),
        median_total_variance_repair=("total_variance_repair", "median"),
        median_abs_total_variance_repair=("abs_total_variance_repair", "median"),
        p95_abs_total_variance_repair=("abs_total_variance_repair", lambda s: float(np.nanpercentile(s, 95))),
        max_abs_total_variance_repair=("abs_total_variance_repair", "max"),
        mean_iv_repair=("iv_repair", "mean"),
        median_iv_repair=("iv_repair", "median"),
        median_abs_iv_repair=("abs_iv_repair", "median"),
        p95_abs_iv_repair=("abs_iv_repair", lambda s: float(np.nanpercentile(s, 95))),
        max_abs_iv_repair=("abs_iv_repair", "max"),
    )
    .reset_index()
)


# ------------------------------------------------------------
# Project repaired grids back to selected contract points
# ------------------------------------------------------------

def project_repaired_grid_to_contracts(
    selected_contract_points: pd.DataFrame,
    repaired_grid: pd.DataFrame,
    *,
    repair_method: str,
) -> pd.DataFrame:
    full_grid = (
        repaired_grid
        .loc[
            (repaired_grid["repair_method"] == repair_method)
            & (repaired_grid["grid_name"] == "common_full_support")
        ]
        .copy()
    )

    if len(full_grid) == 0:
        raise ValueError(f"No full-support repaired grid found for repair_method={repair_method}")

    frames = []

    for expiry, contract_grp in selected_contract_points.groupby("expiry", dropna=False):
        grid_grp = (
            full_grid
            .loc[full_grid["expiry"] == expiry]
            .copy()
            .sort_values("k")
        )

        if len(grid_grp) < 2:
            continue

        contract_grp = contract_grp.copy()

        grid_k = pd.to_numeric(grid_grp["k"], errors="coerce").to_numpy(dtype=float)
        grid_w = pd.to_numeric(grid_grp["repaired_total_variance"], errors="coerce").to_numpy(dtype=float)

        valid_grid = np.isfinite(grid_k) & np.isfinite(grid_w)
        grid_k = grid_k[valid_grid]
        grid_w = grid_w[valid_grid]

        sorter = np.argsort(grid_k)
        grid_k = grid_k[sorter]
        grid_w = grid_w[sorter]

        contract_k = pd.to_numeric(contract_grp["k"], errors="coerce").to_numpy(dtype=float)

        repaired_w = np.interp(
            contract_k,
            grid_k,
            grid_w,
            left=grid_w[0],
            right=grid_w[-1],
        )

        tau = pd.to_numeric(contract_grp["tau_years"], errors="coerce").to_numpy(dtype=float)

        repaired_iv = np.full_like(repaired_w, np.nan, dtype=float)
        valid_iv = np.isfinite(repaired_w) & np.isfinite(tau) & (repaired_w > 0.0) & (tau > 0.0)
        repaired_iv[valid_iv] = np.sqrt(repaired_w[valid_iv] / tau[valid_iv])

        repaired_price = bsm_forward_price_from_total_variance(
            forward=pd.to_numeric(contract_grp["forward"], errors="coerce").to_numpy(dtype=float),
            strike=pd.to_numeric(contract_grp["strike"], errors="coerce").to_numpy(dtype=float),
            tau=tau,
            discount_factor=pd.to_numeric(contract_grp["discount_factor"], errors="coerce").to_numpy(dtype=float),
            total_variance=repaired_w,
            option_type=contract_grp["option_type"].astype(str).to_numpy(),
        )

        contract_grp["source_mode_name"] = N11_PRE_REPAIR_SELECTED_MODE
        contract_grp["repair_method"] = repair_method

        contract_grp = contract_grp.rename(
            columns={
                "observed_total_variance": "market_total_variance",
                "fitted_total_variance": "pre_repair_total_variance",
                "observed_iv": "market_iv",
                "fitted_iv": "pre_repair_iv",
                "fitted_bsm_price": "pre_repair_bsm_price",
            }
        )

        contract_grp["repaired_total_variance"] = repaired_w
        contract_grp["repaired_iv"] = repaired_iv
        contract_grp["repaired_bsm_price"] = repaired_price

        contract_grp["total_variance_repair"] = (
            contract_grp["repaired_total_variance"]
            - contract_grp["pre_repair_total_variance"]
        )
        contract_grp["abs_total_variance_repair"] = contract_grp["total_variance_repair"].abs()

        contract_grp["iv_repair"] = (
            contract_grp["repaired_iv"]
            - contract_grp["pre_repair_iv"]
        )
        contract_grp["abs_iv_repair"] = contract_grp["iv_repair"].abs()

        contract_grp["post_repair_total_variance_residual_to_market"] = (
            contract_grp["repaired_total_variance"]
            - contract_grp["market_total_variance"]
        )
        contract_grp["post_repair_iv_residual_to_market"] = (
            contract_grp["repaired_iv"]
            - contract_grp["market_iv"]
        )
        contract_grp["post_repair_price_residual_to_market"] = (
            contract_grp["repaired_bsm_price"]
            - contract_grp["market_price"]
        )

        contract_grp["abs_post_repair_iv_residual_to_market"] = (
            contract_grp["post_repair_iv_residual_to_market"].abs()
        )
        contract_grp["abs_post_repair_price_residual_to_market"] = (
            contract_grp["post_repair_price_residual_to_market"].abs()
        )

        contract_grp["repair_bias_warn"] = (
            contract_grp["abs_iv_repair"]
            > CONFIG.global_abs_iv_repair_warn
        )
        contract_grp["repair_bias_fail"] = (
            contract_grp["abs_iv_repair"]
            > CONFIG.global_abs_iv_repair_fail
        )

        contract_grp["eligible_for_n12_after_calendar_repair"] = (
            np.isfinite(contract_grp["repaired_total_variance"])
            & (contract_grp["repaired_total_variance"] > CONFIG.positive_total_variance_floor)
            & np.isfinite(contract_grp["repaired_iv"])
            & (contract_grp["repaired_iv"] > 0.0)
            & ~contract_grp["repair_bias_fail"]
        )

        frames.append(contract_grp)

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)


contract_repair_frames = []

for method in ["calendar_pava", "calendar_cummax"]:
    contract_repair_frames.append(
        project_repaired_grid_to_contracts(
            n11_pre_repair_selected_contract_points,
            n11_calendar_repair_grid_candidates,
            repair_method=method,
        )
    )

n11_calendar_repaired_contract_candidates = pd.concat(contract_repair_frames, ignore_index=True)


# ------------------------------------------------------------
# Contract-level repair-bias summary
# ------------------------------------------------------------

def contract_repair_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    df = df.copy()
    df["abs_k"] = pd.to_numeric(df["k"], errors="coerce").abs()

    band_masks = {
        "all": pd.Series(True, index=df.index),
        "inner_abs_k_le_015": df["abs_k"] <= 0.1500,
        "atm_abs_k_le_005": df["abs_k"] <= 0.0500,
    }

    for repair_method, method_df in df.groupby("repair_method", dropna=False):
        for band_name, mask in band_masks.items():
            sub = method_df.loc[mask.loc[method_df.index]].copy()

            if len(sub) == 0:
                rows.append(
                    {
                        "repair_method": repair_method,
                        "band": band_name,
                        "rows": 0,
                        "expiries": 0,
                        "eligible_rows": 0,
                        "repair_warn_rows": 0,
                        "repair_fail_rows": 0,
                        "median_abs_iv_repair": np.nan,
                        "p95_abs_iv_repair": np.nan,
                        "max_abs_iv_repair": np.nan,
                        "median_abs_post_repair_iv_residual_to_market": np.nan,
                        "p95_abs_post_repair_iv_residual_to_market": np.nan,
                        "max_abs_post_repair_iv_residual_to_market": np.nan,
                        "median_abs_post_repair_price_residual_to_market": np.nan,
                        "p95_abs_post_repair_price_residual_to_market": np.nan,
                        "max_abs_post_repair_price_residual_to_market": np.nan,
                    }
                )
                continue

            rows.append(
                {
                    "repair_method": repair_method,
                    "band": band_name,
                    "rows": int(len(sub)),
                    "expiries": int(sub["expiry"].nunique()),
                    "eligible_rows": int(sub["eligible_for_n12_after_calendar_repair"].sum()),
                    "repair_warn_rows": int(sub["repair_bias_warn"].sum()),
                    "repair_fail_rows": int(sub["repair_bias_fail"].sum()),
                    "median_abs_iv_repair": float(np.nanmedian(sub["abs_iv_repair"])),
                    "p95_abs_iv_repair": float(np.nanpercentile(sub["abs_iv_repair"], 95)),
                    "max_abs_iv_repair": float(np.nanmax(sub["abs_iv_repair"])),
                    "median_abs_post_repair_iv_residual_to_market": float(
                        np.nanmedian(sub["abs_post_repair_iv_residual_to_market"])
                    ),
                    "p95_abs_post_repair_iv_residual_to_market": float(
                        np.nanpercentile(sub["abs_post_repair_iv_residual_to_market"], 95)
                    ),
                    "max_abs_post_repair_iv_residual_to_market": float(
                        np.nanmax(sub["abs_post_repair_iv_residual_to_market"])
                    ),
                    "median_abs_post_repair_price_residual_to_market": float(
                        np.nanmedian(sub["abs_post_repair_price_residual_to_market"])
                    ),
                    "p95_abs_post_repair_price_residual_to_market": float(
                        np.nanpercentile(sub["abs_post_repair_price_residual_to_market"], 95)
                    ),
                    "max_abs_post_repair_price_residual_to_market": float(
                        np.nanmax(sub["abs_post_repair_price_residual_to_market"])
                    ),
                }
            )

    return pd.DataFrame(rows)


n11_calendar_repaired_contract_bias_summary = contract_repair_summary(
    n11_calendar_repaired_contract_candidates
)


# ------------------------------------------------------------
# Repair-candidate ranking
# ------------------------------------------------------------

calendar_summary_wide = (
    n11_calendar_repair_summary
    .pivot_table(
        index="mode_name",
        columns="grid_name",
        values=[
            "calendar_pass",
            "calendar_violation_count",
            "calendar_violation_rate",
            "max_calendar_violation_amount",
        ],
        aggfunc="first",
    )
)

calendar_summary_wide.columns = [
    f"{metric}_{grid_name}"
    for metric, grid_name in calendar_summary_wide.columns
]
calendar_summary_wide = calendar_summary_wide.reset_index().rename(columns={"mode_name": "repair_method"})

static_proxy_wide = (
    n11_calendar_repair_static_proxy_by_method
    .pivot_table(
        index="repair_method",
        columns="grid_name",
        values=[
            "total_static_proxy_violations",
            "total_monotonicity_violations",
            "total_convexity_violations",
            "static_proxy_passes",
            "min_slope_diff",
        ],
        aggfunc="first",
    )
)

static_proxy_wide.columns = [
    f"{metric}_{grid_name}"
    for metric, grid_name in static_proxy_wide.columns
]
static_proxy_wide = static_proxy_wide.reset_index()

contract_all_bias = (
    n11_calendar_repaired_contract_bias_summary
    .query("band == 'all'")
    .drop(columns=["band"])
)

contract_atm_bias = (
    n11_calendar_repaired_contract_bias_summary
    .query("band == 'atm_abs_k_le_005'")
    .drop(columns=["band"])
    .rename(
        columns={
            "rows": "atm_rows",
            "expiries": "atm_expiries",
            "eligible_rows": "atm_eligible_rows",
            "repair_warn_rows": "atm_repair_warn_rows",
            "repair_fail_rows": "atm_repair_fail_rows",
            "median_abs_iv_repair": "atm_median_abs_iv_repair",
            "p95_abs_iv_repair": "atm_p95_abs_iv_repair",
            "max_abs_iv_repair": "atm_max_abs_iv_repair",
            "median_abs_post_repair_iv_residual_to_market": "atm_median_abs_post_repair_iv_residual_to_market",
            "p95_abs_post_repair_iv_residual_to_market": "atm_p95_abs_post_repair_iv_residual_to_market",
            "max_abs_post_repair_iv_residual_to_market": "atm_max_abs_post_repair_iv_residual_to_market",
            "median_abs_post_repair_price_residual_to_market": "atm_median_abs_post_repair_price_residual_to_market",
            "p95_abs_post_repair_price_residual_to_market": "atm_p95_abs_post_repair_price_residual_to_market",
            "max_abs_post_repair_price_residual_to_market": "atm_max_abs_post_repair_price_residual_to_market",
        }
    )
)

n11_calendar_repair_candidate_ranking = (
    calendar_summary_wide
    .merge(static_proxy_wide, on="repair_method", how="left")
    .merge(contract_all_bias, on="repair_method", how="left")
    .merge(contract_atm_bias, on="repair_method", how="left")
)

for col in n11_calendar_repair_candidate_ranking.columns:
    if col.startswith("calendar_pass_"):
        n11_calendar_repair_candidate_ranking[col] = (
            n11_calendar_repair_candidate_ranking[col]
            .fillna(False)
            .astype(bool)
        )

# Lower score is better.
n11_calendar_repair_candidate_ranking["calendar_repair_selection_score"] = (
    1_000_000.0
    * (~n11_calendar_repair_candidate_ranking["calendar_pass_common_full_support"]).astype(float)
    + 1_000_000.0
    * (~n11_calendar_repair_candidate_ranking["calendar_pass_common_inner_95pct_support"]).astype(float)
    + 1_000.0
    * n11_calendar_repair_candidate_ranking["p95_abs_iv_repair"].fillna(1.0)
    + 500.0
    * n11_calendar_repair_candidate_ranking["atm_p95_abs_iv_repair"].fillna(1.0)
    + 200.0
    * n11_calendar_repair_candidate_ranking["p95_abs_post_repair_iv_residual_to_market"].fillna(1.0)
    + 50.0
    * n11_calendar_repair_candidate_ranking[
        "total_static_proxy_violations_common_full_support"
    ].fillna(999.0)
    + 25.0
    * n11_calendar_repair_candidate_ranking[
        "total_static_proxy_violations_common_inner_95pct_support"
    ].fillna(999.0)
    + 5.0
    * n11_calendar_repair_candidate_ranking["repair_warn_rows"].fillna(999.0)
    + 100.0
    * n11_calendar_repair_candidate_ranking["repair_fail_rows"].fillna(999.0)
)

n11_calendar_repair_candidate_ranking = (
    n11_calendar_repair_candidate_ranking
    .sort_values("calendar_repair_selection_score")
    .reset_index(drop=True)
)

N11_SELECTED_CALENDAR_REPAIR_METHOD = str(
    n11_calendar_repair_candidate_ranking.loc[0, "repair_method"]
)

N11_CALENDAR_REPAIRED_GRID = (
    n11_calendar_repair_grid_candidates
    .loc[n11_calendar_repair_grid_candidates["repair_method"] == N11_SELECTED_CALENDAR_REPAIR_METHOD]
    .copy()
    .reset_index(drop=True)
)

N11_CALENDAR_REPAIRED_CONTRACT_POINTS = (
    n11_calendar_repaired_contract_candidates
    .loc[
        n11_calendar_repaired_contract_candidates["repair_method"]
        == N11_SELECTED_CALENDAR_REPAIR_METHOD
    ]
    .copy()
    .reset_index(drop=True)
)


n11_selected_calendar_repair_summary = (
    n11_calendar_repair_candidate_ranking
    .loc[n11_calendar_repair_candidate_ranking["repair_method"] == N11_SELECTED_CALENDAR_REPAIR_METHOD]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Calendar repair gate
# ------------------------------------------------------------

selected_full_calendar_pass = bool(
    n11_selected_calendar_repair_summary["calendar_pass_common_full_support"].iloc[0]
)

selected_inner_calendar_pass = bool(
    n11_selected_calendar_repair_summary["calendar_pass_common_inner_95pct_support"].iloc[0]
)

selected_contract_rows = int(len(N11_CALENDAR_REPAIRED_CONTRACT_POINTS))
selected_contract_eligible_rows = int(
    N11_CALENDAR_REPAIRED_CONTRACT_POINTS["eligible_for_n12_after_calendar_repair"].sum()
)

selected_contract_expiries = int(N11_CALENDAR_REPAIRED_CONTRACT_POINTS["expiry"].nunique())

selected_full_static_proxy_violations = int(
    n11_selected_calendar_repair_summary[
        "total_static_proxy_violations_common_full_support"
    ].iloc[0]
)

selected_inner_static_proxy_violations = int(
    n11_selected_calendar_repair_summary[
        "total_static_proxy_violations_common_inner_95pct_support"
    ].iloc[0]
)

selected_repair_fail_rows = int(
    n11_selected_calendar_repair_summary["repair_fail_rows"].iloc[0]
)

n11_calendar_repair_gate = pd.DataFrame(
    [
        {
            "check_name": "calendar_repair_candidates_available",
            "check_type": "blocking",
            "passes": len(n11_calendar_repair_candidate_ranking) >= 2,
            "observed_value": float(len(n11_calendar_repair_candidate_ranking)),
            "threshold_or_requirement": ">= 2",
        },
        {
            "check_name": "selected_calendar_repair_full_grid_pass",
            "check_type": "blocking",
            "passes": selected_full_calendar_pass,
            "observed_value": float(selected_full_calendar_pass),
            "threshold_or_requirement": "True",
        },
        {
            "check_name": "selected_calendar_repair_inner_grid_pass",
            "check_type": "blocking",
            "passes": selected_inner_calendar_pass,
            "observed_value": float(selected_inner_calendar_pass),
            "threshold_or_requirement": "True",
        },
        {
            "check_name": "selected_calendar_repaired_grid_available",
            "check_type": "blocking",
            "passes": len(N11_CALENDAR_REPAIRED_GRID) > 0,
            "observed_value": float(len(N11_CALENDAR_REPAIRED_GRID)),
            "threshold_or_requirement": "> 0",
        },
        {
            "check_name": "selected_calendar_repaired_contract_rows_available",
            "check_type": "blocking",
            "passes": selected_contract_rows > 0,
            "observed_value": float(selected_contract_rows),
            "threshold_or_requirement": "> 0",
        },
        {
            "check_name": "selected_calendar_repaired_contract_expiries",
            "check_type": "blocking",
            "passes": selected_contract_expiries >= CONFIG.min_primary_expiries,
            "observed_value": float(selected_contract_expiries),
            "threshold_or_requirement": f">= {CONFIG.min_primary_expiries}",
        },
        {
            "check_name": "selected_calendar_repaired_eligible_rows",
            "check_type": "blocking",
            "passes": selected_contract_eligible_rows > 0,
            "observed_value": float(selected_contract_eligible_rows),
            "threshold_or_requirement": "> 0",
        },
        {
            "check_name": "selected_calendar_repair_fail_rows",
            "check_type": "warning",
            "passes": selected_repair_fail_rows == 0,
            "observed_value": float(selected_repair_fail_rows),
            "threshold_or_requirement": "preferred 0",
        },
        {
            "check_name": "selected_full_grid_static_proxy_violations",
            "check_type": "warning",
            "passes": selected_full_static_proxy_violations == 0,
            "observed_value": float(selected_full_static_proxy_violations),
            "threshold_or_requirement": "preferred 0; next cell can handle if > 0",
        },
        {
            "check_name": "selected_inner_grid_static_proxy_violations",
            "check_type": "warning",
            "passes": selected_inner_static_proxy_violations == 0,
            "observed_value": float(selected_inner_static_proxy_violations),
            "threshold_or_requirement": "preferred 0; next cell can handle if > 0",
        },
    ]
)

N11_CALENDAR_REPAIR_PASS = bool(
    n11_calendar_repair_gate
    .query("check_type == 'blocking'")["passes"]
    .all()
)


# ------------------------------------------------------------
# Persist outputs
# ------------------------------------------------------------

_write_table_pair(
    n11_calendar_repair_grid_candidates,
    "n11_calendar_repair_grid_candidates",
    N11_GRID_DIR,
)

_write_table_pair(
    n11_calendar_repair_detail,
    "n11_calendar_repair_detail",
    N11_DIAGNOSTIC_DIR,
)

_write_table_pair(
    n11_calendar_repair_summary,
    "n11_calendar_repair_summary",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_calendar_repair_static_proxy_detail,
    "n11_calendar_repair_static_proxy_detail",
    N11_DIAGNOSTIC_DIR,
)

_write_table_pair(
    n11_calendar_repair_static_proxy_summary,
    "n11_calendar_repair_static_proxy_summary",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_calendar_repair_static_proxy_by_method,
    "n11_calendar_repair_static_proxy_by_method",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_calendar_repair_grid_bias_summary,
    "n11_calendar_repair_grid_bias_summary",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_calendar_repaired_contract_candidates,
    "n11_calendar_repaired_contract_candidates",
    N11_HANDOFF_DIR,
)

_write_table_pair(
    n11_calendar_repaired_contract_bias_summary,
    "n11_calendar_repaired_contract_bias_summary",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_calendar_repair_candidate_ranking,
    "n11_calendar_repair_candidate_ranking",
    N11_TABLE_DIR,
)

_write_table_pair(
    N11_CALENDAR_REPAIRED_GRID,
    "n11_selected_calendar_repaired_grid",
    N11_GRID_DIR,
)

_write_table_pair(
    N11_CALENDAR_REPAIRED_CONTRACT_POINTS,
    "n11_selected_calendar_repaired_contract_points",
    N11_HANDOFF_DIR,
)

_write_table_pair(
    n11_selected_calendar_repair_summary,
    "n11_selected_calendar_repair_summary",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_calendar_repair_gate,
    "n11_calendar_repair_gate",
    N11_TABLE_DIR,
)


# ------------------------------------------------------------
# Console summary
# ------------------------------------------------------------

print("Notebook 11 calendar repair layer complete.")
print(f"Calendar repair pass: {N11_CALENDAR_REPAIR_PASS}")
print(f"Selected pre-repair mode: {N11_PRE_REPAIR_SELECTED_MODE}")
print(f"Selected calendar repair method: {N11_SELECTED_CALENDAR_REPAIR_METHOD}")
print(f"Selected repaired grid rows: {len(N11_CALENDAR_REPAIRED_GRID):,}")
print(f"Selected repaired contract rows: {selected_contract_rows:,}")
print(f"Selected repaired eligible rows: {selected_contract_eligible_rows:,}")
print(f"Selected contract expiries: {selected_contract_expiries:,}")
print(f"Selected full-grid static proxy violations: {selected_full_static_proxy_violations:,}")
print(f"Selected inner-grid static proxy violations: {selected_inner_static_proxy_violations:,}")

display(n11_calendar_repair_summary)
display(n11_calendar_repair_static_proxy_by_method)
display(n11_calendar_repair_grid_bias_summary)
display(n11_calendar_repaired_contract_bias_summary)
display(n11_calendar_repair_candidate_ranking)
display(n11_selected_calendar_repair_summary)
display(n11_calendar_repair_gate)

if not N11_CALENDAR_REPAIR_PASS:
    raise RuntimeError(
        "Notebook 11 calendar repair gate failed. "
        "Review n11_calendar_repair_gate before proceeding."
    )

Notebook 11 calendar repair layer complete.
Calendar repair pass: True
Selected pre-repair mode: weighted_broad_svi
Selected calendar repair method: calendar_pava
Selected repaired grid rows: 2,896
Selected repaired contract rows: 1,260
Selected repaired eligible rows: 1,258
Selected contract expiries: 8
Selected full-grid static proxy violations: 231
Selected inner-grid static proxy violations: 9


,grid_name,mode_name,expiry_pairs,k_points,calendar_checks,calendar_violation_count,calendar_violation_rate,max_calendar_violation_amount,min_calendar_diff,calendar_pass
0,common_full_support,calendar_cummax,7,181,1267,0,0.0000000000,0.0000000000,0.0000000000,True
1,common_full_support,calendar_pava,7,181,1267,0,0.0000000000,0.0000000000,0.0000000000,True
2,common_inner_95pct_support,calendar_cummax,7,181,1267,0,0.0000000000,0.0000000000,0.0000000000,True
3,common_inner_95pct_support,calendar_pava,7,181,1267,0,0.0000000000,0.0000000000,0.0000000000,True


,repair_method,grid_name,expiries,total_monotonicity_violations,total_convexity_violations,total_static_proxy_violations,static_proxy_passes,min_slope_diff
0,calendar_cummax,common_full_support,8,39,152,191,0,-0.0006987400
1,calendar_cummax,common_inner_95pct_support,8,0,0,0,8,0.0000000000
2,calendar_pava,common_full_support,8,39,192,231,0,-0.0008912708
3,calendar_pava,common_inner_95pct_support,8,0,9,9,5,-0.0012890737


,repair_method,grid_name,rows,expiries,k_points,mean_total_variance_repair,median_total_variance_repair,median_abs_total_variance_repair,p95_abs_total_variance_repair,max_abs_total_variance_repair,mean_iv_repair,median_iv_repair,median_abs_iv_repair,p95_abs_iv_repair,max_abs_iv_repair
0,calendar_cummax,common_full_support,1448,8,181,0.0092120478,0.0000000000,0.0000000000,0.0603005128,0.0917503199,0.0814063344,0.0000000000,0.0000000000,0.4720085888,1.0136061810
1,calendar_cummax,common_inner_95pct_support,1448,8,181,0.0008929477,0.0000000000,0.0000000000,0.0074730964,0.0197909936,0.0134753254,0.0000000000,0.0000000000,0.1032886138,0.3543225500
2,calendar_pava,common_full_support,1448,8,181,-0.0001931515,0.0000000000,0.0000000000,0.0127802874,0.0716919933,-0.0095076707,0.0000000000,0.0000000000,0.2093596310,1.1067021502
3,calendar_pava,common_inner_95pct_support,1448,8,181,-0.0000302675,0.0000000000,0.0000000000,0.0021991032,0.0137933803,-0.0022934416,0.0000000000,0.0000000000,0.0523402167,0.3565051763


,repair_method,band,rows,expiries,eligible_rows,repair_warn_rows,repair_fail_rows,median_abs_iv_repair,p95_abs_iv_repair,max_abs_iv_repair,median_abs_post_repair_iv_residual_to_market,p95_abs_post_repair_iv_residual_to_market,max_abs_post_repair_iv_residual_to_market,median_abs_post_repair_price_residual_to_market,p95_abs_post_repair_price_residual_to_market,max_abs_post_repair_price_residual_to_market
0,calendar_cummax,all,1260,8,1230,49,30,0.0000098961,0.0000770794,0.3519900244,0.0006436515,0.0169336280,0.3520054739,0.0250476732,0.1300076295,4.2481009049
1,calendar_cummax,inner_abs_k_le_015,1026,8,1026,0,0,0.0000108972,0.0000457534,0.0000733843,0.0006154190,0.0038220523,0.0386580398,0.0272849355,0.1025110385,0.1556858773
2,calendar_cummax,atm_abs_k_le_005,522,8,522,0,0,0.0000135684,0.0000573492,0.0000733843,0.0007062031,0.0024652296,0.0192743291,0.0431909521,0.1246054990,0.1556858773
3,calendar_pava,all,1260,8,1258,7,2,0.0000088733,0.0000553029,0.0939952676,0.0006301606,0.0048042935,0.0873594289,0.0229208380,0.0976708265,0.1556858773
4,calendar_pava,inner_abs_k_le_015,1026,8,1026,0,0,0.0000108972,0.0000457534,0.0000733843,0.0006154190,0.0038220523,0.0386580398,0.0272849355,0.1025110385,0.1556858773
5,calendar_pava,atm_abs_k_le_005,522,8,522,0,0,0.0000135684,0.0000573492,0.0000733843,0.0007062031,0.0024652296,0.0192743291,0.0431909521,0.1246054990,0.1556858773


,repair_method,calendar_pass_common_full_support,calendar_pass_common_inner_95pct_support,calendar_violation_count_common_full_support,calendar_violation_count_common_inner_95pct_support,calendar_violation_rate_common_full_support,calendar_violation_rate_common_inner_95pct_support,max_calendar_violation_amount_common_full_support,max_calendar_violation_amount_common_inner_95pct_support,min_slope_diff_common_full_support,min_slope_diff_common_inner_95pct_support,static_proxy_passes_common_full_support,static_proxy_passes_common_inner_95pct_support,total_convexity_violations_common_full_support,total_convexity_violations_common_inner_95pct_support,total_monotonicity_violations_common_full_support,total_monotonicity_violations_common_inner_95pct_support,total_static_proxy_violations_common_full_support,total_static_proxy_violations_common_inner_95pct_support,rows,expiries,eligible_rows,repair_warn_rows,repair_fail_rows,median_abs_iv_repair,p95_abs_iv_repair,max_abs_iv_repair,median_abs_post_repair_iv_residual_to_market,p95_abs_post_repair_iv_residual_to_market,max_abs_post_repair_iv_residual_to_market,median_abs_post_repair_price_residual_to_market,p95_abs_post_repair_price_residual_to_market,max_abs_post_repair_price_residual_to_market,atm_rows,atm_expiries,atm_eligible_rows,atm_repair_warn_rows,atm_repair_fail_rows,atm_median_abs_iv_repair,atm_p95_abs_iv_repair,atm_max_abs_iv_repair,atm_median_abs_post_repair_iv_residual_to_market,atm_p95_abs_post_repair_iv_residual_to_market,atm_max_abs_post_repair_iv_residual_to_market,atm_median_abs_post_repair_price_residual_to_market,atm_p95_abs_post_repair_price_residual_to_market,atm_max_abs_post_repair_price_residual_to_market,calendar_repair_selection_score
0,calendar_pava,True,True,0,0,0.0000000000,0.0000000000,0.0000000000,0.0000000000,-0.0008912708,-0.0012890737,0,5,192,9,39,0,231,9,1260,8,1258,7,2,0.0000088733,0.0000553029,0.0939952676,0.0006301606,0.0048042935,0.0873594289,0.0229208380,0.0976708265,0.1556858773,522,8,522,0,0,0.0000135684,0.0000573492,0.0000733843,0.0007062031,0.0024652296,0.0192743291,0.0431909521,0.1246054990,0.1556858773,"12,011.0448362183"
1,calendar_cummax,True,True,0,0,0.0000000000,0.0000000000,0.0000000000,0.0000000000,-0.0006987400,0.0000000000,0,8,152,0,39,0,191,0,1260,8,1230,49,30,0.0000098961,0.0000770794,0.3519900244,0.0006436515,0.0169336280,0.3520054739,0.0250476732,0.1300076295,4.2481009049,522,8,522,0,0,0.0000135684,0.0000573492,0.0000733843,0.0007062031,0.0024652296,0.0192743291,0.0431909521,0.1246054990,0.1556858773,"12,798.4924795981"


,repair_method,calendar_pass_common_full_support,calendar_pass_common_inner_95pct_support,calendar_violation_count_common_full_support,calendar_violation_count_common_inner_95pct_support,calendar_violation_rate_common_full_support,calendar_violation_rate_common_inner_95pct_support,max_calendar_violation_amount_common_full_support,max_calendar_violation_amount_common_inner_95pct_support,min_slope_diff_common_full_support,min_slope_diff_common_inner_95pct_support,static_proxy_passes_common_full_support,static_proxy_passes_common_inner_95pct_support,total_convexity_violations_common_full_support,total_convexity_violations_common_inner_95pct_support,total_monotonicity_violations_common_full_support,total_monotonicity_violations_common_inner_95pct_support,total_static_proxy_violations_common_full_support,total_static_proxy_violations_common_inner_95pct_support,rows,expiries,eligible_rows,repair_warn_rows,repair_fail_rows,median_abs_iv_repair,p95_abs_iv_repair,max_abs_iv_repair,median_abs_post_repair_iv_residual_to_market,p95_abs_post_repair_iv_residual_to_market,max_abs_post_repair_iv_residual_to_market,median_abs_post_repair_price_residual_to_market,p95_abs_post_repair_price_residual_to_market,max_abs_post_repair_price_residual_to_market,atm_rows,atm_expiries,atm_eligible_rows,atm_repair_warn_rows,atm_repair_fail_rows,atm_median_abs_iv_repair,atm_p95_abs_iv_repair,atm_max_abs_iv_repair,atm_median_abs_post_repair_iv_residual_to_market,atm_p95_abs_post_repair_iv_residual_to_market,atm_max_abs_post_repair_iv_residual_to_market,atm_median_abs_post_repair_price_residual_to_market,atm_p95_abs_post_repair_price_residual_to_market,atm_max_abs_post_repair_price_residual_to_market,calendar_repair_selection_score
0,calendar_pava,True,True,0,0,0.0000000000,0.0000000000,0.0000000000,0.0000000000,-0.0008912708,-0.0012890737,0,5,192,9,39,0,231,9,1260,8,1258,7,2,0.0000088733,0.0000553029,0.0939952676,0.0006301606,0.0048042935,0.0873594289,0.0229208380,0.0976708265,0.1556858773,522,8,522,0,0,0.0000135684,0.0000573492,0.0000733843,0.0007062031,0.0024652296,0.0192743291,0.0431909521,0.1246054990,0.1556858773,"12,011.0448362183"


,check_name,check_type,passes,observed_value,threshold_or_requirement
0,calendar_repair_candidates_available,blocking,True,2.0000000000,>= 2
1,selected_calendar_repair_full_grid_pass,blocking,True,1.0000000000,True
2,selected_calendar_repair_inner_grid_pass,blocking,True,1.0000000000,True
3,selected_calendar_repaired_grid_available,blocking,True,"2,896.0000000000",> 0
4,selected_calendar_repaired_contract_rows_avail...,blocking,True,"1,260.0000000000",> 0
5,selected_calendar_repaired_contract_expiries,blocking,True,8.0000000000,>= 4
6,selected_calendar_repaired_eligible_rows,blocking,True,"1,258.0000000000",> 0
7,selected_calendar_repair_fail_rows,warning,False,2.0000000000,preferred 0
8,selected_full_grid_static_proxy_violations,warning,False,231.0000000000,preferred 0; next cell can handle if > 0
9,selected_inner_grid_static_proxy_violations,warning,False,9.0000000000,preferred 0; next cell can handle if > 0


In [9]:
# ------------------------------------------------------------
# Notebook 11 cell 09: Static-proxy localization and N12 eligibility filtering
# ------------------------------------------------------------
# Calendar repair removed maturity monotonicity violations, but the repaired
# grid still has local static-proxy warnings, mostly in the wings.
#
# This cell localizes those warnings back to contract-level rows and builds
# two downstream candidates:
#
#   1. broad repaired candidate
#      Calendar-repaired contracts with warnings retained and explicitly flagged.
#
#   2. clean repaired candidate
#      Calendar-repaired contracts after excluding repair-fail rows and rows
#      touched by full-grid static-proxy warning intervals.
#
# This cell does not claim the full repaired surface is globally arbitrage-free.
# It prepares an auditable N12 handoff candidate with eligibility flags.

from __future__ import annotations

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Pre-flight checks
# ------------------------------------------------------------

required_globals_for_static_localization = [
    "N11_CALENDAR_REPAIR_PASS",
    "N11_SELECTED_CALENDAR_REPAIR_METHOD",
    "N11_CALENDAR_REPAIRED_CONTRACT_POINTS",
    "N11_CALENDAR_REPAIRED_GRID",
    "n11_calendar_repair_static_proxy_detail",
    "n11_calendar_repair_static_proxy_summary",
    "n11_selected_calendar_repair_summary",
    "CONFIG",
]

missing_static_localization_globals = [
    name for name in required_globals_for_static_localization if name not in globals()
]

if missing_static_localization_globals:
    raise NameError(
        f"Missing required objects before static-proxy localization: {missing_static_localization_globals}"
    )

if not bool(N11_CALENDAR_REPAIR_PASS):
    raise RuntimeError("Calendar repair did not pass. Run/fix the previous cell first.")


# ------------------------------------------------------------
# Extract selected static-proxy violation intervals
# ------------------------------------------------------------

selected_static_proxy_violations = (
    n11_calendar_repair_static_proxy_detail
    .loc[
        (n11_calendar_repair_static_proxy_detail["repair_method"] == N11_SELECTED_CALENDAR_REPAIR_METHOD)
        & (n11_calendar_repair_static_proxy_detail["violation"].astype(bool))
    ]
    .copy()
    .reset_index(drop=True)
)

selected_static_proxy_violations["k_interval_low"] = selected_static_proxy_violations[
    ["k_left", "k_right"]
].min(axis=1)

selected_static_proxy_violations["k_interval_high"] = selected_static_proxy_violations[
    ["k_left", "k_right"]
].max(axis=1)

selected_static_proxy_violations["abs_k_interval_mid"] = (
    0.5
    * (
        selected_static_proxy_violations["k_interval_low"].abs()
        + selected_static_proxy_violations["k_interval_high"].abs()
    )
)

selected_static_proxy_violations["wing_zone"] = np.select(
    [
        selected_static_proxy_violations["abs_k_interval_mid"] <= 0.0500,
        selected_static_proxy_violations["abs_k_interval_mid"] <= 0.1500,
        selected_static_proxy_violations["abs_k_interval_mid"] <= 0.3000,
    ],
    [
        "atm_abs_k_le_005",
        "inner_abs_k_le_015",
        "mid_abs_k_le_030",
    ],
    default="outer_abs_k_gt_030",
)

selected_static_proxy_violation_summary = (
    selected_static_proxy_violations
    .groupby(["grid_name", "diagnostic_type", "wing_zone"], dropna=False)
    .agg(
        violation_intervals=("diagnostic_value", "size"),
        expiries=("expiry", "nunique"),
        min_k=("k_interval_low", "min"),
        max_k=("k_interval_high", "max"),
        min_diagnostic_value=("diagnostic_value", "min"),
        max_diagnostic_value=("diagnostic_value", "max"),
    )
    .reset_index()
    if len(selected_static_proxy_violations)
    else pd.DataFrame(
        columns=[
            "grid_name",
            "diagnostic_type",
            "wing_zone",
            "violation_intervals",
            "expiries",
            "min_k",
            "max_k",
            "min_diagnostic_value",
            "max_diagnostic_value",
        ]
    )
)


# ------------------------------------------------------------
# Contract-level localization
# ------------------------------------------------------------

def flag_contracts_touched_by_static_proxy_intervals(
    contracts: pd.DataFrame,
    violations: pd.DataFrame,
    *,
    grid_name: str,
    flag_prefix: str,
) -> pd.DataFrame:
    out = contracts.copy()

    out[f"{flag_prefix}_static_proxy_warning"] = False
    out[f"{flag_prefix}_monotonicity_warning"] = False
    out[f"{flag_prefix}_convexity_warning"] = False
    out[f"{flag_prefix}_static_proxy_warning_count"] = 0

    v = violations.loc[violations["grid_name"] == grid_name].copy()

    if len(v) == 0:
        return out

    for _, interval in v.iterrows():
        expiry = interval["expiry"]
        k_low = float(interval["k_interval_low"])
        k_high = float(interval["k_interval_high"])
        diagnostic_type = str(interval["diagnostic_type"])

        touched = (
            (out["expiry"] == expiry)
            & (pd.to_numeric(out["k"], errors="coerce") >= k_low)
            & (pd.to_numeric(out["k"], errors="coerce") <= k_high)
        )

        if not touched.any():
            continue

        out.loc[touched, f"{flag_prefix}_static_proxy_warning"] = True
        out.loc[touched, f"{flag_prefix}_static_proxy_warning_count"] += 1

        if diagnostic_type == "monotonicity":
            out.loc[touched, f"{flag_prefix}_monotonicity_warning"] = True

        if diagnostic_type == "convexity":
            out.loc[touched, f"{flag_prefix}_convexity_warning"] = True

    return out


n11_static_localized_contracts = N11_CALENDAR_REPAIRED_CONTRACT_POINTS.copy()

n11_static_localized_contracts = flag_contracts_touched_by_static_proxy_intervals(
    n11_static_localized_contracts,
    selected_static_proxy_violations,
    grid_name="common_full_support",
    flag_prefix="full_grid",
)

n11_static_localized_contracts = flag_contracts_touched_by_static_proxy_intervals(
    n11_static_localized_contracts,
    selected_static_proxy_violations,
    grid_name="common_inner_95pct_support",
    flag_prefix="inner_grid",
)

n11_static_localized_contracts["abs_k"] = pd.to_numeric(
    n11_static_localized_contracts["k"],
    errors="coerce",
).abs()

n11_static_localized_contracts["moneyness_region_n11"] = np.select(
    [
        n11_static_localized_contracts["abs_k"] <= 0.0500,
        n11_static_localized_contracts["abs_k"] <= 0.1500,
        n11_static_localized_contracts["abs_k"] <= 0.3000,
    ],
    [
        "atm_abs_k_le_005",
        "inner_abs_k_le_015",
        "mid_abs_k_le_030",
    ],
    default="outer_abs_k_gt_030",
)

# Broad candidate keeps warnings but removes hard repair failures.
n11_static_localized_contracts["n12_broad_repaired_candidate"] = (
    n11_static_localized_contracts["eligible_for_n12_after_calendar_repair"].astype(bool)
)

# Clean candidate removes rows touched by any full-grid static-proxy warning.
# This is intentionally stricter than inner-grid-only filtering.
n11_static_localized_contracts["n12_clean_repaired_candidate"] = (
    n11_static_localized_contracts["eligible_for_n12_after_calendar_repair"].astype(bool)
    & ~n11_static_localized_contracts["repair_bias_fail"].astype(bool)
    & ~n11_static_localized_contracts["full_grid_static_proxy_warning"].astype(bool)
)

# Anchor candidate keeps only inner/ATM rows with no inner-grid warnings.
n11_static_localized_contracts["n12_anchor_repaired_candidate"] = (
    n11_static_localized_contracts["eligible_for_n12_after_calendar_repair"].astype(bool)
    & ~n11_static_localized_contracts["repair_bias_fail"].astype(bool)
    & ~n11_static_localized_contracts["inner_grid_static_proxy_warning"].astype(bool)
    & (n11_static_localized_contracts["abs_k"] <= 0.1500)
)

# Final candidate used for N12 by default.
# Prefer the clean candidate if it preserves all expiries and enough rows.
clean_candidate_expiries = int(
    n11_static_localized_contracts
    .loc[n11_static_localized_contracts["n12_clean_repaired_candidate"], "expiry"]
    .nunique()
)

clean_candidate_rows = int(
    n11_static_localized_contracts["n12_clean_repaired_candidate"].sum()
)

if (
    clean_candidate_expiries >= CONFIG.min_primary_expiries
    and clean_candidate_rows >= max(100, CONFIG.min_primary_expiries * CONFIG.min_rows_per_svi_slice)
):
    N11_SELECTED_N12_CANDIDATE_FLAG = "n12_clean_repaired_candidate"
else:
    N11_SELECTED_N12_CANDIDATE_FLAG = "n12_broad_repaired_candidate"

n11_static_localized_contracts["n12_selected_repaired_candidate"] = (
    n11_static_localized_contracts[N11_SELECTED_N12_CANDIDATE_FLAG].astype(bool)
)


# ------------------------------------------------------------
# Candidate summaries
# ------------------------------------------------------------

candidate_flags = [
    "n12_broad_repaired_candidate",
    "n12_clean_repaired_candidate",
    "n12_anchor_repaired_candidate",
    "n12_selected_repaired_candidate",
]

candidate_summary_rows = []

for flag in candidate_flags:
    sub = n11_static_localized_contracts.loc[n11_static_localized_contracts[flag]].copy()

    if len(sub) == 0:
        candidate_summary_rows.append(
            {
                "candidate_flag": flag,
                "rows": 0,
                "expiries": 0,
                "unique_strikes": 0,
                "min_tau": np.nan,
                "max_tau": np.nan,
                "min_k": np.nan,
                "max_k": np.nan,
                "k_width": np.nan,
                "repair_warn_rows": 0,
                "repair_fail_rows": 0,
                "full_grid_static_proxy_warning_rows": 0,
                "inner_grid_static_proxy_warning_rows": 0,
                "median_abs_iv_repair": np.nan,
                "p95_abs_iv_repair": np.nan,
                "max_abs_iv_repair": np.nan,
                "median_abs_post_repair_iv_residual_to_market": np.nan,
                "p95_abs_post_repair_iv_residual_to_market": np.nan,
                "max_abs_post_repair_iv_residual_to_market": np.nan,
                "median_abs_post_repair_price_residual_to_market": np.nan,
                "p95_abs_post_repair_price_residual_to_market": np.nan,
                "max_abs_post_repair_price_residual_to_market": np.nan,
            }
        )
        continue

    candidate_summary_rows.append(
        {
            "candidate_flag": flag,
            "rows": int(len(sub)),
            "expiries": int(sub["expiry"].nunique()),
            "unique_strikes": int(sub["strike"].nunique()),
            "min_tau": float(pd.to_numeric(sub["tau_years"], errors="coerce").min()),
            "max_tau": float(pd.to_numeric(sub["tau_years"], errors="coerce").max()),
            "min_k": float(pd.to_numeric(sub["k"], errors="coerce").min()),
            "max_k": float(pd.to_numeric(sub["k"], errors="coerce").max()),
            "k_width": float(pd.to_numeric(sub["k"], errors="coerce").max() - pd.to_numeric(sub["k"], errors="coerce").min()),
            "repair_warn_rows": int(sub["repair_bias_warn"].sum()),
            "repair_fail_rows": int(sub["repair_bias_fail"].sum()),
            "full_grid_static_proxy_warning_rows": int(sub["full_grid_static_proxy_warning"].sum()),
            "inner_grid_static_proxy_warning_rows": int(sub["inner_grid_static_proxy_warning"].sum()),
            "median_abs_iv_repair": float(np.nanmedian(sub["abs_iv_repair"])),
            "p95_abs_iv_repair": float(np.nanpercentile(sub["abs_iv_repair"], 95)),
            "max_abs_iv_repair": float(np.nanmax(sub["abs_iv_repair"])),
            "median_abs_post_repair_iv_residual_to_market": float(
                np.nanmedian(sub["abs_post_repair_iv_residual_to_market"])
            ),
            "p95_abs_post_repair_iv_residual_to_market": float(
                np.nanpercentile(sub["abs_post_repair_iv_residual_to_market"], 95)
            ),
            "max_abs_post_repair_iv_residual_to_market": float(
                np.nanmax(sub["abs_post_repair_iv_residual_to_market"])
            ),
            "median_abs_post_repair_price_residual_to_market": float(
                np.nanmedian(sub["abs_post_repair_price_residual_to_market"])
            ),
            "p95_abs_post_repair_price_residual_to_market": float(
                np.nanpercentile(sub["abs_post_repair_price_residual_to_market"], 95)
            ),
            "max_abs_post_repair_price_residual_to_market": float(
                np.nanmax(sub["abs_post_repair_price_residual_to_market"])
            ),
        }
    )

n11_repaired_candidate_summary = pd.DataFrame(candidate_summary_rows)


n11_repaired_candidate_by_expiry = (
    n11_static_localized_contracts
    .groupby("expiry", dropna=False)
    .agg(
        total_rows=("contract_key", "size"),
        broad_rows=("n12_broad_repaired_candidate", "sum"),
        clean_rows=("n12_clean_repaired_candidate", "sum"),
        anchor_rows=("n12_anchor_repaired_candidate", "sum"),
        selected_rows=("n12_selected_repaired_candidate", "sum"),
        full_grid_static_proxy_warning_rows=("full_grid_static_proxy_warning", "sum"),
        inner_grid_static_proxy_warning_rows=("inner_grid_static_proxy_warning", "sum"),
        repair_warn_rows=("repair_bias_warn", "sum"),
        repair_fail_rows=("repair_bias_fail", "sum"),
        min_k=("k", "min"),
        max_k=("k", "max"),
        median_repaired_iv=("repaired_iv", "median"),
        median_abs_iv_repair=("abs_iv_repair", "median"),
        max_abs_iv_repair=("abs_iv_repair", "max"),
    )
    .reset_index()
)

n11_repaired_candidate_by_region = (
    n11_static_localized_contracts
    .groupby("moneyness_region_n11", dropna=False)
    .agg(
        total_rows=("contract_key", "size"),
        broad_rows=("n12_broad_repaired_candidate", "sum"),
        clean_rows=("n12_clean_repaired_candidate", "sum"),
        anchor_rows=("n12_anchor_repaired_candidate", "sum"),
        selected_rows=("n12_selected_repaired_candidate", "sum"),
        full_grid_static_proxy_warning_rows=("full_grid_static_proxy_warning", "sum"),
        inner_grid_static_proxy_warning_rows=("inner_grid_static_proxy_warning", "sum"),
        repair_warn_rows=("repair_bias_warn", "sum"),
        repair_fail_rows=("repair_bias_fail", "sum"),
        median_abs_iv_repair=("abs_iv_repair", "median"),
        p95_abs_iv_repair=("abs_iv_repair", lambda s: float(np.nanpercentile(s, 95))),
        max_abs_iv_repair=("abs_iv_repair", "max"),
    )
    .reset_index()
)


# ------------------------------------------------------------
# Build selected N12 handoff candidate
# ------------------------------------------------------------

N11_REPAIRED_N12_HANDOFF_CANDIDATE = (
    n11_static_localized_contracts
    .loc[n11_static_localized_contracts["n12_selected_repaired_candidate"]]
    .copy()
    .sort_values(["expiry", "k", "strike"])
    .reset_index(drop=True)
)

N11_REPAIRED_N12_HANDOFF_CANDIDATE["n11_selected_calendar_repair_method"] = (
    N11_SELECTED_CALENDAR_REPAIR_METHOD
)
N11_REPAIRED_N12_HANDOFF_CANDIDATE["n11_selected_pre_repair_mode"] = (
    N11_PRE_REPAIR_SELECTED_MODE
)
N11_REPAIRED_N12_HANDOFF_CANDIDATE["n11_selected_candidate_flag"] = (
    N11_SELECTED_N12_CANDIDATE_FLAG
)

# Keep a compact downstream column order, preserving any remaining columns after.
preferred_handoff_cols = [
    "contract_key",
    "expiry",
    "tau_years",
    "option_type",
    "strike",
    "forward",
    "discount_factor",
    "k",
    "abs_k",
    "moneyness_region_n11",
    "market_price",
    "market_total_variance",
    "market_iv",
    "pre_repair_total_variance",
    "pre_repair_iv",
    "pre_repair_bsm_price",
    "repaired_total_variance",
    "repaired_iv",
    "repaired_bsm_price",
    "total_variance_repair",
    "abs_total_variance_repair",
    "iv_repair",
    "abs_iv_repair",
    "post_repair_total_variance_residual_to_market",
    "post_repair_iv_residual_to_market",
    "post_repair_price_residual_to_market",
    "abs_post_repair_iv_residual_to_market",
    "abs_post_repair_price_residual_to_market",
    "fit_weight",
    "normalized_weight",
    "repair_bias_warn",
    "repair_bias_fail",
    "full_grid_static_proxy_warning",
    "full_grid_monotonicity_warning",
    "full_grid_convexity_warning",
    "inner_grid_static_proxy_warning",
    "inner_grid_monotonicity_warning",
    "inner_grid_convexity_warning",
    "n12_broad_repaired_candidate",
    "n12_clean_repaired_candidate",
    "n12_anchor_repaired_candidate",
    "n12_selected_repaired_candidate",
    "n11_selected_pre_repair_mode",
    "n11_selected_calendar_repair_method",
    "n11_selected_candidate_flag",
]

existing_preferred_cols = [
    col for col in preferred_handoff_cols
    if col in N11_REPAIRED_N12_HANDOFF_CANDIDATE.columns
]

remaining_cols = [
    col for col in N11_REPAIRED_N12_HANDOFF_CANDIDATE.columns
    if col not in existing_preferred_cols
]

N11_REPAIRED_N12_HANDOFF_CANDIDATE = N11_REPAIRED_N12_HANDOFF_CANDIDATE[
    existing_preferred_cols + remaining_cols
].copy()


# ------------------------------------------------------------
# Final gate for this layer
# ------------------------------------------------------------

selected_candidate_rows = int(len(N11_REPAIRED_N12_HANDOFF_CANDIDATE))
selected_candidate_expiries = int(N11_REPAIRED_N12_HANDOFF_CANDIDATE["expiry"].nunique())
selected_candidate_repair_fail_rows = int(N11_REPAIRED_N12_HANDOFF_CANDIDATE["repair_bias_fail"].sum())
selected_candidate_full_static_warning_rows = int(
    N11_REPAIRED_N12_HANDOFF_CANDIDATE["full_grid_static_proxy_warning"].sum()
)
selected_candidate_inner_static_warning_rows = int(
    N11_REPAIRED_N12_HANDOFF_CANDIDATE["inner_grid_static_proxy_warning"].sum()
)

selected_candidate_positive_variance_rows = int(
    (
        np.isfinite(N11_REPAIRED_N12_HANDOFF_CANDIDATE["repaired_total_variance"])
        & (N11_REPAIRED_N12_HANDOFF_CANDIDATE["repaired_total_variance"] > CONFIG.positive_total_variance_floor)
    ).sum()
)

selected_candidate_positive_iv_rows = int(
    (
        np.isfinite(N11_REPAIRED_N12_HANDOFF_CANDIDATE["repaired_iv"])
        & (N11_REPAIRED_N12_HANDOFF_CANDIDATE["repaired_iv"] > 0.0)
    ).sum()
)

n11_static_localization_gate = pd.DataFrame(
    [
        {
            "check_name": "static_proxy_violations_loaded",
            "check_type": "blocking",
            "passes": selected_static_proxy_violations is not None,
            "observed_value": float(len(selected_static_proxy_violations)),
            "threshold_or_requirement": "loaded; count may be zero or positive",
        },
        {
            "check_name": "localized_contract_table_available",
            "check_type": "blocking",
            "passes": len(n11_static_localized_contracts) > 0,
            "observed_value": float(len(n11_static_localized_contracts)),
            "threshold_or_requirement": "> 0",
        },
        {
            "check_name": "selected_n12_candidate_rows",
            "check_type": "blocking",
            "passes": selected_candidate_rows > 0,
            "observed_value": float(selected_candidate_rows),
            "threshold_or_requirement": "> 0",
        },
        {
            "check_name": "selected_n12_candidate_expiries",
            "check_type": "blocking",
            "passes": selected_candidate_expiries >= CONFIG.min_primary_expiries,
            "observed_value": float(selected_candidate_expiries),
            "threshold_or_requirement": f">= {CONFIG.min_primary_expiries}",
        },
        {
            "check_name": "selected_n12_candidate_positive_variance_rows",
            "check_type": "blocking",
            "passes": selected_candidate_positive_variance_rows == selected_candidate_rows,
            "observed_value": float(selected_candidate_positive_variance_rows),
            "threshold_or_requirement": "all selected rows",
        },
        {
            "check_name": "selected_n12_candidate_positive_iv_rows",
            "check_type": "blocking",
            "passes": selected_candidate_positive_iv_rows == selected_candidate_rows,
            "observed_value": float(selected_candidate_positive_iv_rows),
            "threshold_or_requirement": "all selected rows",
        },
        {
            "check_name": "selected_n12_candidate_repair_fail_rows",
            "check_type": "blocking",
            "passes": selected_candidate_repair_fail_rows == 0,
            "observed_value": float(selected_candidate_repair_fail_rows),
            "threshold_or_requirement": "0",
        },
        {
            "check_name": "selected_n12_candidate_full_static_proxy_warning_rows",
            "check_type": "warning",
            "passes": selected_candidate_full_static_warning_rows == 0,
            "observed_value": float(selected_candidate_full_static_warning_rows),
            "threshold_or_requirement": "preferred 0",
        },
        {
            "check_name": "selected_n12_candidate_inner_static_proxy_warning_rows",
            "check_type": "warning",
            "passes": selected_candidate_inner_static_warning_rows == 0,
            "observed_value": float(selected_candidate_inner_static_warning_rows),
            "threshold_or_requirement": "preferred 0",
        },
    ]
)

N11_STATIC_LOCALIZATION_PASS = bool(
    n11_static_localization_gate
    .query("check_type == 'blocking'")["passes"]
    .all()
)


# ------------------------------------------------------------
# Persist outputs
# ------------------------------------------------------------

_write_table_pair(
    selected_static_proxy_violations,
    "n11_selected_static_proxy_violation_intervals",
    N11_DIAGNOSTIC_DIR,
)

_write_table_pair(
    selected_static_proxy_violation_summary,
    "n11_selected_static_proxy_violation_summary",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_static_localized_contracts,
    "n11_static_localized_repaired_contracts",
    N11_HANDOFF_DIR,
)

_write_table_pair(
    n11_repaired_candidate_summary,
    "n11_repaired_candidate_summary",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_repaired_candidate_by_expiry,
    "n11_repaired_candidate_by_expiry",
    N11_TABLE_DIR,
)

_write_table_pair(
    n11_repaired_candidate_by_region,
    "n11_repaired_candidate_by_region",
    N11_TABLE_DIR,
)

_write_table_pair(
    N11_REPAIRED_N12_HANDOFF_CANDIDATE,
    "n11_repaired_n12_handoff_candidate",
    N11_HANDOFF_DIR,
)

_write_table_pair(
    n11_static_localization_gate,
    "n11_static_localization_gate",
    N11_TABLE_DIR,
)


# ------------------------------------------------------------
# Console summary
# ------------------------------------------------------------

print("Notebook 11 static-proxy localization and N12 eligibility filtering complete.")
print(f"Static localization pass: {N11_STATIC_LOCALIZATION_PASS}")
print(f"Selected N12 candidate flag: {N11_SELECTED_N12_CANDIDATE_FLAG}")
print(f"Selected handoff rows: {selected_candidate_rows:,}")
print(f"Selected handoff expiries: {selected_candidate_expiries:,}")
print(f"Selected repair-fail rows: {selected_candidate_repair_fail_rows:,}")
print(f"Selected full-grid static-warning rows: {selected_candidate_full_static_warning_rows:,}")
print(f"Selected inner-grid static-warning rows: {selected_candidate_inner_static_warning_rows:,}")

display(selected_static_proxy_violation_summary)
display(n11_repaired_candidate_summary)
display(n11_repaired_candidate_by_expiry)
display(n11_repaired_candidate_by_region)
display(n11_static_localization_gate)

if not N11_STATIC_LOCALIZATION_PASS:
    raise RuntimeError(
        "Notebook 11 static-proxy localization gate failed. "
        "Review n11_static_localization_gate before final handoff construction."
    )

Notebook 11 static-proxy localization and N12 eligibility filtering complete.
Static localization pass: True
Selected N12 candidate flag: n12_clean_repaired_candidate
Selected handoff rows: 1,250
Selected handoff expiries: 8
Selected repair-fail rows: 0
Selected full-grid static-warning rows: 0
Selected inner-grid static-warning rows: 0


,grid_name,diagnostic_type,wing_zone,violation_intervals,expiries,min_k,max_k,min_diagnostic_value,max_diagnostic_value
0,common_full_support,convexity,mid_abs_k_le_030,6,2,-0.2790706254,-0.2112264661,-0.0008912708,-0.0004418097
1,common_full_support,convexity,outer_abs_k_gt_030,186,7,-0.5334862228,-0.3129927051,-0.0006345917,-0.0000003963
2,common_full_support,monotonicity,mid_abs_k_le_030,39,3,0.1703969300,0.2297605693,0.0000001853,0.0000109897
3,common_inner_95pct_support,convexity,mid_abs_k_le_030,6,2,-0.2762447271,-0.2139642296,-0.0012890737,-0.0002718442
4,common_inner_95pct_support,convexity,outer_abs_k_gt_030,3,3,-0.3229551002,-0.3177650587,-0.0008127191,-0.0008127191


,candidate_flag,rows,expiries,unique_strikes,min_tau,max_tau,min_k,max_k,k_width,repair_warn_rows,repair_fail_rows,full_grid_static_proxy_warning_rows,inner_grid_static_proxy_warning_rows,median_abs_iv_repair,p95_abs_iv_repair,max_abs_iv_repair,median_abs_post_repair_iv_residual_to_market,p95_abs_post_repair_iv_residual_to_market,max_abs_post_repair_iv_residual_to_market,median_abs_post_repair_price_residual_to_market,p95_abs_post_repair_price_residual_to_market,max_abs_post_repair_price_residual_to_market
0,n12_broad_repaired_candidate,1258,8,260,0.0136986301,0.2054794521,-0.5084862228,0.2047605693,0.7132467922,5,0,8,0,0.0000088588,0.0000547731,0.0591598478,0.0006299940,0.0045738869,0.0559549224,0.0228173694,0.0955601400,0.1556858773
1,n12_clean_repaired_candidate,1250,8,253,0.0136986301,0.2054794521,-0.4252077096,0.1603088068,0.5855165164,4,0,0,0,0.0000088156,0.0000494663,0.0591598478,0.0006264169,0.0039988947,0.0559549224,0.0226467406,0.0959944446,0.1556858773
2,n12_anchor_repaired_candidate,1026,8,187,0.0136986301,0.2054794521,-0.1493818519,0.1391985836,0.2885804356,0,0,0,0,0.0000108972,0.0000457534,0.0000733843,0.0006154190,0.0038220523,0.0386580398,0.0272849355,0.1025110385,0.1556858773
3,n12_selected_repaired_candidate,1250,8,253,0.0136986301,0.2054794521,-0.4252077096,0.1603088068,0.5855165164,4,0,0,0,0.0000088156,0.0000494663,0.0591598478,0.0006264169,0.0039988947,0.0559549224,0.0226467406,0.0959944446,0.1556858773


,expiry,total_rows,broad_rows,clean_rows,anchor_rows,selected_rows,full_grid_static_proxy_warning_rows,inner_grid_static_proxy_warning_rows,repair_warn_rows,repair_fail_rows,min_k,max_k,median_repaired_iv,median_abs_iv_repair,max_abs_iv_repair
0,2026-07-10 00:00:00+00:00,97,97,97,97,97,0,0,0,0,-0.1242195547,0.0303652471,0.1806038007,0.0000492209,0.0000733843
1,2026-07-17 00:00:00+00:00,130,128,127,113,127,2,0,6,2,-0.3318085078,0.0516084631,0.1955179898,0.0000237789,0.0939952676
2,2026-07-24 00:00:00+00:00,152,152,152,138,152,0,0,0,0,-0.2958546667,0.0685381653,0.1854396765,0.0000148191,0.0128960197
3,2026-07-31 00:00:00+00:00,175,175,175,141,175,0,0,1,0,-0.4009080739,0.0815180753,0.2049685476,0.0000100972,0.0429275861
4,2026-08-07 00:00:00+00:00,131,131,131,112,131,0,0,0,0,-0.3342401078,0.0988765317,0.1693018977,0.0000102665,0.0000241494
5,2026-08-21 00:00:00+00:00,173,173,167,136,167,6,0,0,0,-0.5084862228,0.1391985836,0.1937336831,0.0000069492,0.0247309754
6,2026-08-31 00:00:00+00:00,187,187,187,144,187,0,0,0,0,-0.3842214430,0.1207044586,0.1950474615,0.0000051900,0.0000252678
7,2026-09-18 00:00:00+00:00,215,215,214,145,214,1,0,0,0,-0.4252077096,0.2047605693,0.2068117243,0.0000038204,0.0000218632


,moneyness_region_n11,total_rows,broad_rows,clean_rows,anchor_rows,selected_rows,full_grid_static_proxy_warning_rows,inner_grid_static_proxy_warning_rows,repair_warn_rows,repair_fail_rows,median_abs_iv_repair,p95_abs_iv_repair,max_abs_iv_repair
0,atm_abs_k_le_005,522,522,522,522,522,0,0,0,0,0.0000135684,0.0000573492,0.0000733843
1,inner_abs_k_le_015,504,504,504,504,504,0,0,0,0,0.0000076379,0.0000307295,0.0000670696
2,mid_abs_k_le_030,183,183,181,0,181,2,0,4,0,0.0000030609,0.0043617839,0.0591598478
3,outer_abs_k_gt_030,51,49,43,0,43,7,0,3,2,0.0000016317,0.0338292808,0.0939952676


,check_name,check_type,passes,observed_value,threshold_or_requirement
0,static_proxy_violations_loaded,blocking,True,240.0000000000,loaded; count may be zero or positive
1,localized_contract_table_available,blocking,True,"1,260.0000000000",> 0
2,selected_n12_candidate_rows,blocking,True,"1,250.0000000000",> 0
3,selected_n12_candidate_expiries,blocking,True,8.0000000000,>= 4
4,selected_n12_candidate_positive_variance_rows,blocking,True,"1,250.0000000000",all selected rows
5,selected_n12_candidate_positive_iv_rows,blocking,True,"1,250.0000000000",all selected rows
6,selected_n12_candidate_repair_fail_rows,blocking,True,0.0000000000,0
7,selected_n12_candidate_full_static_proxy_warni...,warning,True,0.0000000000,preferred 0
8,selected_n12_candidate_inner_static_proxy_warn...,warning,True,0.0000000000,preferred 0


In [10]:
# ------------------------------------------------------------
# Notebook 11 cell 10: Final repaired handoff, manifest, and status gate
# ------------------------------------------------------------
# This cell closes Notebook 11.
#
# It writes:
#
#   1. broad repaired N12 handoff
#   2. clean repaired N12 handoff
#   3. anchor repaired N12 handoff
#   4. selected repaired N12 handoff
#   5. excluded / warning diagnostic table
#   6. final validation ledger
#   7. final status table
#   8. final artifact inventory
#   9. machine-readable manifest and path registry
#
# Final status rule:
#
#   READY_FOR_12
#       Blocking checks pass and warning checks pass.
#
#   READY_FOR_12_WITH_WARNINGS
#       Blocking checks pass, but warning checks remain. This is expected when
#       the selected contract handoff is clean but the wider repaired grid still
#       contains documented wing/static-proxy or repair-bias warnings.
#
#   REPAIR_ONLY_NOT_CALIBRATION_READY
#       Calendar/static localization produced artifacts, but blocking checks fail.
#
#   FAILED_SURFACE_CONSTRUCTION
#       Required handoff objects are missing or unusable.

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Pre-flight checks
# ------------------------------------------------------------

required_globals_for_final = [
    "N11_STATIC_LOCALIZATION_PASS",
    "N11_SELECTED_N12_CANDIDATE_FLAG",
    "N11_SELECTED_CALENDAR_REPAIR_METHOD",
    "N11_PRE_REPAIR_SELECTED_MODE",
    "N11_REPAIRED_N12_HANDOFF_CANDIDATE",
    "N11_CALENDAR_REPAIRED_GRID",
    "n11_static_localized_contracts",
    "n11_repaired_candidate_summary",
    "n11_repaired_candidate_by_expiry",
    "n11_repaired_candidate_by_region",
    "n11_calendar_repair_candidate_ranking",
    "n11_static_localization_gate",
    "CONFIG",
]

missing_final_globals = [name for name in required_globals_for_final if name not in globals()]
if missing_final_globals:
    raise NameError(f"Missing required objects before final handoff: {missing_final_globals}")

if not bool(N11_STATIC_LOCALIZATION_PASS):
    raise RuntimeError("Static localization did not pass. Run/fix the previous cell first.")


# ------------------------------------------------------------
# Fallback helpers
# ------------------------------------------------------------

def write_json_safe(obj: dict, path: Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    def _default(x):
        if isinstance(x, (pd.Timestamp,)):
            return x.isoformat()
        if isinstance(x, Path):
            return str(x)
        if isinstance(x, (np.integer,)):
            return int(x)
        if isinstance(x, (np.floating,)):
            return float(x)
        if isinstance(x, (np.bool_,)):
            return bool(x)
        return str(x)

    path.write_text(
        json.dumps(obj, indent=2, default=_default),
        encoding="utf-8",
    )
    return path


def safe_path_string(path: Path) -> str:
    return str(Path(path)).replace("\\", "/")


def table_exists(path_dict: dict[str, Path]) -> bool:
    return all(Path(p).exists() for p in path_dict.values())


# ------------------------------------------------------------
# Build N12 handoff schema
# ------------------------------------------------------------

def make_n12_handoff_table(df: pd.DataFrame, candidate_label: str) -> pd.DataFrame:
    out = df.copy()

    out["n11_candidate_label"] = candidate_label
    out["n11_run_tag"] = RUN_TAG
    out["n11_source_pre_repair_mode"] = N11_PRE_REPAIR_SELECTED_MODE
    out["n11_calendar_repair_method"] = N11_SELECTED_CALENDAR_REPAIR_METHOD

    # Canonical N12 field names.
    out["log_moneyness"] = pd.to_numeric(out["k"], errors="coerce")
    out["selected_total_variance"] = pd.to_numeric(out["repaired_total_variance"], errors="coerce")
    out["selected_iv"] = pd.to_numeric(out["repaired_iv"], errors="coerce")
    out["selected_model_price"] = pd.to_numeric(out["repaired_bsm_price"], errors="coerce")
    out["selected_market_price"] = pd.to_numeric(out["market_price"], errors="coerce")

    out["pre_repair_model_total_variance"] = pd.to_numeric(
        out["pre_repair_total_variance"],
        errors="coerce",
    )
    out["pre_repair_model_iv"] = pd.to_numeric(out["pre_repair_iv"], errors="coerce")
    out["pre_repair_model_price"] = pd.to_numeric(out["pre_repair_bsm_price"], errors="coerce")

    out["market_total_variance_target"] = pd.to_numeric(out["market_total_variance"], errors="coerce")
    out["market_iv_target"] = pd.to_numeric(out["market_iv"], errors="coerce")
    out["market_price_target"] = pd.to_numeric(out["market_price"], errors="coerce")

    # Calibration weight: retain upstream fit weight, but reduce rows with repair warnings.
    base_weight = pd.to_numeric(out["fit_weight"], errors="coerce").fillna(0.0).clip(lower=0.0)

    repair_warn_multiplier = np.where(out["repair_bias_warn"].fillna(False).astype(bool), 0.25, 1.00)
    static_warning_multiplier = np.where(
        out["full_grid_static_proxy_warning"].fillna(False).astype(bool)
        | out["inner_grid_static_proxy_warning"].fillna(False).astype(bool),
        0.00,
        1.00,
    )

    anchor_multiplier = np.where(
        out.get("n12_anchor_repaired_candidate", pd.Series(False, index=out.index)).fillna(False).astype(bool),
        1.50,
        1.00,
    )

    raw_calibration_weight = (
        base_weight
        * repair_warn_multiplier
        * static_warning_multiplier
        * anchor_multiplier
    )

    positive_mean = raw_calibration_weight.loc[raw_calibration_weight > 0].mean()
    if not np.isfinite(positive_mean) or positive_mean <= 0:
        positive_mean = 1.0

    out["n12_calibration_weight_raw"] = raw_calibration_weight
    out["n12_calibration_weight"] = raw_calibration_weight / positive_mean

    out["n12_handoff_warning_count"] = (
        out["repair_bias_warn"].fillna(False).astype(int)
        + out["full_grid_static_proxy_warning"].fillna(False).astype(int)
        + out["inner_grid_static_proxy_warning"].fillna(False).astype(int)
    )

    out["n12_handoff_blocking_issue_count"] = (
        out["repair_bias_fail"].fillna(False).astype(int)
        + (~np.isfinite(out["selected_total_variance"])).astype(int)
        + (out["selected_total_variance"] <= CONFIG.positive_total_variance_floor).astype(int)
        + (~np.isfinite(out["selected_iv"])).astype(int)
        + (out["selected_iv"] <= 0.0).astype(int)
        + (out["n12_calibration_weight"] <= 0.0).astype(int)
    )

    out["n12_handoff_ready"] = out["n12_handoff_blocking_issue_count"] == 0

    out["n12_handoff_bucket"] = np.select(
        [
            out.get("n12_anchor_repaired_candidate", pd.Series(False, index=out.index)).fillna(False).astype(bool),
            out.get("n12_clean_repaired_candidate", pd.Series(False, index=out.index)).fillna(False).astype(bool),
            out.get("n12_broad_repaired_candidate", pd.Series(False, index=out.index)).fillna(False).astype(bool),
        ],
        [
            "anchor_repaired",
            "clean_repaired",
            "broad_repaired",
        ],
        default="diagnostic_only",
    )

    preferred_cols = [
        "contract_key",
        "n11_run_tag",
        "n11_candidate_label",
        "n11_source_pre_repair_mode",
        "n11_calendar_repair_method",
        "n12_handoff_bucket",
        "n12_handoff_ready",
        "n12_handoff_warning_count",
        "n12_handoff_blocking_issue_count",
        "expiry",
        "tau_years",
        "option_type",
        "strike",
        "forward",
        "discount_factor",
        "log_moneyness",
        "abs_k",
        "moneyness_region_n11",
        "selected_market_price",
        "selected_model_price",
        "selected_total_variance",
        "selected_iv",
        "market_price_target",
        "market_total_variance_target",
        "market_iv_target",
        "pre_repair_model_price",
        "pre_repair_model_total_variance",
        "pre_repair_model_iv",
        "total_variance_repair",
        "abs_total_variance_repair",
        "iv_repair",
        "abs_iv_repair",
        "post_repair_total_variance_residual_to_market",
        "post_repair_iv_residual_to_market",
        "post_repair_price_residual_to_market",
        "abs_post_repair_iv_residual_to_market",
        "abs_post_repair_price_residual_to_market",
        "fit_weight",
        "n12_calibration_weight_raw",
        "n12_calibration_weight",
        "repair_bias_warn",
        "repair_bias_fail",
        "full_grid_static_proxy_warning",
        "full_grid_monotonicity_warning",
        "full_grid_convexity_warning",
        "inner_grid_static_proxy_warning",
        "inner_grid_monotonicity_warning",
        "inner_grid_convexity_warning",
        "n12_broad_repaired_candidate",
        "n12_clean_repaired_candidate",
        "n12_anchor_repaired_candidate",
        "n12_selected_repaired_candidate",
    ]

    existing_preferred = [col for col in preferred_cols if col in out.columns]
    remaining = [col for col in out.columns if col not in existing_preferred]

    return out[existing_preferred + remaining].copy()


N11_TO_N12_BROAD_HANDOFF = make_n12_handoff_table(
    n11_static_localized_contracts.loc[
        n11_static_localized_contracts["n12_broad_repaired_candidate"]
    ].copy(),
    candidate_label="broad_repaired",
)

N11_TO_N12_CLEAN_HANDOFF = make_n12_handoff_table(
    n11_static_localized_contracts.loc[
        n11_static_localized_contracts["n12_clean_repaired_candidate"]
    ].copy(),
    candidate_label="clean_repaired",
)

N11_TO_N12_ANCHOR_HANDOFF = make_n12_handoff_table(
    n11_static_localized_contracts.loc[
        n11_static_localized_contracts["n12_anchor_repaired_candidate"]
    ].copy(),
    candidate_label="anchor_repaired",
)

N11_TO_N12_SELECTED_HANDOFF = make_n12_handoff_table(
    N11_REPAIRED_N12_HANDOFF_CANDIDATE.copy(),
    candidate_label="selected_repaired",
)

N11_TO_N12_EXCLUDED_DIAGNOSTIC = (
    n11_static_localized_contracts
    .loc[~n11_static_localized_contracts["n12_selected_repaired_candidate"]]
    .copy()
    .sort_values(["expiry", "k", "strike"])
    .reset_index(drop=True)
)

N11_TO_N12_EXCLUDED_DIAGNOSTIC["n11_exclusion_from_selected_reason"] = np.select(
    [
        N11_TO_N12_EXCLUDED_DIAGNOSTIC["repair_bias_fail"].fillna(False).astype(bool),
        N11_TO_N12_EXCLUDED_DIAGNOSTIC["full_grid_static_proxy_warning"].fillna(False).astype(bool),
        N11_TO_N12_EXCLUDED_DIAGNOSTIC["inner_grid_static_proxy_warning"].fillna(False).astype(bool),
        ~N11_TO_N12_EXCLUDED_DIAGNOSTIC["eligible_for_n12_after_calendar_repair"].fillna(False).astype(bool),
    ],
    [
        "repair_bias_fail",
        "full_grid_static_proxy_warning",
        "inner_grid_static_proxy_warning",
        "not_calendar_repair_eligible",
    ],
    default="not_selected_by_candidate_flag",
)


# ------------------------------------------------------------
# Handoff summaries
# ------------------------------------------------------------

def summarize_final_handoff(name: str, df: pd.DataFrame) -> dict[str, object]:
    if len(df) == 0:
        return {
            "handoff_name": name,
            "rows": 0,
            "expiries": 0,
            "unique_strikes": 0,
            "min_tau": np.nan,
            "max_tau": np.nan,
            "min_k": np.nan,
            "max_k": np.nan,
            "repair_warn_rows": 0,
            "repair_fail_rows": 0,
            "full_grid_static_warning_rows": 0,
            "inner_grid_static_warning_rows": 0,
            "ready_rows": 0,
            "positive_weight_rows": 0,
            "duplicate_contract_keys": 0,
            "median_selected_iv": np.nan,
            "p95_abs_iv_repair": np.nan,
            "max_abs_iv_repair": np.nan,
            "p95_abs_post_repair_iv_residual_to_market": np.nan,
            "max_abs_post_repair_iv_residual_to_market": np.nan,
        }

    contract_key = df["contract_key"].astype(str)

    return {
        "handoff_name": name,
        "rows": int(len(df)),
        "expiries": int(df["expiry"].nunique()),
        "unique_strikes": int(df["strike"].nunique()),
        "min_tau": float(pd.to_numeric(df["tau_years"], errors="coerce").min()),
        "max_tau": float(pd.to_numeric(df["tau_years"], errors="coerce").max()),
        "min_k": float(pd.to_numeric(df["log_moneyness"], errors="coerce").min()),
        "max_k": float(pd.to_numeric(df["log_moneyness"], errors="coerce").max()),
        "repair_warn_rows": int(df["repair_bias_warn"].fillna(False).sum()),
        "repair_fail_rows": int(df["repair_bias_fail"].fillna(False).sum()),
        "full_grid_static_warning_rows": int(df["full_grid_static_proxy_warning"].fillna(False).sum()),
        "inner_grid_static_warning_rows": int(df["inner_grid_static_proxy_warning"].fillna(False).sum()),
        "ready_rows": int(df["n12_handoff_ready"].fillna(False).sum()),
        "positive_weight_rows": int((pd.to_numeric(df["n12_calibration_weight"], errors="coerce") > 0).sum()),
        "duplicate_contract_keys": int(contract_key.duplicated().sum()),
        "median_selected_iv": float(np.nanmedian(pd.to_numeric(df["selected_iv"], errors="coerce"))),
        "p95_abs_iv_repair": float(np.nanpercentile(pd.to_numeric(df["abs_iv_repair"], errors="coerce"), 95)),
        "max_abs_iv_repair": float(np.nanmax(pd.to_numeric(df["abs_iv_repair"], errors="coerce"))),
        "p95_abs_post_repair_iv_residual_to_market": float(
            np.nanpercentile(pd.to_numeric(df["abs_post_repair_iv_residual_to_market"], errors="coerce"), 95)
        ),
        "max_abs_post_repair_iv_residual_to_market": float(
            np.nanmax(pd.to_numeric(df["abs_post_repair_iv_residual_to_market"], errors="coerce"))
        ),
    }


n11_final_handoff_summary = pd.DataFrame(
    [
        summarize_final_handoff("broad_repaired", N11_TO_N12_BROAD_HANDOFF),
        summarize_final_handoff("clean_repaired", N11_TO_N12_CLEAN_HANDOFF),
        summarize_final_handoff("anchor_repaired", N11_TO_N12_ANCHOR_HANDOFF),
        summarize_final_handoff("selected_repaired", N11_TO_N12_SELECTED_HANDOFF),
    ]
)

n11_final_handoff_by_expiry = (
    N11_TO_N12_SELECTED_HANDOFF
    .groupby("expiry", dropna=False)
    .agg(
        rows=("contract_key", "size"),
        unique_strikes=("strike", "nunique"),
        min_tau=("tau_years", "min"),
        max_tau=("tau_years", "max"),
        min_k=("log_moneyness", "min"),
        max_k=("log_moneyness", "max"),
        median_selected_iv=("selected_iv", "median"),
        median_selected_total_variance=("selected_total_variance", "median"),
        median_calibration_weight=("n12_calibration_weight", "median"),
        repair_warn_rows=("repair_bias_warn", "sum"),
        repair_fail_rows=("repair_bias_fail", "sum"),
        full_grid_static_warning_rows=("full_grid_static_proxy_warning", "sum"),
        inner_grid_static_warning_rows=("inner_grid_static_proxy_warning", "sum"),
        ready_rows=("n12_handoff_ready", "sum"),
        max_abs_iv_repair=("abs_iv_repair", "max"),
        p95_abs_post_repair_iv_residual_to_market=(
            "abs_post_repair_iv_residual_to_market",
            lambda s: float(np.nanpercentile(s, 95)),
        ),
    )
    .reset_index()
)

n11_final_handoff_by_bucket = (
    N11_TO_N12_SELECTED_HANDOFF
    .groupby(["n12_handoff_bucket", "moneyness_region_n11"], dropna=False)
    .agg(
        rows=("contract_key", "size"),
        expiries=("expiry", "nunique"),
        unique_strikes=("strike", "nunique"),
        median_selected_iv=("selected_iv", "median"),
        median_calibration_weight=("n12_calibration_weight", "median"),
        repair_warn_rows=("repair_bias_warn", "sum"),
        repair_fail_rows=("repair_bias_fail", "sum"),
        full_grid_static_warning_rows=("full_grid_static_proxy_warning", "sum"),
        inner_grid_static_warning_rows=("inner_grid_static_proxy_warning", "sum"),
        max_abs_iv_repair=("abs_iv_repair", "max"),
    )
    .reset_index()
)


# ------------------------------------------------------------
# Write final artifacts
# ------------------------------------------------------------

artifact_paths = {}

artifact_paths["broad_handoff"] = _write_table_pair(
    N11_TO_N12_BROAD_HANDOFF,
    "n11_to_n12_broad_repaired_handoff",
    N11_HANDOFF_DIR,
)

artifact_paths["clean_handoff"] = _write_table_pair(
    N11_TO_N12_CLEAN_HANDOFF,
    "n11_to_n12_clean_repaired_handoff",
    N11_HANDOFF_DIR,
)

artifact_paths["anchor_handoff"] = _write_table_pair(
    N11_TO_N12_ANCHOR_HANDOFF,
    "n11_to_n12_anchor_repaired_handoff",
    N11_HANDOFF_DIR,
)

artifact_paths["selected_handoff"] = _write_table_pair(
    N11_TO_N12_SELECTED_HANDOFF,
    "n11_to_n12_selected_repaired_handoff",
    N11_HANDOFF_DIR,
)

artifact_paths["excluded_diagnostic"] = _write_table_pair(
    N11_TO_N12_EXCLUDED_DIAGNOSTIC,
    "n11_to_n12_excluded_or_warning_diagnostic",
    N11_HANDOFF_DIR,
)

artifact_paths["selected_repaired_grid"] = _write_table_pair(
    N11_CALENDAR_REPAIRED_GRID,
    "n11_to_n12_selected_repaired_surface_grid",
    N11_GRID_DIR,
)

artifact_paths["final_handoff_summary"] = _write_table_pair(
    n11_final_handoff_summary,
    "n11_final_handoff_summary",
    N11_TABLE_DIR,
)

artifact_paths["final_handoff_by_expiry"] = _write_table_pair(
    n11_final_handoff_by_expiry,
    "n11_final_handoff_by_expiry",
    N11_TABLE_DIR,
)

artifact_paths["final_handoff_by_bucket"] = _write_table_pair(
    n11_final_handoff_by_bucket,
    "n11_final_handoff_by_bucket",
    N11_TABLE_DIR,
)


# ------------------------------------------------------------
# Final validation ledger
# ------------------------------------------------------------

selected_rows = int(len(N11_TO_N12_SELECTED_HANDOFF))
selected_expiries = int(N11_TO_N12_SELECTED_HANDOFF["expiry"].nunique())
selected_ready_rows = int(N11_TO_N12_SELECTED_HANDOFF["n12_handoff_ready"].sum())
selected_duplicate_contracts = int(N11_TO_N12_SELECTED_HANDOFF["contract_key"].astype(str).duplicated().sum())
selected_positive_weight_rows = int(
    (pd.to_numeric(N11_TO_N12_SELECTED_HANDOFF["n12_calibration_weight"], errors="coerce") > 0).sum()
)
selected_repair_fail_rows = int(N11_TO_N12_SELECTED_HANDOFF["repair_bias_fail"].sum())
selected_repair_warn_rows = int(N11_TO_N12_SELECTED_HANDOFF["repair_bias_warn"].sum())
selected_full_static_warning_rows = int(N11_TO_N12_SELECTED_HANDOFF["full_grid_static_proxy_warning"].sum())
selected_inner_static_warning_rows = int(N11_TO_N12_SELECTED_HANDOFF["inner_grid_static_proxy_warning"].sum())

selected_positive_variance_rows = int(
    (
        np.isfinite(N11_TO_N12_SELECTED_HANDOFF["selected_total_variance"])
        & (N11_TO_N12_SELECTED_HANDOFF["selected_total_variance"] > CONFIG.positive_total_variance_floor)
    ).sum()
)
selected_positive_iv_rows = int(
    (
        np.isfinite(N11_TO_N12_SELECTED_HANDOFF["selected_iv"])
        & (N11_TO_N12_SELECTED_HANDOFF["selected_iv"] > 0.0)
    ).sum()
)

selected_max_abs_iv_repair = float(
    np.nanmax(pd.to_numeric(N11_TO_N12_SELECTED_HANDOFF["abs_iv_repair"], errors="coerce"))
)
selected_p95_abs_iv_repair = float(
    np.nanpercentile(pd.to_numeric(N11_TO_N12_SELECTED_HANDOFF["abs_iv_repair"], errors="coerce"), 95)
)

selected_p95_abs_post_repair_iv_residual = float(
    np.nanpercentile(
        pd.to_numeric(N11_TO_N12_SELECTED_HANDOFF["abs_post_repair_iv_residual_to_market"], errors="coerce"),
        95,
    )
)

final_validation_rows = [
    {
        "check_name": "static_localization_passed",
        "check_type": "blocking",
        "passes": bool(N11_STATIC_LOCALIZATION_PASS),
        "observed_value": float(N11_STATIC_LOCALIZATION_PASS),
        "threshold_or_requirement": "True",
    },
    {
        "check_name": "selected_handoff_rows",
        "check_type": "blocking",
        "passes": selected_rows > 0,
        "observed_value": float(selected_rows),
        "threshold_or_requirement": "> 0",
    },
    {
        "check_name": "selected_handoff_expiries",
        "check_type": "blocking",
        "passes": selected_expiries >= CONFIG.min_primary_expiries,
        "observed_value": float(selected_expiries),
        "threshold_or_requirement": f">= {CONFIG.min_primary_expiries}",
    },
    {
        "check_name": "selected_handoff_ready_rows",
        "check_type": "blocking",
        "passes": selected_ready_rows == selected_rows,
        "observed_value": float(selected_ready_rows),
        "threshold_or_requirement": "all selected rows",
    },
    {
        "check_name": "selected_handoff_positive_variance",
        "check_type": "blocking",
        "passes": selected_positive_variance_rows == selected_rows,
        "observed_value": float(selected_positive_variance_rows),
        "threshold_or_requirement": "all selected rows",
    },
    {
        "check_name": "selected_handoff_positive_iv",
        "check_type": "blocking",
        "passes": selected_positive_iv_rows == selected_rows,
        "observed_value": float(selected_positive_iv_rows),
        "threshold_or_requirement": "all selected rows",
    },
    {
        "check_name": "selected_handoff_positive_calibration_weights",
        "check_type": "blocking",
        "passes": selected_positive_weight_rows == selected_rows,
        "observed_value": float(selected_positive_weight_rows),
        "threshold_or_requirement": "all selected rows",
    },
    {
        "check_name": "selected_handoff_duplicate_contract_keys",
        "check_type": "blocking",
        "passes": selected_duplicate_contracts == 0,
        "observed_value": float(selected_duplicate_contracts),
        "threshold_or_requirement": "0",
    },
    {
        "check_name": "selected_handoff_repair_fail_rows",
        "check_type": "blocking",
        "passes": selected_repair_fail_rows == 0,
        "observed_value": float(selected_repair_fail_rows),
        "threshold_or_requirement": "0",
    },
    {
        "check_name": "selected_handoff_full_static_proxy_warning_rows",
        "check_type": "blocking",
        "passes": selected_full_static_warning_rows == 0,
        "observed_value": float(selected_full_static_warning_rows),
        "threshold_or_requirement": "0",
    },
    {
        "check_name": "selected_handoff_inner_static_proxy_warning_rows",
        "check_type": "blocking",
        "passes": selected_inner_static_warning_rows == 0,
        "observed_value": float(selected_inner_static_warning_rows),
        "threshold_or_requirement": "0",
    },
    {
        "check_name": "selected_p95_abs_iv_repair",
        "check_type": "warning",
        "passes": selected_p95_abs_iv_repair <= CONFIG.global_abs_iv_repair_warn,
        "observed_value": float(selected_p95_abs_iv_repair),
        "threshold_or_requirement": f"<= {CONFIG.global_abs_iv_repair_warn}",
    },
    {
        "check_name": "selected_max_abs_iv_repair",
        "check_type": "warning",
        "passes": selected_max_abs_iv_repair <= CONFIG.global_abs_iv_repair_fail,
        "observed_value": float(selected_max_abs_iv_repair),
        "threshold_or_requirement": f"<= {CONFIG.global_abs_iv_repair_fail}",
    },
    {
        "check_name": "selected_repair_warn_rows",
        "check_type": "warning",
        "passes": selected_repair_warn_rows == 0,
        "observed_value": float(selected_repair_warn_rows),
        "threshold_or_requirement": "preferred 0",
    },
    {
        "check_name": "selected_p95_abs_post_repair_iv_residual_to_market",
        "check_type": "warning",
        "passes": selected_p95_abs_post_repair_iv_residual <= CONFIG.global_abs_iv_repair_warn,
        "observed_value": float(selected_p95_abs_post_repair_iv_residual),
        "threshold_or_requirement": f"<= {CONFIG.global_abs_iv_repair_warn}",
    },
    {
        "check_name": "final_artifacts_written",
        "check_type": "blocking",
        "passes": all(table_exists(paths) for paths in artifact_paths.values()),
        "observed_value": float(sum(table_exists(paths) for paths in artifact_paths.values())),
        "threshold_or_requirement": f"== {len(artifact_paths)} artifact groups",
    },
]

n11_final_validation_ledger = pd.DataFrame(final_validation_rows)

N11_FINAL_BLOCKING_PASS = bool(
    n11_final_validation_ledger
    .query("check_type == 'blocking'")["passes"]
    .all()
)

N11_FINAL_WARNING_PASS = bool(
    n11_final_validation_ledger
    .query("check_type == 'warning'")["passes"]
    .all()
)

if N11_FINAL_BLOCKING_PASS and N11_FINAL_WARNING_PASS:
    N11_FINAL_STATUS = "READY_FOR_12"
elif N11_FINAL_BLOCKING_PASS:
    N11_FINAL_STATUS = "READY_FOR_12_WITH_WARNINGS"
elif selected_rows > 0:
    N11_FINAL_STATUS = "REPAIR_ONLY_NOT_CALIBRATION_READY"
else:
    N11_FINAL_STATUS = "FAILED_SURFACE_CONSTRUCTION"


# ------------------------------------------------------------
# Final status table, artifact inventory, manifest
# ------------------------------------------------------------

n11_final_status_table = pd.DataFrame(
    [
        {
            "notebook_id": NOTEBOOK_ID,
            "notebook_name": NOTEBOOK_NAME,
            "run_tag": RUN_TAG,
            "final_status": N11_FINAL_STATUS,
            "blocking_pass": N11_FINAL_BLOCKING_PASS,
            "warning_pass": N11_FINAL_WARNING_PASS,
            "selected_pre_repair_mode": N11_PRE_REPAIR_SELECTED_MODE,
            "selected_calendar_repair_method": N11_SELECTED_CALENDAR_REPAIR_METHOD,
            "selected_n12_candidate_flag": N11_SELECTED_N12_CANDIDATE_FLAG,
            "selected_handoff_rows": selected_rows,
            "selected_handoff_expiries": selected_expiries,
            "selected_handoff_unique_strikes": int(N11_TO_N12_SELECTED_HANDOFF["strike"].nunique()),
            "selected_repair_warn_rows": selected_repair_warn_rows,
            "selected_repair_fail_rows": selected_repair_fail_rows,
            "selected_full_static_proxy_warning_rows": selected_full_static_warning_rows,
            "selected_inner_static_proxy_warning_rows": selected_inner_static_warning_rows,
            "selected_p95_abs_iv_repair": selected_p95_abs_iv_repair,
            "selected_max_abs_iv_repair": selected_max_abs_iv_repair,
            "selected_p95_abs_post_repair_iv_residual_to_market": selected_p95_abs_post_repair_iv_residual,
            "allowed_claim": (
                "Selected N12 handoff is calendar-repaired, positive-variance, "
                "butterfly-clean on prior SVI slice diagnostics, and free of localized "
                "static-proxy warning intervals after filtering."
            ),
            "disallowed_claim": (
                "Do not claim the full repaired grid is globally arbitrage-free; "
                "wing static-proxy warnings were documented and filtered from the selected handoff."
            ),
        }
    ]
)

artifact_inventory_rows = []

for artifact_key, paths in artifact_paths.items():
    for format_key, path in paths.items():
        artifact_inventory_rows.append(
            {
                "artifact_key": artifact_key,
                "format_key": format_key,
                "path": str(path),
                "exists": Path(path).exists(),
            }
        )

n11_final_artifact_inventory = pd.DataFrame(artifact_inventory_rows)

# Write final validation/status/inventory after status is known.
artifact_paths["final_validation_ledger"] = _write_table_pair(
    n11_final_validation_ledger,
    "n11_final_validation_ledger",
    N11_TABLE_DIR,
)

artifact_paths["final_status_table"] = _write_table_pair(
    n11_final_status_table,
    "n11_final_status_table",
    N11_TABLE_DIR,
)

artifact_paths["final_artifact_inventory"] = _write_table_pair(
    n11_final_artifact_inventory,
    "n11_final_artifact_inventory",
    N11_TABLE_DIR,
)

final_manifest = {
    "notebook_id": NOTEBOOK_ID,
    "notebook_name": NOTEBOOK_NAME,
    "run_tag": RUN_TAG,
    "final_status": N11_FINAL_STATUS,
    "blocking_pass": N11_FINAL_BLOCKING_PASS,
    "warning_pass": N11_FINAL_WARNING_PASS,
    "selected_pre_repair_mode": N11_PRE_REPAIR_SELECTED_MODE,
    "selected_calendar_repair_method": N11_SELECTED_CALENDAR_REPAIR_METHOD,
    "selected_n12_candidate_flag": N11_SELECTED_N12_CANDIDATE_FLAG,
    "selected_handoff_rows": selected_rows,
    "selected_handoff_expiries": selected_expiries,
    "selected_handoff_unique_strikes": int(N11_TO_N12_SELECTED_HANDOFF["strike"].nunique()),
    "selected_repair_warn_rows": selected_repair_warn_rows,
    "selected_repair_fail_rows": selected_repair_fail_rows,
    "selected_full_static_proxy_warning_rows": selected_full_static_warning_rows,
    "selected_inner_static_proxy_warning_rows": selected_inner_static_warning_rows,
    "selected_p95_abs_iv_repair": selected_p95_abs_iv_repair,
    "selected_max_abs_iv_repair": selected_max_abs_iv_repair,
    "selected_p95_abs_post_repair_iv_residual_to_market": selected_p95_abs_post_repair_iv_residual,
    "artifacts": {
        key: {subkey: safe_path_string(path) for subkey, path in paths.items()}
        for key, paths in artifact_paths.items()
    },
    "latest_handoff_paths": {
        "broad_handoff_latest": safe_path_string(artifact_paths["broad_handoff"]["latest_parquet"]),
        "clean_handoff_latest": safe_path_string(artifact_paths["clean_handoff"]["latest_parquet"]),
        "anchor_handoff_latest": safe_path_string(artifact_paths["anchor_handoff"]["latest_parquet"]),
        "selected_handoff_latest": safe_path_string(artifact_paths["selected_handoff"]["latest_parquet"]),
        "selected_repaired_grid_latest": safe_path_string(artifact_paths["selected_repaired_grid"]["latest_parquet"]),
        "final_status_table_latest": safe_path_string(artifact_paths["final_status_table"]["latest_csv"]),
        "final_validation_ledger_latest": safe_path_string(artifact_paths["final_validation_ledger"]["latest_csv"]),
    },
    "allowed_claims": [
        "The selected handoff contains positive repaired total variances and positive repaired implied volatilities.",
        "The selected handoff passed calendar repair checks on the common full and inner grids.",
        "The selected handoff excludes rows touched by localized full-grid and inner-grid static-proxy warning intervals.",
        "The selected handoff is suitable as a repaired empirical target for Notebook 12 calibration experiments.",
    ],
    "disallowed_claims": [
        "Do not claim the entire repaired grid is globally arbitrage-free.",
        "Do not claim Heston/Bates calibration readiness beyond the selected repaired handoff.",
        "Do not claim repair introduced no bias; repair bias is measured and must be carried forward.",
    ],
}

manifest_path = write_json_safe(
    final_manifest,
    N11_MANIFEST_DIR / f"n11_final_manifest_{RUN_TAG}.json",
)

manifest_latest_path = write_json_safe(
    final_manifest,
    N11_MANIFEST_DIR / "n11_final_manifest_latest.json",
)

path_registry_lines = [
    f'N11_STATUS = "{N11_FINAL_STATUS}"',
    f'N11_RUN_TAG = "{RUN_TAG}"',
    f'N11_SELECTED_PRE_REPAIR_MODE = "{N11_PRE_REPAIR_SELECTED_MODE}"',
    f'N11_SELECTED_CALENDAR_REPAIR_METHOD = "{N11_SELECTED_CALENDAR_REPAIR_METHOD}"',
    f'N11_SELECTED_N12_CANDIDATE_FLAG = "{N11_SELECTED_N12_CANDIDATE_FLAG}"',
    f'N11_TO_N12_BROAD_HANDOFF_LATEST = r"{safe_path_string(artifact_paths["broad_handoff"]["latest_parquet"])}"',
    f'N11_TO_N12_CLEAN_HANDOFF_LATEST = r"{safe_path_string(artifact_paths["clean_handoff"]["latest_parquet"])}"',
    f'N11_TO_N12_ANCHOR_HANDOFF_LATEST = r"{safe_path_string(artifact_paths["anchor_handoff"]["latest_parquet"])}"',
    f'N11_TO_N12_SELECTED_HANDOFF_LATEST = r"{safe_path_string(artifact_paths["selected_handoff"]["latest_parquet"])}"',
    f'N11_TO_N12_SELECTED_REPAIRED_GRID_LATEST = r"{safe_path_string(artifact_paths["selected_repaired_grid"]["latest_parquet"])}"',
    f'N11_FINAL_STATUS_TABLE_LATEST = r"{safe_path_string(artifact_paths["final_status_table"]["latest_csv"])}"',
    f'N11_FINAL_VALIDATION_LEDGER_LATEST = r"{safe_path_string(artifact_paths["final_validation_ledger"]["latest_csv"])}"',
    f'N11_FINAL_MANIFEST_LATEST = r"{safe_path_string(manifest_latest_path)}"',
]

registry_path = N11_MANIFEST_DIR / f"n11_paths_registry_{RUN_TAG}.txt"
registry_latest_path = N11_MANIFEST_DIR / "n11_paths_registry_latest.txt"

registry_path.write_text("\n".join(path_registry_lines) + "\n", encoding="utf-8")
registry_latest_path.write_text("\n".join(path_registry_lines) + "\n", encoding="utf-8")


# ------------------------------------------------------------
# Console summary
# ------------------------------------------------------------

print("Notebook 11 final handoff and status complete.")
print(f"Final status: {N11_FINAL_STATUS}")
print(f"Blocking pass: {N11_FINAL_BLOCKING_PASS}")
print(f"Warning pass: {N11_FINAL_WARNING_PASS}")
print(f"Selected handoff rows: {selected_rows:,}")
print(f"Selected handoff expiries: {selected_expiries:,}")
print(f"Selected candidate flag: {N11_SELECTED_N12_CANDIDATE_FLAG}")
print(f"Selected pre-repair mode: {N11_PRE_REPAIR_SELECTED_MODE}")
print(f"Selected calendar repair method: {N11_SELECTED_CALENDAR_REPAIR_METHOD}")
print(f"Final manifest: {manifest_latest_path}")
print(f"Path registry: {registry_latest_path}")

display(n11_final_status_table)
display(n11_final_validation_ledger)
display(n11_final_handoff_summary)
display(n11_final_handoff_by_expiry)
display(n11_final_handoff_by_bucket)
display(n11_final_artifact_inventory)

if not N11_FINAL_BLOCKING_PASS:
    raise RuntimeError(
        "Notebook 11 final blocking gate failed. "
        "Review n11_final_validation_ledger before starting Notebook 12."
    )

Notebook 11 final handoff and status complete.
Final status: READY_FOR_12_WITH_WARNINGS
Blocking pass: True
Warning pass: False
Selected handoff rows: 1,250
Selected handoff expiries: 8
Selected candidate flag: n12_clean_repaired_candidate
Selected pre-repair mode: weighted_broad_svi
Selected calendar repair method: calendar_pava
Final manifest: D:\Derivative Pricing Project v1.0+\V1.1\outputs\11_arbitrage_aware_svi_ssvi_surface_repair_and_handoff\manifest\n11_final_manifest_latest.json
Path registry: D:\Derivative Pricing Project v1.0+\V1.1\outputs\11_arbitrage_aware_svi_ssvi_surface_repair_and_handoff\manifest\n11_paths_registry_latest.txt


,notebook_id,notebook_name,run_tag,final_status,blocking_pass,warning_pass,selected_pre_repair_mode,selected_calendar_repair_method,selected_n12_candidate_flag,selected_handoff_rows,selected_handoff_expiries,selected_handoff_unique_strikes,selected_repair_warn_rows,selected_repair_fail_rows,selected_full_static_proxy_warning_rows,selected_inner_static_proxy_warning_rows,selected_p95_abs_iv_repair,selected_max_abs_iv_repair,selected_p95_abs_post_repair_iv_residual_to_market,allowed_claim,disallowed_claim
0,11,11_arbitrage_aware_svi_ssvi_surface_repair_and...,n11_20260705T235629Z,READY_FOR_12_WITH_WARNINGS,True,False,weighted_broad_svi,calendar_pava,n12_clean_repaired_candidate,1250,8,253,4,0,0,0,0.0000494663,0.0591598478,0.0039988947,"Selected N12 handoff is calendar-repaired, pos...",Do not claim the full repaired grid is globall...


,check_name,check_type,passes,observed_value,threshold_or_requirement
0,static_localization_passed,blocking,True,1.0000000000,True
1,selected_handoff_rows,blocking,True,"1,250.0000000000",> 0
2,selected_handoff_expiries,blocking,True,8.0000000000,>= 4
3,selected_handoff_ready_rows,blocking,True,"1,250.0000000000",all selected rows
4,selected_handoff_positive_variance,blocking,True,"1,250.0000000000",all selected rows
5,selected_handoff_positive_iv,blocking,True,"1,250.0000000000",all selected rows
6,selected_handoff_positive_calibration_weights,blocking,True,"1,250.0000000000",all selected rows
7,selected_handoff_duplicate_contract_keys,blocking,True,0.0000000000,0
8,selected_handoff_repair_fail_rows,blocking,True,0.0000000000,0
9,selected_handoff_full_static_proxy_warning_rows,blocking,True,0.0000000000,0


,handoff_name,rows,expiries,unique_strikes,min_tau,max_tau,min_k,max_k,repair_warn_rows,repair_fail_rows,full_grid_static_warning_rows,inner_grid_static_warning_rows,ready_rows,positive_weight_rows,duplicate_contract_keys,median_selected_iv,p95_abs_iv_repair,max_abs_iv_repair,p95_abs_post_repair_iv_residual_to_market,max_abs_post_repair_iv_residual_to_market
0,broad_repaired,1258,8,260,0.0136986301,0.2054794521,-0.5084862228,0.2047605693,5,0,8,0,1250,1250,0,0.1922158227,0.0000547731,0.0591598478,0.0045738869,0.0559549224
1,clean_repaired,1250,8,253,0.0136986301,0.2054794521,-0.4252077096,0.1603088068,4,0,0,0,1250,1250,0,0.1914496400,0.0000494663,0.0591598478,0.0039988947,0.0559549224
2,anchor_repaired,1026,8,187,0.0136986301,0.2054794521,-0.1493818519,0.1391985836,0,0,0,0,1026,1026,0,0.1732423288,0.0000457534,0.0000733843,0.0038220523,0.0386580398
3,selected_repaired,1250,8,253,0.0136986301,0.2054794521,-0.4252077096,0.1603088068,4,0,0,0,1250,1250,0,0.1914496400,0.0000494663,0.0591598478,0.0039988947,0.0559549224


,expiry,rows,unique_strikes,min_tau,max_tau,min_k,max_k,median_selected_iv,median_selected_total_variance,median_calibration_weight,repair_warn_rows,repair_fail_rows,full_grid_static_warning_rows,inner_grid_static_warning_rows,ready_rows,max_abs_iv_repair,p95_abs_post_repair_iv_residual_to_market
0,2026-07-10 00:00:00+00:00,97,97,0.0136986301,0.0136986301,-0.1242195547,0.0303652471,0.1806038007,0.0004468183,0.5160280164,0,0,0,0,97,0.0000733843,0.0055596585
1,2026-07-17 00:00:00+00:00,127,127,0.0328767123,0.0328767123,-0.2951071410,0.0516084631,0.1926747287,0.0012205003,0.7373528884,3,0,0,0,127,0.0591598478,0.0155841352
2,2026-07-24 00:00:00+00:00,152,152,0.0520547945,0.0520547945,-0.2958546667,0.0685381653,0.1854396765,0.0017900844,0.6084681558,0,0,0,0,152,0.0128960197,0.0122263630
3,2026-07-31 00:00:00+00:00,175,175,0.0712328767,0.0712328767,-0.4009080739,0.0815180753,0.2049685476,0.0029926431,0.6950502419,1,0,0,0,175,0.0429275861,0.0045183181
4,2026-08-07 00:00:00+00:00,131,131,0.0904109589,0.0904109589,-0.3342401078,0.0988765317,0.1693018977,0.0025914613,0.6560791824,0,0,0,0,131,0.0000241494,0.0033582185
5,2026-08-21 00:00:00+00:00,167,167,0.1287671233,0.1287671233,-0.4233284145,0.1391985836,0.1892060277,0.0046097241,0.7016840115,0,0,0,0,167,0.0000255687,0.0014474074
6,2026-08-31 00:00:00+00:00,187,187,0.1561643836,0.1561643836,-0.3842214430,0.1207044586,0.1950474615,0.0059410416,0.6049595923,0,0,0,0,187,0.0000252678,0.0011316634
7,2026-09-18 00:00:00+00:00,214,214,0.2054794521,0.2054794521,-0.4252077096,0.1603088068,0.2073082753,0.0088308837,0.3829953006,0,0,0,0,214,0.0000218632,0.0009467519


,n12_handoff_bucket,moneyness_region_n11,rows,expiries,unique_strikes,median_selected_iv,median_calibration_weight,repair_warn_rows,repair_fail_rows,full_grid_static_warning_rows,inner_grid_static_warning_rows,max_abs_iv_repair
0,anchor_repaired,atm_abs_k_le_005,522,8,86,0.1414143716,0.7998812560,0,0,0,0,0.0000733843
1,anchor_repaired,inner_abs_k_le_015,504,8,111,0.2178726226,0.4196489245,0,0,0,0,0.0000670696
2,clean_repaired,mid_abs_k_le_030,181,7,56,0.3032925306,0.0610870795,3,0,0,0,0.0591598478
3,clean_repaired,outer_abs_k_gt_030,43,5,14,0.4147962423,0.0313128229,1,0,0,0,0.0429275861


,artifact_key,format_key,path,exists
0,broad_handoff,csv,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True
1,broad_handoff,parquet,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True
2,broad_handoff,latest_csv,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True
3,broad_handoff,latest_parquet,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True
4,clean_handoff,csv,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True
5,clean_handoff,parquet,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True
6,clean_handoff,latest_csv,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True
7,clean_handoff,latest_parquet,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True
8,anchor_handoff,csv,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True
9,anchor_handoff,parquet,D:\Derivative Pricing Project v1.0+\V1.1\outpu...,True
